# Hunt-and-Kill y Búsqueda no Informada

**Curso:** Inteligencia Artificial · **Referencia:** AIMA, capítulo 3  
**Modalidad:** individual · **Duración sugerida:** 3 semanas  
**Producto:** un cuaderno Jupyter con tres implementaciones comparables

## Problema

Se proporciona solamente un generador de laberintos perfectos mediante **Hunt-and-Kill**. A partir del grafo producido, el estudiante deberá diseñar, implementar, verificar y comparar algoritmos de búsqueda no informada para encontrar un camino entre dos celdas.

El proyecto debe realizarse en tres versiones:

1. **Desde cero:** sin bibliotecas de búsqueda ni de grafos.
2. **SimpleAI:** modelando el laberinto como un `SearchProblem`.
3. **AIMA-Python:** modelándolo como una subclase de `Problem`.

> Este cuaderno es deliberadamente incompleto. No contiene implementaciones Python de DFS, BFS, UCS, IDDFS, búsqueda bidireccional ni Lee. El trabajo evaluado consiste en convertir las especificaciones y el pseudocódigo en soluciones propias.

## Resultados de aprendizaje

Al finalizar, el estudiante estará en capacidad de:

- formular un entorno como espacio de estados;
- separar el problema de la estrategia de búsqueda;
- implementar búsqueda en árbol y búsqueda en grafo;
- justificar completitud, optimalidad y complejidad;
- adaptar un mismo problema a dos bibliotecas de IA;
- construir experimentos reproducibles;
- interpretar el algoritmo de Lee como BFS por frentes de onda;
- defender decisiones de diseño y diagnosticar errores sin depender de una solución externa.

## Condiciones y restricciones

- No se permite `networkx` ni funciones externas que ya resuelvan caminos.
- En la versión desde cero no se permite SimpleAI, AIMA-Python ni otra biblioteca de búsqueda.
- Los estados deben ser inmutables y utilizables como claves de diccionario.
- Cada algoritmo debe devolver solución y métricas, no solamente imprimirlas.
- Todas las versiones deben trabajar con el **mismo grafo**, inicio, meta y costos.
- Los resultados deben ser reproducibles mediante semillas.
- No se acepta una imagen como evidencia: deben entregarse estructuras de datos verificables.
- El estudiante debe citar cualquier recurso consultado y conservar una bitácora breve de decisiones.

### Evidencia de autoría y comprensión

La evaluación incluye:

1. historial de al menos seis hitos o commits;
2. una traza manual sobre un laberinto pequeño;
3. defensa oral individual de 5–8 minutos;
4. modificación en vivo de inicio, meta, semilla o costo;
5. explicación de una decisión que produjo un error y cómo se corrigió.

Durante la defensa se podrá solicitar reconstruir una parte pequeña del algoritmo sin consultar el cuaderno.

## 1. Código suministrado: generador Hunt-and-Kill

Este es el único algoritmo completo entregado. El resultado es un diccionario de adyacencia no dirigido:

```text
grafo[celda] = conjunto de celdas conectadas por un corredor
```

Una celda se representa como `(fila, columna)`.

In [11]:
import random


class LaberintoHuntKill:
    """Genera un laberinto perfecto como grafo no dirigido."""

    DIRECCIONES = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self, filas=25, columnas=25, semilla=2026):
        self.filas = filas
        self.columnas = columnas
        self.rng = random.Random(semilla)
        self.grafo = {
            (fila, columna): set()
            for fila in range(filas)
            for columna in range(columnas)
        }

    def vecinos_geometricos(self, celda):
        fila, columna = celda
        for df, dc in self.DIRECCIONES:
            vecino = (fila + df, columna + dc)
            if (0 <= vecino[0] < self.filas and
                    0 <= vecino[1] < self.columnas):
                yield vecino

    def conectar(self, origen, destino):
        self.grafo[origen].add(destino)
        self.grafo[destino].add(origen)

    def generar(self):
        no_visitadas = set(self.grafo)
        actual = self.rng.choice(tuple(no_visitadas))
        no_visitadas.remove(actual)

        while no_visitadas:
            # KILL: avanzar aleatoriamente hacia una celda no visitada.
            libres = [
                vecino
                for vecino in self.vecinos_geometricos(actual)
                if vecino in no_visitadas
            ]

            if libres:
                siguiente = self.rng.choice(libres)
                self.conectar(actual, siguiente)
                no_visitadas.remove(siguiente)
                actual = siguiente
                continue

            # HUNT: localizar una celda libre vecina del árbol construido.
            candidatas = []
            for celda in no_visitadas:
                visitados = [
                    vecino
                    for vecino in self.vecinos_geometricos(celda)
                    if vecino not in no_visitadas
                ]
                if visitados:
                    candidatas.append((celda, visitados))

            actual, visitados = self.rng.choice(candidatas)
            vecino = self.rng.choice(visitados)
            self.conectar(actual, vecino)
            no_visitadas.remove(actual)

        return self.grafo

### Actividad 1 — Auditoría del generador

Antes de resolver el laberinto:

1. Explique las fases Hunt y Kill señalando las instrucciones correspondientes.
2. Demuestre que el grafo generado es conexo.
3. Justifique por qué contiene exactamente $|V|-1$ aristas.
4. Explique por qué esas dos propiedades implican que es un árbol.
5. Diseñe pruebas para comprobar:
   - simetría de las adyacencias;
   - ausencia de conexiones diagonales;
   - validez de coordenadas;
   - conectividad;
   - ausencia de ciclos;
   - reproducibilidad con una semilla.

No basta con ejecutar el generador: las propiedades deben verificarse automáticamente.

**Respuesta y pruebas del estudiante:**

### 1.1 Las fases Hunt y Kill, señaladas sobre las instrucciones

El generador mantiene una partición de las celdas en **dos conjuntos**:
`no_visitadas`, que llamaremos $U$, y su complemento, las ya incorporadas al
laberinto, que llamaremos $T$. La inicialización, antes del bucle, establece
$T=\{\texttt{actual}\}$ y $U = V \setminus T$:

```python
no_visitadas = set(self.grafo)                     # U = V
actual = self.rng.choice(tuple(no_visitadas))
no_visitadas.remove(actual)                        # T = {actual}
```

**Fase KILL — «camina mientras puedas».** Es un paseo aleatorio que solo pisa
celdas nuevas:

```python
libres = [vecino for vecino in self.vecinos_geometricos(actual)
          if vecino in no_visitadas]               # vecinos de actual que estan en U
if libres:
    siguiente = self.rng.choice(libres)
    self.conectar(actual, siguiente)               # arista: actual en T, siguiente en U
    no_visitadas.remove(siguiente)                 # siguiente pasa a T
    actual = siguiente                             # el paseo continua
    continue
```

El paseo «muere» cuando `libres` queda vacía, es decir cuando todos los vecinos
geométricos de `actual` ya pertenecen a $T$ y el paseo está encerrado.

**Fase HUNT — «busca dónde renacer».** Recorre $U$ buscando celdas que toquen el
árbol ya construido, engancha una y reinicia el paseo desde ella:

```python
candidatas = []
for celda in no_visitadas:                         # celda en U
    visitados = [vecino for vecino in self.vecinos_geometricos(celda)
                 if vecino not in no_visitadas]    # vecinos de celda que estan en T
    if visitados:
        candidatas.append((celda, visitados))

actual, visitados = self.rng.choice(candidatas)    # celda de U vecina de T
vecino = self.rng.choice(visitados)                # su ancla en T
self.conectar(actual, vecino)
no_visitadas.remove(actual)                        # actual pasa a T
```

**Observación sobre la fidelidad al algoritmo original.** El Hunt canónico de la
literatura **escanea la rejilla fila por fila** y toma la *primera* celda no
visitada que tenga un vecino visitado. Esta implementación recolecta **todas** las
candidatas y elige **una al azar** (`self.rng.choice(candidatas)`). No es el mismo
algoritmo y la diferencia es observable:

- el escaneo por filas introduce un sesgo direccional que produce laberintos con
  corredores largos y predominantemente horizontales; el azar lo elimina y
  reparte los puntos de reinicio de forma uniforme;
- el costo cambia: cada Hunt de esta versión es $O(|U|)$ porque recorre todo $U$,
  y como puede haber $\Theta(|V|)$ fases Hunt, el generador es $O(|V|^2)$ en el
  peor caso. Con $|V| = 500$ es irrelevante, pero explica por qué en la sección 10
  el tiempo de **generación** se mide separado del tiempo de **búsqueda**.

### 1.2 Demostración de que el grafo generado es conexo

No se demuestra sobre el resultado final, sino con un **invariante de bucle**.
Sea $E$ el conjunto de aristas creadas hasta el momento.

> **Invariante $I$.** Al inicio de cada iteración del `while`, el subgrafo
> $(T, E)$ es conexo y $|E| = |T| - 1$.

**Base.** Antes del bucle $T=\{\texttt{actual}\}$ y $E=\varnothing$. Un vértice
aislado es conexo y $|E| = 0 = |T| - 1$. $I$ se cumple.

**Paso inductivo.** Todo el argumento se apoya en un único hecho verificable por
inspección: en todo el programa `conectar` se invoca en **dos** lugares, y en
ambos **exactamente un extremo está en $T$ y el otro en $U$**.

| Fase | Invocación | Extremo en $T$ | Extremo en $U$ |
|---|---|---|---|
| KILL | `conectar(actual, siguiente)` | `actual` | `siguiente` |
| HUNT | `conectar(actual, vecino)` | `vecino` | `actual` |

Que `siguiente` esté en $U$ lo garantiza el filtro `if vecino in no_visitadas`;
que `vecino` esté en $T$ lo garantiza el filtro `if vecino not in no_visitadas`.
En consecuencia, cada iteración agrega **un vértice nuevo** a $T$ y **una única
arista** que lo une al resto. Colgar un vértice de un grafo conexo mediante una
arista preserva la conexidad, y el conteo pasa de $|T|-1$ a $|T'|-1$ con
$|T'| = |T|+1$. $I$ se preserva.

**Terminación.** Cada iteración ejecuta exactamente un `no_visitadas.remove(...)`
—uno en la rama KILL y uno en la rama HUNT—, de modo que $|U|$ decrece en $1$ por
iteración. El bucle ejecuta exactamente $|V|-1$ iteraciones y termina.

**Conclusión.** Al salir del bucle $U=\varnothing$, luego $T=V$, y por $I$ el
grafo $(V,E)$ es conexo. $\blacksquare$

**Condición oculta de la que depende la corrección.** La fase Hunt asume que
`candidatas` nunca está vacía; si lo estuviera, `rng.choice([])` lanzaría
`IndexError`. Nunca ocurre porque mientras $U\neq\varnothing$ y $T\neq\varnothing$
la **rejilla geométrica** es conexa, y por tanto existe al menos una arista de la
rejilla que cruza el corte $(T, U)$. La corrección del generador **depende de que
la rejilla de partida sea conexa**: si el enunciado admitiera celdas bloqueadas de
antemano, el generador fallaría con `IndexError` en lugar de producir un laberinto
parcial.

### 1.3 Justificación de que contiene exactamente $|V|-1$ aristas

Del apartado anterior: el bucle ejecuta $|V|-1$ iteraciones y cada una invoca
`conectar` exactamente una vez, luego hay $|V|-1$ invocaciones.

Falta cerrar un hueco que suele pasarse por alto: `conectar` usa `set.add`, que
**descarta duplicados en silencio**. Si algún par se conectara dos veces,
tendríamos $|V|-1$ invocaciones pero **menos** de $|V|-1$ aristas. No ocurre
porque toda arista creada es incidente al vértice que entra a $T$ en esa
iteración, y cada vértice entra a $T$ una sola vez; las $|V|-1$ aristas son por
tanto distintas dos a dos. $\blacksquare$

Formulación equivalente y directamente verificable en código, ya que el
diccionario almacena cada arista en ambos sentidos:

$$\sum_{v \in V} \bigl|\,\texttt{grafo}[v]\,\bigr| = 2\,(|V|-1)$$

### 1.4 Por qué esas dos propiedades implican que es un árbol

Teorema estándar de teoría de grafos: para un grafo de $n$ vértices, de las tres
propiedades

1. es conexo,
2. es acíclico,
3. tiene exactamente $n-1$ aristas,

**cualesquiera dos implican la tercera**. Habiendo demostrado (1) en 1.2 y (3) en
1.3, se concluye (2), y un grafo conexo y acíclico es por definición un **árbol**.

La intuición del porqué: un grafo conexo sobre $n$ vértices necesita **al menos**
$n-1$ aristas; si tiene exactamente $n-1$ no le sobra ninguna, y un ciclo sería
precisamente una arista sobrante respecto de un árbol de expansión.

**Consecuencia que gobierna el resto del taller.** En un árbol existe **un único
camino simple** entre dos vértices cualesquiera. De ahí se sigue que:

- el laberinto es *perfecto*: sin ciclos y sin regiones inalcanzables;
- **DFS, BFS y UCS devolverán el mismo camino**, no porque los algoritmos sean
  equivalentes, sino porque no existe ninguna alternativa que escoger;
- por tanto un laberinto perfecto **oculta las diferencias de calidad de ruta**, y
  la sección 9 (ciclos y costos) no es un añadido opcional: sin ella la
  optimalidad de UCS es inobservable y ningún experimento puede distinguirla de
  BFS.

### 1.5 Diseño de las pruebas

El enunciado exige verificar seis propiedades. La batería implementa **once**: las
seis exigidas y cinco adicionales que atrapan errores que las primeras no ven.

| # | Propiedad | Qué error atraparía | Exigida |
|---|---|---|---|
| 1 | Simetría: $v \in G[u] \iff u \in G[v]$ | un `conectar` escrito como arista dirigida | sí |
| 2 | Ortogonalidad: $\lvert\Delta f\rvert + \lvert\Delta c\rvert = 1$ | vecindad geométrica con diagonales | sí |
| 3 | Dominio: claves $=$ rejilla, sin coordenadas fuera de rango | celdas faltantes, sobrantes o inválidas | sí |
| 4 | Conectividad por inundación propia | regiones aisladas | sí |
| 5 | Aciclicidad por DFS con arista de retorno | ciclos, **detectados directamente** | sí |
| 6 | Reproducibilidad: misma semilla $\Rightarrow$ mismo grafo | azar no sembrado | sí |
| 7 | $\lvert E\rvert = \lvert V\rvert-1$, $\sum \deg = 2\lvert E\rvert$, $\deg \le 4$ | verifica el teorema de 1.4 | no |
| 8 | Unicidad del camino simple | que sea árbol *en la práctica*, no solo en la demostración | no |
| 9 | Ausencia de lazos ($u \notin G[u]$) | caso degenerado | no |
| 10 | Reproducibilidad **entre procesos** con `PYTHONHASHSEED` distinto | dependencia del hash del proceso | no |
| 11 | `generar()` es de un solo uso | contrato implícito de la clase | no |

Tres decisiones de diseño merecen justificación explícita.

**(a) Las utilidades de verificación son independientes de los buscadores.** La
inundación y el detector de ciclos de esta sección se escriben aparte y no se
reutilizan de la Versión 1. Si validáramos el grafo con el mismo buscador que
luego va a correr sobre ese grafo, un error compartido por ambos **se cancelaría**
y la prueba pasaría estando las dos cosas mal. Una prueba solo tiene valor si
puede fallar de forma independiente de aquello que verifica.

**(b) La aciclicidad se comprueba dos veces por caminos distintos.** La
comprobación 7 la deduce del teorema ($|E| = |V|-1$ junto con conexidad) y la 5 la
detecta directamente buscando una arista de retorno en una DFS. Duplicar el
argumento evita que toda la conclusión «es un árbol» dependa de una sola línea de
código.

**(c) Dos propiedades no son universales, y reconocerlo es parte de la
auditoría.** Se marcan como **no aplicables** en lugar de forzarlas o
silenciarlas:

- La **sensibilidad a la semilla** (comprobación 6, parte «semilla distinta
  $\Rightarrow$ grafo distinto») no puede exigirse en una rejilla $1\times N$ o
  $N\times 1$: la rejilla *es* un camino y su único árbol de expansión es ella
  misma, de modo que ninguna semilla puede producir un laberinto diferente.
- La **no idempotencia** (comprobación 11) tampoco se observa en esas rejillas,
  por la razón que se explica en 1.6.

**Por qué existe la comprobación 10.** El generador ejecuta
`self.rng.choice(tuple(no_visitadas))` y en la fase Hunt recorre `no_visitadas`
con un `for`: es decir, **el orden de iteración de un conjunto alimenta al
generador aleatorio**. Eso resulta reproducible únicamente porque las celdas son
tuplas de enteros, y CPython no aleatoriza el hash de enteros ni de tuplas de
enteros. Si las celdas se representaran como cadenas —por ejemplo `"3,4"`—, la
variable de entorno `PYTHONHASHSEED` cambiaría el orden del conjunto en cada
proceso y **la semilla dejaría de garantizar nada**. La comprobación 6 corre en un
solo proceso y es estructuralmente incapaz de detectarlo; la 10 lanza tres
subprocesos con `PYTHONHASHSEED` $\in \{0, 1, 42\}$ y compara huellas SHA-256.
Esta es la razón de fondo por la que el enunciado insiste en que *«los estados
deben ser inmutables y utilizables como claves de diccionario»*: la elección de
representación no es cosmética, condiciona la reproducibilidad.

**(d) La comprobación 10 incluye su propio contra-experimento.** Una prueba que
no puede fallar no vale nada. Si `PYTHONHASHSEED` no llegara al subproceso —por
un error al construir el entorno, por ejemplo—, las tres huellas coincidirían por
una razón trivial y la comprobación pasaría **sin haber comprobado nada**. Para
descartarlo se ejecuta en paralelo el mismo experimento con las celdas
representadas como **cadenas**, donde el orden del conjunto *sí* debe variar. La
comprobación exige las dos condiciones simultáneamente: invariancia con tuplas de
enteros y variación con cadenas. Medido: las tres huellas con tuplas coinciden y
las tres con cadenas son distintas entre sí.

### 1.6 Hallazgos de la auditoría

**Hallazgo 1 — `generar()` no es idempotente; el objeto es de un solo uso.**
Una segunda llamada a `generar()` sobre el mismo objeto reinicia `no_visitadas` a
partir de `self.grafo`, pero **las aristas de la primera pasada siguen allí**. La
segunda pasada agrega otras $|V|-1$ aristas y el resultado deja de ser un árbol.
Medido sobre la instancia individual: la primera pasada deja $499$ aristas
$=|V|-1$; la segunda deja $722$ y aparece un ciclo. No es un error del código
suministrado sino un **contrato implícito** de la clase, y por eso todo el
cuaderno construye una instancia nueva por laberinto y nunca reutiliza una.

**Hallazgo 2 — el defecto anterior no es universal, y lo descubrió la propia
prueba.** La comprobación 11 falló al ejecutarla sobre una rejilla $1\times 5$.
El diagnóstico: en una rejilla degenerada la única arista posible entre dos celdas
vecinas ya existe tras la primera pasada, así que la segunda intenta recrear
**las mismas** aristas y `set.add` las descarta en silencio; el conteo permanece
en $|V|-1$ y no aparece ningún ciclo. La no idempotencia se manifiesta únicamente
si la rejilla posee ciclos propios, es decir si $\text{filas} \ge 2$ y
$\text{columnas} \ge 2$. La prueba estaba mal planteada, no el generador: se
corrigió marcándola no aplicable en rejillas degeneradas. Queda registrado en
`BITACORA.md` como **D-09**.

**Hallazgo 3 — `inspect.getsource` no sirve para leer una celda bajo
`nbconvert`.** La comprobación 10 necesita el código fuente de
`LaberintoHuntKill` para re-ejecutarlo en un subproceso limpio. La primera
implementación usaba `inspect.getsource(clase)`, que funciona en una sesión
interactiva de Jupyter porque IPython registra el código de cada celda en
`linecache`. Al comprobar que el cuaderno corre de arriba abajo con
`jupyter nbconvert --execute` —justamente el modo que exige el enunciado—
`inspect.getsource` lanzó `OSError: source code not available` y la comprobación
quedó marcada como no aplicable: la auditoría *parecía* pasar con un agujero
dentro. Corrección: una función `fuente_del_generador` que intenta
`inspect.getsource` y, si falla, **lee el propio `.ipynb`** del directorio de
trabajo y localiza la celda que define la clase **por el nombre de la clase, no
por su índice**, de modo que reordenar celdas no rompa la prueba. Registrado como
**D-10**. La lección general es que una comprobación que se degrada a «no
aplicable» en silencio es peor que una que falla: por eso el resumen de la
auditoría imprime explícitamente cuántas comprobaciones quedaron sin aplicar.

**Hallazgo 4 — la distribución de grados describe la textura del laberinto.** En
la instancia individual: $58$ celdas de grado $1$ (callejones sin salida), $387$
de grado $2$ (corredor recto o curva), $54$ de grado $3$ (bifurcación) y $1$ de
grado $4$ (cruce). Un grado máximo de $4$ es consistente con la vecindad
ortogonal, y la abundancia de grado $2$ con el paseo aleatorio de la fase Kill,
que produce corredores largos. Estas cifras se retoman en la sección 11 para
explicar el factor de ramificación efectivo $b$: aunque $b \le 4$ en teoría, el
$b$ **medio observado** es cercano a $2$, y esa es la razón por la que las cotas
$O(b^d)$ sobreestiman ampliamente el número real de nodos expandidos.

In [ ]:
# ===========================================================================
# Actividad 1 - Auditoria automatica del generador Hunt-and-Kill
#
# Las propiedades de un laberinto no se "ven" en un dibujo: se comprueban.
# Cada comprobacion devuelve un objeto `Comprobacion` con evidencia
# verificable -estructuras de datos, no imagenes-, como exige el enunciado.
#
# DECISION DE DISENO (D-07): la inundacion y el detector de ciclos de esta
# seccion son utilidades de VERIFICACION, escritas aparte y a proposito
# independientes de los buscadores de la Version 1. Si validaramos el grafo
# con el mismo buscador que luego corre sobre ese grafo, un error compartido
# se cancelaria y la prueba pasaria estando ambos mal.
# ===========================================================================

from collections import deque
from dataclasses import dataclass, field
import glob
import hashlib
import inspect
import io
import json
import os
import subprocess
import sys
import tempfile


@dataclass
class Comprobacion:
    """Resultado de una sola propiedad auditada."""
    nombre: str
    aprobada: bool
    detalle: str
    evidencia: dict = field(default_factory=dict)
    aplicable: bool = True   # una propiedad puede no tener sentido en un caso limite


# --------------------------------------------------------------------------
# Utilidades sobre el grafo de adyacencia
# --------------------------------------------------------------------------

def aristas_canonicas(grafo):
    """Conjunto de aristas no dirigidas en forma canonica (menor, mayor).

    El grafo guarda cada arista dos veces (u en G[v] y v en G[u]). Ordenar el
    par elimina esa duplicidad y permite contar aristas de verdad.
    """
    return {(u, v) if u <= v else (v, u)
            for u, vecinos in grafo.items()
            for v in vecinos}


def grados(grafo):
    """Grado de cada celda: cuantos corredores salen de ella."""
    return {celda: len(vecinos) for celda, vecinos in grafo.items()}


def huella(grafo):
    """SHA-256 de una serializacion canonica del grafo.

    Sirve para comparar dos grafos por igualdad exacta sin volcar 500 celdas.
    La serializacion se ordena, asi que no depende del orden de iteracion de
    los diccionarios ni de los conjuntos.
    """
    texto = ";".join(
        "{0}->{1}".format(celda, sorted(grafo[celda]))
        for celda in sorted(grafo)
    )
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def inundar(grafo, origen):
    """Inundacion propia (BFS sin metricas) para medir alcanzabilidad.

    No es el BFS de la Version 1: aqui solo interesa el CONJUNTO alcanzado,
    no el camino ni el contrato de metricas. Ver D-07.
    """
    vistos = {origen}
    cola = deque([origen])
    while cola:
        actual = cola.popleft()
        for vecino in grafo[actual]:
            if vecino not in vistos:
                vistos.add(vecino)
                cola.append(vecino)
    return vistos


def buscar_ciclo(grafo):
    """Devuelve una arista de retorno si existe un ciclo, o None.

    DFS iterativa marcando al APILAR y saltando la arista hacia el padre.
    En un grafo no dirigido y simple, encontrar un vecino ya visitado que no
    sea el padre significa que esa arista cierra un ciclo con el arbol DFS.
    """
    visitados = set()
    for raiz in grafo:
        if raiz in visitados:
            continue
        visitados.add(raiz)
        pila = [(raiz, None)]
        while pila:
            nodo, padre = pila.pop()
            for vecino in grafo[nodo]:
                if vecino == padre:
                    continue
                if vecino in visitados:
                    return (nodo, vecino)
                visitados.add(vecino)
                pila.append((vecino, nodo))
    return None


def caminos_simples(grafo, inicio, meta, tope=2):
    """Enumera hasta `tope` caminos simples entre dos celdas.

    Se corta en `tope` porque solo interesa distinguir "exactamente uno" de
    "mas de uno": enumerar todos los caminos de un grafo con ciclos es
    exponencial y aqui no hace falta.
    """
    encontrados = []
    pila = [(inicio, (inicio,))]
    while pila and len(encontrados) < tope:
        nodo, camino = pila.pop()
        if nodo == meta:
            encontrados.append(camino)
            continue
        for vecino in sorted(grafo[nodo]):
            if vecino not in camino:          # camino SIMPLE: sin repetir
                pila.append((vecino, camino + (vecino,)))
    return encontrados

In [ ]:
# --------------------------------------------------------------------------
# Las once comprobaciones
#
# Dos de ellas NO son universales, y reconocerlo es parte de la auditoria:
#   - la sensibilidad a la semilla no puede exigirse en una rejilla 1xN,
#     porque un camino tiene un unico arbol de expansion posible;
#   - la no idempotencia de `generar()` no puede observarse con |V| = 1,
#     porque no hay ninguna arista que duplicar.
# Se marcan como NO APLICABLES en lugar de forzarlas o de silenciarlas.
# --------------------------------------------------------------------------

def comprobar_simetria(grafo):
    """1. v en G[u] <=> u en G[v]. Atrapa un `conectar` escrito como dirigido."""
    fallos = [(u, v) for u, vecinos in grafo.items()
              for v in vecinos if u not in grafo.get(v, ())]
    return Comprobacion(
        "1. Simetria de las adyacencias",
        not fallos,
        "las {0} aristas son bidireccionales".format(len(aristas_canonicas(grafo)))
        if not fallos else "{0} adyacencias asimetricas".format(len(fallos)),
        {"pares_asimetricos": fallos[:5]},
    )


def comprobar_ortogonalidad(grafo):
    """2. Sin diagonales: la distancia Manhattan de toda arista debe ser 1."""
    fallos = [(u, v) for u, vecinos in grafo.items() for v in vecinos
              if abs(u[0] - v[0]) + abs(u[1] - v[1]) != 1]
    return Comprobacion(
        "2. Ausencia de conexiones diagonales",
        not fallos,
        "toda arista tiene distancia Manhattan 1"
        if not fallos else "{0} aristas no ortogonales".format(len(fallos)),
        {"aristas_invalidas": fallos[:5]},
    )


def comprobar_dominio(grafo, filas, columnas):
    """3. El grafo cubre exactamente la rejilla filas x columnas, sin extras."""
    esperadas = {(f, c) for f in range(filas) for c in range(columnas)}
    presentes = set(grafo)
    fuera = {v for vecinos in grafo.values() for v in vecinos} - esperadas
    ok = presentes == esperadas and not fuera
    return Comprobacion(
        "3. Validez de coordenadas y cobertura",
        ok,
        "{0} celdas = {1}x{2}, ninguna coordenada fuera de rango".format(
            len(presentes), filas, columnas)
        if ok else "faltan {0}, sobran {1}, vecinos invalidos {2}".format(
            len(esperadas - presentes), len(presentes - esperadas), len(fuera)),
        {"faltantes": sorted(esperadas - presentes)[:5],
         "sobrantes": sorted(presentes - esperadas)[:5],
         "vecinos_fuera_de_rango": sorted(fuera)[:5]},
    )


def comprobar_conectividad(grafo):
    """4. Una sola componente: desde cualquier celda se alcanzan todas."""
    origen = next(iter(grafo))
    alcanzadas = inundar(grafo, origen)
    ok = len(alcanzadas) == len(grafo)
    return Comprobacion(
        "4. Conectividad (una sola componente)",
        ok,
        "desde {0} se alcanzan las {1} celdas".format(origen, len(alcanzadas))
        if ok else "solo {0} de {1} celdas alcanzables".format(
            len(alcanzadas), len(grafo)),
        {"origen": origen, "alcanzadas": len(alcanzadas), "total": len(grafo),
         "aisladas_ejemplo": sorted(set(grafo) - alcanzadas)[:5]},
    )


def comprobar_aciclicidad(grafo):
    """5. Sin ciclos, detectado DIRECTAMENTE y no por el conteo de aristas.

    Importa que sea directo: el teorema "conexo + |V|-1 aristas => arbol" ya
    se usa en la comprobacion 7. Detectar el ciclo con una DFS independiente
    evita que toda la conclusion dependa de un unico argumento.
    """
    arista = buscar_ciclo(grafo)
    return Comprobacion(
        "5. Ausencia de ciclos (DFS con arista de retorno)",
        arista is None,
        "ninguna arista de retorno en la DFS"
        if arista is None else "ciclo cerrado por la arista {0}".format(arista),
        {"arista_de_retorno": arista},
    )


def comprobar_reproducibilidad(clase, filas, columnas, semilla,
                               exigir_sensibilidad=True):
    """6. Misma semilla -> grafo identico; semilla distinta -> grafo distinto.

    `exigir_sensibilidad=False` en rejillas degeneradas (1xN o Nx1): un camino
    tiene un unico arbol de expansion, asi que ninguna semilla puede producir
    un laberinto distinto. Exigirlo alli seria una prueba mal planteada, no un
    defecto del generador.
    """
    h1 = huella(clase(filas, columnas, semilla).generar())
    h2 = huella(clase(filas, columnas, semilla).generar())
    h3 = huella(clase(filas, columnas, semilla + 1).generar())

    determinista = h1 == h2
    sensible = h1 != h3
    ok = determinista and (sensible or not exigir_sensibilidad)
    if not determinista:
        detalle = "la misma semilla produjo dos grafos distintos"
    elif exigir_sensibilidad and not sensible:
        detalle = "semillas distintas produjeron el mismo grafo"
    elif exigir_sensibilidad:
        detalle = "semilla {0} reproduce la huella; {1} produce otra".format(
            semilla, semilla + 1)
    else:
        detalle = ("determinista; sensibilidad no exigida (rejilla degenerada: "
                   "el arbol de expansion es unico)")
    return Comprobacion(
        "6. Reproducibilidad con semilla", ok, detalle,
        {"huella_semilla": h1[:16], "huella_repeticion": h2[:16],
         "huella_semilla_mas_uno": h3[:16],
         "determinista": determinista, "sensible_a_la_semilla": sensible},
    )


def comprobar_conteo_aristas(grafo):
    """7. |E| == |V|-1, suma de grados == 2|E|, grado maximo <= 4."""
    aristas = aristas_canonicas(grafo)
    gs = grados(grafo)
    suma_grados = sum(gs.values())
    n = len(grafo)
    grado_max = max(gs.values())
    ok = (len(aristas) == n - 1 and suma_grados == 2 * (n - 1) and grado_max <= 4)
    return Comprobacion(
        "7. Conteo de aristas |E| = |V|-1 y grados <= 4", ok,
        "|V|={0}, |E|={1}, suma de grados={2}, grado max={3}".format(
            n, len(aristas), suma_grados, grado_max),
        {"vertices": n, "aristas": len(aristas), "esperadas": n - 1,
         "suma_grados": suma_grados, "grado_maximo": grado_max,
         "distribucion_grados": {g: list(gs.values()).count(g)
                                 for g in sorted(set(gs.values()))}},
    )


def comprobar_unicidad_camino(grafo, inicio, meta):
    """8. En un arbol existe EXACTAMENTE un camino simple entre dos celdas.

    Esta es la propiedad que hace "perfecto" al laberinto, y la razon por la
    que DFS, BFS y UCS devolveran el MISMO camino: no hay alternativa que
    escoger. Sin la seccion 9 (ciclos) la optimalidad seria invisible.
    """
    caminos = caminos_simples(grafo, inicio, meta, tope=2)
    ok = len(caminos) == 1
    return Comprobacion(
        "8. Unicidad del camino simple", ok,
        "un unico camino de {0} a {1}, de {2} celdas".format(
            inicio, meta, len(caminos[0])) if ok
        else "se encontraron {0} caminos entre {1} y {2}".format(
            len(caminos), inicio, meta),
        {"numero_de_caminos": len(caminos),
         "longitud_en_celdas": len(caminos[0]) if caminos else 0,
         "primeras_celdas": list(caminos[0][:5]) if caminos else []},
    )


def comprobar_sin_lazos(grafo):
    """9. Ninguna celda es vecina de si misma."""
    lazos = [u for u, vecinos in grafo.items() if u in vecinos]
    return Comprobacion(
        "9. Ausencia de lazos", not lazos,
        "ninguna celda se conecta consigo misma"
        if not lazos else "{0} lazos".format(len(lazos)),
        {"lazos": lazos[:5]},
    )


def fuente_del_generador(clase):
    """Devuelve el codigo fuente de la clase generadora, o None.

    Hace falta para re-ejecutar el generador en un SUBPROCESO limpio, y
    obtenerlo no es trivial cuando la clase se define en una celda:

    1. `inspect.getsource` funciona en una sesion interactiva de IPython
       -que registra el codigo de cada celda en `linecache`- pero FALLA con
       OSError bajo `jupyter nbconvert --execute`, que es precisamente el modo
       en que se comprueba que el cuaderno corre de arriba abajo.
    2. Como respaldo se lee el propio `.ipynb` del directorio de trabajo y se
       localiza la celda que define la clase por su nombre, no por su indice,
       para que reordenar celdas no rompa la prueba.
    """
    try:
        return inspect.getsource(clase)
    except (OSError, TypeError):
        pass

    firma = "class " + clase.__name__
    for ruta in sorted(glob.glob("*.ipynb")):
        try:
            cuaderno = json.load(io.open(ruta, encoding="utf-8"))
        except (ValueError, OSError):
            continue
        for celda in cuaderno.get("cells", []):
            fuente = "".join(celda.get("source", []))
            if firma in fuente:
                return fuente
    return None


def _ejecutar_en_subprocesos(guion, valores=("0", "1", "42")):
    """Corre un guion una vez por cada valor de PYTHONHASHSEED."""
    ruta = os.path.join(tempfile.mkdtemp(), "replica.py")
    with io.open(ruta, "w", encoding="utf-8") as archivo:
        archivo.write(guion)
    salidas = {}
    for valor in valores:
        entorno = dict(os.environ, PYTHONHASHSEED=valor)
        proceso = subprocess.run([sys.executable, ruta], capture_output=True,
                                 text=True, env=entorno, check=True)
        salidas[valor] = proceso.stdout.strip()
    return salidas


def comprobar_reproducibilidad_entre_procesos(clase, filas, columnas, semilla):
    """10. Misma semilla en procesos distintos con PYTHONHASHSEED distinto.

    POR QUE ESTA PRUEBA EXISTE (D-08):
    el generador ejecuta `rng.choice(tuple(no_visitadas))` y en la fase Hunt
    recorre `no_visitadas` con un `for`. Es decir, EL ORDEN DE ITERACION DE UN
    CONJUNTO ALIMENTA AL GENERADOR ALEATORIO. Eso resulta reproducible solo
    porque las celdas son tuplas de enteros, y CPython no aleatoriza el hash
    de enteros ni de tuplas de enteros. Si las celdas fueran cadenas -por
    ejemplo "3,4"-, `PYTHONHASHSEED` cambiaria el orden del conjunto en cada
    proceso y la semilla NO garantizaria nada. La comprobacion 6 corre en un
    solo proceso y es estructuralmente incapaz de detectarlo.

    LA PRUEBA INCLUYE SU PROPIO CONTRA-EXPERIMENTO (D-10). Una prueba que no
    puede fallar no vale nada: si `PYTHONHASHSEED` no llegara al subproceso,
    las tres huellas coincidirian por una razon trivial y la prueba pasaria
    sin haber comprobado nada. Por eso se corre en paralelo el MISMO
    experimento con celdas representadas como cadenas, donde el orden del
    conjunto SI debe cambiar. La comprobacion exige las dos cosas a la vez:
    invariancia con tuplas y variacion con cadenas.
    """
    fuente = fuente_del_generador(clase)
    if fuente is None:
        return Comprobacion(
            "10. Reproducibilidad entre procesos (PYTHONHASHSEED)", True,
            "no aplicable: no se pudo recuperar el fuente de la clase",
            {"motivo": "ni inspect.getsource ni la lectura del .ipynb"},
            aplicable=False)

    guion_generador = (
        "import random, hashlib\n" + fuente + "\n"
        + "g = {0}({1}, {2}, {3}).generar()\n".format(
            clase.__name__, filas, columnas, semilla)
        + 'texto = ";".join("{0}->{1}".format(c, sorted(g[c])) for c in sorted(g))\n'
        + 'print(hashlib.sha256(texto.encode("utf-8")).hexdigest())\n'
    )

    # Contra-experimento: las mismas celdas, pero como CADENAS.
    guion_cadenas = (
        "import hashlib\n"
        + "celdas = ['{0},{1}'.format(f, c)\n"
        + "          for f in range({0}) for c in range({1})]\n".format(
            filas, columnas)
        + "orden = ';'.join(set(celdas))\n"
        + 'print(hashlib.sha256(orden.encode("utf-8")).hexdigest())\n'
    )

    con_tuplas = _ejecutar_en_subprocesos(guion_generador)
    con_cadenas = _ejecutar_en_subprocesos(guion_cadenas)
    referencia = huella(clase(filas, columnas, semilla).generar())

    invariante = len(set(con_tuplas.values())) == 1
    coincide = con_tuplas["0"] == referencia
    contraste = len(set(con_cadenas.values())) > 1
    ok = invariante and coincide and contraste

    if not invariante:
        detalle = "el laberinto depende del PYTHONHASHSEED del proceso"
    elif not coincide:
        detalle = "el subproceso no reproduce el grafo de este proceso"
    elif not contraste:
        detalle = ("PRUEBA VACIA: con cadenas el orden tampoco cambio, luego "
                   "PYTHONHASHSEED no esta surtiendo efecto")
    else:
        detalle = ("invariante con tuplas de enteros y variable con cadenas: "
                   "{0} ordenes distintos".format(len(set(con_cadenas.values()))))

    return Comprobacion(
        "10. Reproducibilidad entre procesos (PYTHONHASHSEED)", ok, detalle,
        {"huellas_con_tuplas": {k: v[:16] for k, v in con_tuplas.items()},
         "huella_en_este_proceso": referencia[:16],
         "huellas_del_contraexperimento_con_cadenas":
             {k: v[:16] for k, v in con_cadenas.items()},
         "invariante_con_tuplas": invariante,
         "varia_con_cadenas": contraste},
    )


def comprobar_un_solo_uso(clase, filas, columnas, semilla):
    """11. Fragilidad documentada: `generar()` NO es idempotente.

    Llamar `generar()` dos veces sobre el MISMO objeto reinicia
    `no_visitadas` desde `self.grafo`, pero las aristas de la primera pasada
    siguen ahi. La segunda pasada agrega otras |V|-1 aristas y CREA CICLOS:
    el resultado deja de ser un arbol.

    La prueba no denuncia un error del codigo suministrado: documenta un
    contrato implicito de la clase. Todo el cuaderno construye una instancia
    nueva por laberinto y nunca reutiliza una.

    PERO EL DEFECTO NO ES UNIVERSAL (hallazgo, ver D-09). En una rejilla
    degenerada 1xN o Nx1 la segunda pasada NO rompe nada: la rejilla es un
    camino, su unico arbol de expansion es ella misma, asi que las aristas que
    la segunda pasada intenta crear YA EXISTEN y `set.add` las descarta en
    silencio. El conteo sigue en |V|-1 y no aparece ningun ciclo.
    Conclusion: la no idempotencia se observa solo si la rejilla tiene ciclos
    propios, es decir si filas >= 2 y columnas >= 2.
    """
    n = filas * columnas
    if filas < 2 or columnas < 2:
        return Comprobacion(
            "11. `generar()` es de un solo uso (fragilidad documentada)", True,
            "no aplicable: en una rejilla {0}x{1} el arbol de expansion es "
            "unico, asi que repetir `generar()` reescribe las mismas aristas "
            "y `set.add` las descarta".format(filas, columnas),
            {"vertices": n, "filas": filas, "columnas": columnas},
            aplicable=False)

    laberinto = clase(filas, columnas, semilla)
    primera = huella(laberinto.generar())
    aristas_primera = len(aristas_canonicas(laberinto.grafo))
    laberinto.generar()                   # segunda llamada: rompe el arbol
    aristas_segunda = len(aristas_canonicas(laberinto.grafo))
    ciclo = buscar_ciclo(laberinto.grafo)

    ok = (aristas_primera == n - 1 and aristas_segunda > n - 1
          and ciclo is not None)
    return Comprobacion(
        "11. `generar()` es de un solo uso (fragilidad documentada)", ok,
        "1a pasada {0} aristas (=|V|-1); 2a pasada {1} y aparece el ciclo "
        "{2}".format(aristas_primera, aristas_segunda, ciclo) if ok
        else "el comportamiento observado no coincide con el esperado",
        {"aristas_primera_pasada": aristas_primera,
         "aristas_segunda_pasada": aristas_segunda,
         "esperado_arbol": n - 1, "ciclo_detectado": ciclo,
         "huella_primera_pasada": primera[:16]},
    )

In [ ]:
# --------------------------------------------------------------------------
# Ejecutor de la bateria
# --------------------------------------------------------------------------

def auditar(clase, filas, columnas, semilla, inicio=None, meta=None,
            grafo=None, entre_procesos=True):
    """Corre la bateria y DEVUELVE la lista de resultados; no imprime.

    Quien llama decide como presentarlos: el enunciado exige evidencia
    verificable, no salida por pantalla.

    - `inicio`/`meta`: por omision, esquinas opuestas de la rejilla.
    - `grafo`: permite auditar un grafo ya generado en lugar de generar otro.
    - `entre_procesos`: la comprobacion 10 lanza tres subprocesos, asi que se
      desactiva en las configuraciones secundarias por costo, no por dudas.
    """
    if grafo is None:
        grafo = clase(filas, columnas, semilla).generar()
    if inicio is None:
        inicio = (0, 0)
    if meta is None:
        meta = (filas - 1, columnas - 1)

    degenerada = filas < 2 or columnas < 2

    resultados = [
        comprobar_simetria(grafo),
        comprobar_ortogonalidad(grafo),
        comprobar_dominio(grafo, filas, columnas),
        comprobar_conectividad(grafo),
        comprobar_aciclicidad(grafo),
        comprobar_reproducibilidad(clase, filas, columnas, semilla,
                                   exigir_sensibilidad=not degenerada),
        comprobar_conteo_aristas(grafo),
        comprobar_unicidad_camino(grafo, inicio, meta),
        comprobar_sin_lazos(grafo),
    ]
    if entre_procesos:
        resultados.append(
            comprobar_reproducibilidad_entre_procesos(
                clase, filas, columnas, semilla))
    resultados.append(comprobar_un_solo_uso(clase, filas, columnas, semilla))
    return resultados


def imprimir_auditoria(resultados, titulo="AUDITORIA DEL GENERADOR HUNT-AND-KILL"):
    """Tabla legible + `assert`: si algo falla, el cuaderno se detiene aqui."""
    ancho = max(len(r.nombre) for r in resultados)
    print(titulo)
    print("-" * (ancho + 62))
    for r in resultados:
        if not r.aplicable:
            marca = "N/A "
        elif r.aprobada:
            marca = " OK "
        else:
            marca = "FALL"
        print("[{0}] {1}  {2}".format(marca, r.nombre.ljust(ancho), r.detalle))
    print("-" * (ancho + 62))

    aplicables = [r for r in resultados if r.aplicable]
    fallidas = [r.nombre for r in aplicables if not r.aprobada]
    print("{0}/{1} comprobaciones aplicables aprobadas ({2} no aplicables)".format(
        len(aplicables) - len(fallidas), len(aplicables),
        len(resultados) - len(aplicables)))
    assert not fallidas, "comprobaciones fallidas: {0}".format(fallidas)
    return resultados

In [ ]:
# --------------------------------------------------------------------------
# Ejecucion de la bateria
#
# Se audita el generador en CINCO configuraciones y no solo en la instancia
# individual: un generador correcto debe seguir siendolo en los casos limite,
# y ahi es donde aparecieron los dos hallazgos del apartado 1.6.
#
# La instancia individual se vuelve a auditar en la seccion 2, una vez
# definidos formalmente sus parametros, para no crear aqui una dependencia
# hacia adelante (el cuaderno debe ejecutarse de arriba abajo).
# --------------------------------------------------------------------------

CONFIGURACIONES_AUDITORIA = [
    (1,  1,  97950, "caso limite: una sola celda"),
    (1,  5,  97950, "caso limite: rejilla degenerada 1xN"),
    (5,  1,  97950, "caso limite: rejilla degenerada Nx1"),
    (3,  3,  97950, "laberinto minimo no trivial"),
    (20, 25, 97950, "tamano de la instancia individual"),
]

AUDITORIAS = {}
for _filas, _columnas, _semilla, _nota in CONFIGURACIONES_AUDITORIA:
    # La comprobacion 10 lanza tres subprocesos; se activa solo en la
    # configuracion principal por costo, no por dudas sobre su validez.
    _principal = (_filas, _columnas) == (20, 25)
    AUDITORIAS[(_filas, _columnas)] = imprimir_auditoria(
        auditar(LaberintoHuntKill, _filas, _columnas, _semilla,
                entre_procesos=_principal),
        "AUDITORIA {0}x{1}  -  {2}".format(_filas, _columnas, _nota))
    print()

# El enunciado exige evidencia verificable, no solo salida por pantalla:
# `AUDITORIAS` conserva los objetos Comprobacion con su campo `evidencia`.
print("Evidencia estructurada de tres comprobaciones sobre la rejilla 20x25:")
for _comprobacion in AUDITORIAS[(20, 25)]:
    if _comprobacion.nombre[:2] in ("7.", "10", "11"):
        print(" ", _comprobacion.nombre)
        for _clave, _valor in _comprobacion.evidencia.items():
            print("     {0} = {1}".format(_clave, _valor))

### 1.7 Herramienta auxiliar: leer el grafo dibujándolo

El enunciado advierte que *«no se acepta una imagen como evidencia: deben
entregarse estructuras de datos verificables»*. Las once comprobaciones
anteriores son la evidencia; este dibujo **no demuestra nada**. Se incluye por
dos razones distintas y ambas legítimas:

1. **Para leer la representación.** La idea central del modelado es que
   *no hay paredes almacenadas en ninguna parte*. El diccionario guarda
   únicamente pasillos, y **una pared es exactamente la ausencia de la arista
   correspondiente**. El dibujo hace visible esa equivalencia: para cada
   frontera entre dos celdas pregunta si la arista existe, y si no existe traza
   un muro.
2. **Para detectar un error grosero de un vistazo.** Una prueba dice *qué*
   falló; un dibujo sugiere *por qué*. Si la comprobación de conectividad
   fallara, ver una región cerrada ahorra media hora de depuración.

La función recibe las marcas como un **diccionario** `{celda: texto}` y no como
una lista de celdas. Eso no es un capricho: en la sección 7 el algoritmo de Lee
necesitará superponer sobre cada celda su **tiempo de llegada** del frente de
onda, no un simple asterisco. Diseñar ahora la firma general evita reescribir
la función después.

In [ ]:
# --------------------------------------------------------------------------
# Herramienta auxiliar: dibujar el laberinto en texto
#
# El enunciado advierte que "no se acepta una imagen como evidencia": las
# propiedades se demuestran con las comprobaciones de arriba, no con un dibujo.
# Pero un dibujo sirve para LEER el grafo y para detectar a simple vista un
# error grosero, asi que se incluye como apoyo, no como prueba.
#
# La clave para entender la representacion: NO HAY PAREDES ALMACENADAS. El
# diccionario solo guarda pasillos, y una pared es exactamente la AUSENCIA de
# la arista correspondiente. Por eso el dibujo pregunta, para cada frontera
# entre dos celdas, si la arista existe.
#
# `marcas` permite superponer texto sobre las celdas: un asterisco para el
# camino, o numeros para el tiempo de llegada del frente de onda de Lee en la
# seccion 7. Por eso la funcion recibe un diccionario y no una lista.
# --------------------------------------------------------------------------

def dibujar_laberinto(grafo, filas, columnas, marcas=None, ancho=3):
    """Devuelve el laberinto como una cadena de texto con paredes.

    - `marcas`: dict {(fila, columna): texto} que se centra dentro de la celda.
    - `ancho`: caracteres de ancho interior de cada celda. 3 se lee bien en
      laberintos pequenos; 1 hace caber una rejilla de 25 columnas en pantalla.
    """
    marcas = marcas or {}
    barra = "-" * ancho
    hueco = " " * ancho

    lineas = ["+" + (barra + "+") * columnas]
    for fila in range(filas):
        interior = "|"
        inferior = "+"
        for columna in range(columnas):
            celda = (fila, columna)
            texto = str(marcas[celda]).center(ancho)[:ancho] if celda in marcas else hueco
            interior += texto
            # Pared derecha: existe si la arista hacia la celda de al lado NO esta.
            interior += " " if (fila, columna + 1) in grafo[celda] else "|"
            # Pared inferior: idem hacia abajo.
            inferior += (hueco if (fila + 1, columna) in grafo[celda] else barra) + "+"
        lineas.append(interior)
        lineas.append(inferior)
    return "\n".join(lineas)


def marcas_de_camino(camino, simbolo="*"):
    """Convierte una secuencia de celdas en marcas para `dibujar_laberinto`."""
    return {celda: simbolo for celda in camino}


# --- Demostracion 1: un laberinto pequeno, legible celda por celda ---------
_GRAFO_DEMO = LaberintoHuntKill(5, 5, 97950).generar()

print("LABERINTO 5x5 (semilla 97950)")
print(dibujar_laberinto(_GRAFO_DEMO, 5, 5))
print()
print("El generador no devuelve ese dibujo, devuelve este diccionario:")
for _celda in [(0, 0), (2, 2), (4, 4)]:
    print("   grafo[{0}] = {1}".format(_celda, sorted(_GRAFO_DEMO[_celda])))
print()
print("Lectura: desde (0, 0) solo se puede pasar a {0}; hacia las otras dos "
      "celdas vecinas hay pared, es decir, NO hay arista.".format(
          sorted(_GRAFO_DEMO[(0, 0)])))

# --- Demostracion 2: el camino unico, superpuesto sobre el mismo laberinto -
_CAMINO_DEMO = caminos_simples(_GRAFO_DEMO, (0, 0), (4, 4), tope=1)[0]
print()
print("UNICO CAMINO DE (0,0) A (4,4): {0} celdas, {1} pasos".format(
    len(_CAMINO_DEMO), len(_CAMINO_DEMO) - 1))
print(dibujar_laberinto(_GRAFO_DEMO, 5, 5, marcas_de_camino(_CAMINO_DEMO)))
print()
print("Como secuencia de estados, que es lo que deberan devolver los siete")
print("algoritmos de la Version 1:")
print("   " + " -> ".join(str(c) for c in _CAMINO_DEMO))

# --- Demostracion 3: la rejilla del tamano de la instancia individual ------
# Se usa ancho=1 para que 25 columnas quepan en pantalla. La instancia formal,
# con su inicio y su meta, se define en la seccion 2; aqui solo se comprueba
# que el dibujo escala y que el camino unico sigue siendo unico.
_GRAFO_20x25 = LaberintoHuntKill(20, 25, 97950).generar()
_CAMINO_20x25 = caminos_simples(_GRAFO_20x25, (0, 0), (19, 24), tope=1)[0]
print()
print("REJILLA 20x25 (semilla 97950) CON EL CAMINO DE (0,0) A (19,24)")
print("{0} celdas de camino, {1} pasos".format(
    len(_CAMINO_20x25), len(_CAMINO_20x25) - 1))
print(dibujar_laberinto(_GRAFO_20x25, 20, 25,
                        marcas_de_camino(_CAMINO_20x25), ancho=1))

## 2. Instancia individual

Cada estudiante debe construir una instancia reproducible:

- `semilla = últimos 5 dígitos del código estudiantil`;
- `filas = 20 + último dígito`;
- `columnas = 20 + penúltimo dígito`;
- inicio y meta deben estar en cuadrantes opuestos, pero no necesariamente en las esquinas.

Registre esos valores en una tabla. El docente podrá cambiar cualquiera de ellos durante la defensa.

Código estudiantil: **1089097950**.

| Parámetro | Valor | Origen |
|---|---|---|
| Semilla | **97950** | últimos 5 dígitos |
| Filas | **20** | 20 + último dígito (0) |
| Columnas | **25** | 20 + penúltimo dígito (5) |
| Inicio | **(3, 4)** | cuadrante noroeste |
| Meta | **(16, 20)** | cuadrante sureste |

Inicio y meta están en cuadrantes opuestos y **no** en las esquinas: la sección 1
ya auditó el recorrido esquina a esquina `(0, 0) → (19, 24)`, así que la
instancia individual usa puntos interiores, lejos de las líneas centrales, para
que su clasificación por cuadrante no dependa de ningún desempate (D-15).

La celda siguiente **genera esta misma tabla** a partir de los valores reales.
Si en la defensa se cambia un parámetro, manda la salida de la celda, no este
texto: basta editar las líneas marcadas `<-- PARAMETRO` y ejecutar todo.

In [ ]:
# --------------------------------------------------------------------------
# Seccion 2 - Instancia individual
#
# PARA LA DEFENSA: el docente puede cambiar semilla, filas, columnas, inicio
# o meta. Basta con editar las lineas marcadas con  <-- PARAMETRO  y ejecutar
# el cuaderno completo. Todas las secciones posteriores leen `INSTANCIA` y
# ninguna repite estos valores (D-12).
#
# `construir_instancia` VALIDA los parametros antes de generar: un inicio
# fuera de la rejilla o escrito como lista en vez de tupla debe fallar aqui,
# con un mensaje claro, y no diez celdas mas abajo dentro de un buscador.
# --------------------------------------------------------------------------

from types import MappingProxyType

CODIGO_ESTUDIANTIL = "1089097950"


def parametros_desde_codigo(codigo):
    """Aplica literalmente la regla del enunciado al codigo estudiantil."""
    semilla = int(codigo[-5:])          # ultimos 5 digitos
    filas = 20 + int(codigo[-1])        # 20 + ultimo digito
    columnas = 20 + int(codigo[-2])     # 20 + penultimo digito
    return semilla, filas, columnas


SEMILLA, FILAS, COLUMNAS = parametros_desde_codigo(CODIGO_ESTUDIANTIL)
# Para cambiar la semilla en la defensa, descomentar:
# SEMILLA = 12345                                             # <-- PARAMETRO
INICIO = (3, 4)                                               # <-- PARAMETRO
META = (16, 20)                                               # <-- PARAMETRO


def cuadrante(celda, filas, columnas):
    """'NO', 'NE', 'SO' o 'SE'.

    Con una dimension impar, la linea central pertenece a la mitad norte u
    oeste (2*i < n). Es un desempate arbitrario pero declarado; la instancia
    elegida esta lejos de las lineas centrales y no depende de el.
    """
    fila, columna = celda
    vertical = "N" if 2 * fila < filas else "S"
    horizontal = "O" if 2 * columna < columnas else "E"
    return vertical + horizontal


def cuadrantes_opuestos(a, b, filas, columnas):
    """Opuestos = distintos en AMBOS ejes (NO-SE o NE-SO)."""
    qa, qb = cuadrante(a, filas, columnas), cuadrante(b, filas, columnas)
    return qa[0] != qb[0] and qa[1] != qb[1]


def _validar_celda(nombre, celda, filas, columnas):
    # Tupla y no lista: el estado debe ser inmutable y usable como clave de
    # diccionario (D-08). Una lista fallaria recien dentro de `alcanzados`.
    if not (isinstance(celda, tuple) and len(celda) == 2
            and all(type(x) is int for x in celda)):
        raise ValueError("{0} debe ser una tupla (fila, columna) de enteros; "
                         "se recibio {1!r}".format(nombre, celda))
    if not (0 <= celda[0] < filas and 0 <= celda[1] < columnas):
        raise ValueError("{0} = {1} esta fuera de la rejilla {2}x{3}".format(
            nombre, celda, filas, columnas))


@dataclass(frozen=True)
class Instancia:
    """Todo lo que define un problema concreto. Inmutable: nadie la altera."""
    semilla: int
    filas: int
    columnas: int
    inicio: tuple
    meta: tuple
    grafo: MappingProxyType     # {celda: frozenset(vecinos)}, de solo lectura


def construir_instancia(semilla, filas, columnas, inicio, meta,
                        exigir_cuadrantes_opuestos=True):
    """Valida los parametros, genera el laberinto y lo congela.

    `exigir_cuadrantes_opuestos=False` existe para las pruebas de la seccion 8,
    que necesitan instancias diminutas (1x1, inicio == meta) donde la regla
    del enunciado no tiene sentido.
    """
    if not (type(filas) is int and type(columnas) is int
            and filas >= 1 and columnas >= 1):
        raise ValueError("filas y columnas deben ser enteros >= 1")
    _validar_celda("inicio", inicio, filas, columnas)
    _validar_celda("meta", meta, filas, columnas)
    if exigir_cuadrantes_opuestos and not cuadrantes_opuestos(
            inicio, meta, filas, columnas):
        raise ValueError("inicio {0} ({1}) y meta {2} ({3}) no estan en "
                         "cuadrantes opuestos".format(
                             inicio, cuadrante(inicio, filas, columnas),
                             meta, cuadrante(meta, filas, columnas)))

    # Un generador NUEVO en cada llamada: `generar()` es de un solo uso
    # (comprobacion 11). Reutilizar el objeto produciria ciclos.
    crudo = LaberintoHuntKill(filas, columnas, semilla).generar()

    # Se congela (D-13): las tres versiones y los siete algoritmos comparten
    # este mismo grafo. Si uno lo modificara por error, contaminaria a los
    # demas en silencio; asi, el intento lanza TypeError/AttributeError.
    grafo = MappingProxyType({celda: frozenset(vecinos)
                              for celda, vecinos in crudo.items()})
    return Instancia(semilla, filas, columnas, inicio, meta, grafo)


INSTANCIA = construir_instancia(SEMILLA, FILAS, COLUMNAS, INICIO, META)

# --- La tabla de la seccion 2, generada desde los valores reales -----------
print("INSTANCIA INDIVIDUAL (codigo {0})".format(CODIGO_ESTUDIANTIL))
print("  Semilla   {0}".format(INSTANCIA.semilla))
print("  Filas     {0}".format(INSTANCIA.filas))
print("  Columnas  {0}".format(INSTANCIA.columnas))
print("  Inicio    {0}  cuadrante {1}".format(
    INSTANCIA.inicio, cuadrante(INSTANCIA.inicio, FILAS, COLUMNAS)))
print("  Meta      {0}  cuadrante {1}".format(
    INSTANCIA.meta, cuadrante(INSTANCIA.meta, FILAS, COLUMNAS)))
print("  Huella    {0}".format(huella(INSTANCIA.grafo)[:16]))
print()

# --- Auditoria de ESTA instancia, ahora con su inicio y su meta reales -----
# La seccion 1 audito la rejilla con inicio y meta en las esquinas; aqui la
# comprobacion 8 (camino unico) se repite entre los extremos verdaderos.
AUDITORIA_INSTANCIA = imprimir_auditoria(
    auditar(LaberintoHuntKill, INSTANCIA.filas, INSTANCIA.columnas,
            INSTANCIA.semilla, INSTANCIA.inicio, INSTANCIA.meta,
            grafo=INSTANCIA.grafo),
    "AUDITORIA DE LA INSTANCIA INDIVIDUAL")
print()

# --- Camino de referencia ---------------------------------------------------
# Obtenido con `caminos_simples`, la utilidad de VERIFICACION, no con un
# buscador (D-07). Como el laberinto es un arbol, este camino es el UNICO
# camino simple: todo algoritmo completo de la Version 1 debe devolver
# exactamente esta secuencia. La seccion 8 lo usara como oraculo.
CAMINO_REFERENCIA = caminos_simples(INSTANCIA.grafo, INSTANCIA.inicio,
                                    INSTANCIA.meta, tope=1)[0]
print("Camino de referencia: {0} celdas, profundidad {1}".format(
    len(CAMINO_REFERENCIA), len(CAMINO_REFERENCIA) - 1))

_marcas = marcas_de_camino(CAMINO_REFERENCIA)
_marcas[INSTANCIA.inicio] = "I"
_marcas[INSTANCIA.meta] = "M"
print(dibujar_laberinto(INSTANCIA.grafo, INSTANCIA.filas, INSTANCIA.columnas,
                        _marcas, ancho=1))

# --- El grafo realmente es de solo lectura ----------------------------------
try:
    INSTANCIA.grafo[INSTANCIA.inicio].add(INSTANCIA.meta)
except AttributeError:
    print("\nIntento de modificar el grafo compartido: rechazado, como se espera.")
else:
    raise AssertionError("el grafo de la instancia deberia ser inmutable")

## 3. Formulación formal del problema

Defina rigurosamente:

| Componente | Especificación |
|---|---|
| Conjunto de estados | S = {(f, c) ∈ ℤ² : 0 ≤ f < 20, 0 ≤ c < 25}, es decir, las 500 claves del grafo. Cada estado es una tupla inmutable de dos enteros (D-08). |
| Estado inicial | s₀ = (3, 4) |
| Acciones aplicables | ACCIONES(s) = { a ∈ ⟨N, E, S, O⟩ : s + δ(a) ∈ grafo[s] }, devueltas siempre en el orden N, E, S, O (D-14). Una dirección con pared **no** es aplicable. |
| Modelo de transición | RESULTADO(s, a) = s + δ(a), con δ(N) = (−1, 0), δ(E) = (0, +1), δ(S) = (+1, 0), δ(O) = (0, −1). Solo está definido si a ∈ ACCIONES(s); en otro caso es un error, no una acción que deja el estado igual. |
| Prueba de objetivo | ES_META(s) ⇔ s = (16, 20) |
| Costo de paso | c(s, a, s′) > 0. En el laberinto perfecto (secciones 1 a 8) c = 1 para todo paso. La sección 9 inyecta otra función de costo sin cambiar esta formulación. |
| Costo de una solución | Para s₀ →a₁→ s₁ → … →aₙ→ sₙ con ES_META(sₙ): g = Σᵢ c(sᵢ₋₁, aᵢ, sᵢ). Con costo unitario, g = n = profundidad. |

Las **acciones son direcciones**, no celdas destino. Si fueran celdas, la
secuencia `acciones` del contrato (sección 4) repetiría `camino[1:]` sin aportar
información; como direcciones son la descripción del movimiento, y el camino es
su consecuencia.

Además:

1. diferencie **estado**, **nodo de búsqueda** y **celda dibujada**;
2. indique cuándo dos nodos distintos pueden contener el mismo estado;
3. explique por qué debe usarse búsqueda en grafo;
4. establezca invariantes que deben cumplirse durante una búsqueda.

### 3.1 Estado, nodo de búsqueda y celda dibujada

Son tres objetos distintos que en este problema es fácil confundir porque los tres «hablan» de la misma casilla.

| Objeto | Qué es | Ejemplo en la instancia |
|---|---|---|
| **Estado** | La tupla `(fila, columna)`. Es toda la información necesaria para decidir qué se puede hacer después: la historia de cómo se llegó no cambia las acciones disponibles. | `(3, 4)` |
| **Nodo de búsqueda** | Estructura interna del algoritmo: *estado + padre + acción que lo produjo + g(n) + profundidad*. Representa **un camino** desde el inicio hasta ese estado. | nodo con estado `(3, 3)`, padre = nodo de `(3, 4)`, acción `O`, g = 1, profundidad 1 |
| **Celda dibujada** | Tres caracteres de texto rodeados de `+`, `-` y `\|`. Es solo una vista para leer el grafo; ningún algoritmo la consulta, y sus paredes no se almacenan: una pared es la **ausencia** de una arista. | `\|* I\|` en la fila 3 del dibujo |

Hay **un estado por celda**, pero **puede haber varios nodos por estado**: el estado dice *dónde* estoy; el nodo dice además *por dónde vine*.

### 3.2 Cuándo dos nodos distintos contienen el mismo estado

Cuando existen **dos caminos distintos** desde el inicio hasta ese estado; cada camino es un nodo diferente.

- **En el laberinto perfecto** hay un único camino *simple* entre dos celdas (comprobación 8), así que la única forma de repetir un estado es un camino **no simple**: ir y volver. Desde `(3, 4)` la acción `O` lleva a `(3, 3)` y la acción `E` desde ahí devuelve a `(3, 4)`. El nodo raíz y ese nieto tienen el mismo estado, profundidades 0 y 2.
- **En el laberinto con ciclos** de la sección 9 aparece el caso interesante: dos caminos *simples* distintos al mismo estado, con costos distintos. Ahí decidir cuál conservar es lo que hace óptimo (o no) a un algoritmo.

### 3.3 Por qué debe usarse búsqueda en grafo

El grafo es **no dirigido**: toda acción es reversible. Por eso el árbol de búsqueda es **infinito** aunque el espacio de estados tenga solo 500 estados: `s0 → A → s0 → A → …` nunca se acaba.

- Una **DFS en árbol** puede quedar atrapada en ese vaivén y no terminar nunca: **pierde la completitud**.
- Una **BFS en árbol** sí termina, pero repite trabajo de forma exponencial. La tabla medida en la celda siguiente, desde el inicio `(3, 4)`, lo muestra: a profundidad 20 el árbol tiene **3 004 626 nodos** en ese solo nivel, y todos corresponden a **56 estados** distintos. La meta está a profundidad 59.

La búsqueda en grafo guarda el conjunto de estados **alcanzados** y no vuelve a expandir ninguno. Eso acota el trabajo a O(|V| + |E|) = O(500 + 499) y devuelve la completitud a DFS.

### 3.4 Invariantes que debe cumplir toda búsqueda

Se enuncian aquí; la sección 8 los convierte en pruebas.

1. **Validez de estados.** Todo estado que entra en la frontera o en *alcanzados* es una clave del grafo.
2. **Validez de caminos.** Siguiendo los padres desde cualquier nodo se llega al inicio, y cada par padre→hijo es una arista del grafo. Equivale a `validar_solucion` aplicada al camino de ese nodo.
3. **Coherencia de costos.** `g(n)` es la suma de los costos de paso del camino de `n`, y `profundidad(n)` su número de acciones. Con costo unitario, ambos coinciden.
4. **Expansión única.** Ningún estado se expande dos veces. En UCS una entrada de la frontera cuyo `g` ya fue superado es **obsoleta** y se descarta al extraerla, sin expandirla.
5. **Orden de extracción.** En BFS las profundidades extraídas no decrecen; en UCS los valores `g` extraídos no decrecen.
6. **Cotas de las métricas.** `expandidos ≤ |V|`, `repetidos_descartados ≤ generados`, y el grafo compartido no se modifica (se garantiza congelándolo, D-13).
7. **Solución.** Si `encontrado` es verdadero, el camino empieza en el inicio, termina en la meta y tiene exactamente una acción menos que estados. Si es falso, el camino es vacío.

In [ ]:
# --------------------------------------------------------------------------
# Seccion 3 - La formulacion formal, escrita como codigo
#
# La tabla de arriba se traduce a una clase con los cinco componentes del
# problema de busqueda (AIMA cap. 3). Las tres versiones usaran ESTA
# definicion: SimpleAI y AIMA-Python recibiran adaptadores delgados sobre
# ella, no una segunda formulacion escrita a mano que pueda discrepar.
# --------------------------------------------------------------------------

# Orden FIJO de las acciones (D-14). No se itera `grafo[estado]`: es un
# conjunto y su orden, aunque reproducible, depende de la historia interna
# del generador y no se puede explicar. Con un orden declarado, "DFS prueba
# primero el norte" es una afirmacion verificable, y las tres versiones
# desempatan igual.
ACCIONES = ("N", "E", "S", "O")
DESPLAZAMIENTO = {"N": (-1, 0), "E": (0, 1), "S": (1, 0), "O": (0, -1)}


def desplazar(estado, accion):
    """Celda geometricamente contigua; NO consulta si hay pared."""
    df, dc = DESPLAZAMIENTO[accion]
    return (estado[0] + df, estado[1] + dc)


def costo_unitario(estado, accion, siguiente):
    return 1


class ProblemaLaberinto:
    """Problema de busqueda sobre un laberinto dado como grafo.

    estados        celdas (fila, columna) del grafo
    ACCIONES(s)    direcciones con corredor abierto desde s, en orden N-E-S-O
    RESULTADO(s,a) la celda contigua en la direccion a
    ES_META(s)     s == meta
    c(s, a, s')    costo del paso; 1 por omision (la seccion 9 inyecta otro)
    """

    def __init__(self, grafo, inicio, meta, costo_paso=costo_unitario):
        if inicio not in grafo or meta not in grafo:
            raise ValueError("inicio y meta deben ser celdas del grafo")
        self.grafo = grafo
        self.estado_inicial = inicio
        self.meta = meta
        self._costo_paso = costo_paso

    @classmethod
    def desde_instancia(cls, instancia, costo_paso=costo_unitario):
        return cls(instancia.grafo, instancia.inicio, instancia.meta, costo_paso)

    def acciones(self, estado):
        vecinos = self.grafo[estado]
        return tuple(a for a in ACCIONES if desplazar(estado, a) in vecinos)

    def resultado(self, estado, accion):
        siguiente = desplazar(estado, accion)
        # Una accion no aplicable es un error del buscador, no un "no pasa
        # nada": devolver `estado` ocultaria el fallo y crearia lazos.
        if siguiente not in self.grafo[estado]:
            raise ValueError("accion {0} no aplicable en {1}: hay pared".format(
                accion, estado))
        return siguiente

    def es_meta(self, estado):
        return estado == self.meta

    def costo_paso(self, estado, accion, siguiente):
        costo = self._costo_paso(estado, accion, siguiente)
        if not costo > 0:
            raise ValueError("costo de paso no positivo: {0}".format(costo))
        return costo


def validar_solucion(problema, camino, acciones):
    """Comprueba que (camino, acciones) sea una solucion del problema.

    Es la forma EJECUTABLE de las condiciones de la tabla: la seccion 8 la
    aplicara al resultado de cada algoritmo. Devuelve el costo de la solucion.
    """
    assert camino, "el camino no puede ser vacio si hay solucion"
    assert camino[0] == problema.estado_inicial, "el camino no empieza en el inicio"
    assert problema.es_meta(camino[-1]), "el camino no termina en la meta"
    assert len(acciones) == len(camino) - 1, "debe haber una accion por paso"
    costo = 0
    for estado, accion, siguiente in zip(camino, acciones, camino[1:]):
        assert accion in problema.acciones(estado), \
            "{0} no es aplicable en {1}".format(accion, estado)
        assert problema.resultado(estado, accion) == siguiente, \
            "{0} desde {1} no lleva a {2}".format(accion, estado, siguiente)
        costo += problema.costo_paso(estado, accion, siguiente)
    return costo


def acciones_de_camino(problema, camino):
    """Traduce una secuencia de estados a la secuencia de acciones."""
    return tuple(next(a for a in problema.acciones(u) if desplazar(u, a) == v)
                 for u, v in zip(camino, camino[1:]))


PROBLEMA = ProblemaLaberinto.desde_instancia(INSTANCIA)

# --- Los componentes, evaluados sobre la instancia real ---------------------
_s0 = PROBLEMA.estado_inicial
print("Estado inicial        ", _s0)
print("ACCIONES(s0)          ", PROBLEMA.acciones(_s0))
for _a in PROBLEMA.acciones(_s0):
    print("RESULTADO(s0, {0})      {1}".format(_a, PROBLEMA.resultado(_s0, _a)))
print("ES_META(s0)           ", PROBLEMA.es_meta(_s0))
print("ES_META(meta)         ", PROBLEMA.es_meta(PROBLEMA.meta))

# Una accion geometricamente posible pero bloqueada por una pared. Se busca
# primero en s0 y, si s0 no tiene ninguna (inicio cambiado en la defensa),
# en la primera celda que la tenga.
_con_pared, _bloqueada = next(
    (s, a) for s in [_s0] + sorted(PROBLEMA.grafo) for a in ACCIONES
    if a not in PROBLEMA.acciones(s) and desplazar(s, a) in PROBLEMA.grafo)
try:
    PROBLEMA.resultado(_con_pared, _bloqueada)
except ValueError as error:
    print("RESULTADO({0}, {1}) rechazada -> {2}".format(
        _con_pared, _bloqueada, error))

# --- El camino de referencia como solucion formal ---------------------------
ACCIONES_REFERENCIA = acciones_de_camino(PROBLEMA, CAMINO_REFERENCIA)
COSTO_REFERENCIA = validar_solucion(PROBLEMA, CAMINO_REFERENCIA,
                                    ACCIONES_REFERENCIA)
print()
print("Solucion de referencia: {0} acciones, costo {1}".format(
    len(ACCIONES_REFERENCIA), COSTO_REFERENCIA))
print("  " + "".join(ACCIONES_REFERENCIA))

# El validador debe poder FALLAR (D-11): una solucion alterada se rechaza.
# Se invierte la primera accion: o no es aplicable, o lleva a otra celda.
_OPUESTA = {"N": "S", "S": "N", "E": "O", "O": "E"}
_alterada = list(ACCIONES_REFERENCIA)
_alterada[0] = _OPUESTA[_alterada[0]]
try:
    validar_solucion(PROBLEMA, CAMINO_REFERENCIA, _alterada)
except AssertionError as error:
    print("Contra-prueba: solucion alterada rechazada -> {0}".format(error))
else:
    raise AssertionError("el validador acepto una solucion invalida")

# --- Por que busqueda en GRAFO: evidencia medida ----------------------------
# Una busqueda en ARBOL (sin conjunto de alcanzados) trata como distintos los
# nodos s0 -> A -> s0 -> A ... aunque repitan estado. Se cuentan los nodos
# que tendria el arbol de busqueda en cada profundidad (caminatas desde s0)
# frente a los estados distintos que realmente existen a esa distancia.
print()
print("Nodos de un arbol de busqueda sin control de repetidos:")
print("  prof.   nodos en ese nivel   estados distintos a distancia <= prof.")
_nivel = {_s0: 1}
_alcanzados = {_s0}
for _prof in range(1, 21):
    _siguiente = {}
    for _estado, _veces in _nivel.items():
        for _vecino in PROBLEMA.grafo[_estado]:
            _siguiente[_vecino] = _siguiente.get(_vecino, 0) + _veces
    _nivel = _siguiente
    _alcanzados |= set(_nivel)
    if _prof % 5 == 0:
        print("  {0:>5}   {1:>18,}   {2:>10}".format(
            _prof, sum(_nivel.values()), len(_alcanzados)))

## 4. Contrato común de las tres versiones

Todos los algoritmos deben entregar un registro equivalente a:

```text
ResultadoBusqueda
    encontrado: booleano
    camino: secuencia de estados
    acciones: secuencia de acciones
    costo: número
    profundidad: entero
    expandidos: entero
    generados: entero
    repetidos_descartados: entero
    frontera_maxima: entero
    tiempo_ms: número
```

### Convenciones obligatorias

- Un estado se cuenta como **expandido** al retirarlo de la frontera y generar sus sucesores.
- Un nodo meta retirado de la frontera no se cuenta como expandido si no se generan sucesores.
- **Generado** significa nodo sucesor construido, aunque luego sea descartado.
- La política para estados repetidos debe documentarse.
- El camino debe incluir inicio y meta.
- Si no hay solución, `camino` debe ser vacío y `encontrado` falso.

Estas convenciones son necesarias para comparar implementaciones de manera justa.

### 4.1 Decisiones del contrato

- **`ResultadoBusqueda` es inmutable** (`dataclass(frozen=True)`). Un resultado ya devuelto no puede alterarse al imprimirlo o compararlo, así que la tabla de la sección 10 siempre refleja lo que midió el algoritmo.
- **Sin solución:** `encontrado = False`, `camino = ()`, `acciones = ()`, `costo = ∞` y `profundidad = −1`. Se eligen valores **imposibles** para una solución real, y no `0`, porque `0` es la profundidad legítima del caso *inicio = meta* (D-20).
- **El nodo inicial no cuenta como generado.** El enunciado define *generado* como «nodo **sucesor** construido», y la raíz no es sucesor de nadie. Así, en el caso *inicio = meta*, el resultado es `expandidos = generados = 0`.
- **`tiempo_ms`** mide todo lo que hace la búsqueda, incluida la reconstrucción del camino, con `time.perf_counter`. No incluye construir el problema.
- **El `Nodo` guarda una referencia al padre, no el camino.** Ver la pregunta 4 de la sección 5.
- Se añaden dos campos con valor por omisión, que no cambian el contrato. `algoritmo` sirve para rotular las tablas. `corte` distingue, en la búsqueda limitada, un **corte** (no hay solución *dentro del límite*) de un **fracaso** definitivo (D-25). Los demás algoritmos lo dejan en `False`.

`verificar_resultado` comprueba estas reglas sobre cualquier resultado. La sección 8 la aplica a los algoritmos de las tres versiones.

In [ ]:
# --------------------------------------------------------------------------
# Seccion 4 - Contrato comun: ResultadoBusqueda y Nodo
#
# Todo algoritmo, de cualquier version, DEVUELVE un ResultadoBusqueda. Nada
# se imprime desde dentro de un buscador: quien llama decide que mostrar.
# --------------------------------------------------------------------------

import math
import time


@dataclass(frozen=True)
class ResultadoBusqueda:
    """Registro comun de las tres versiones (seccion 4 del enunciado).

    Sin solucion: encontrado=False, camino=() y acciones=(), costo=inf y
    profundidad=-1. Se usan valores IMPOSIBLES para una solucion real, y no
    0, porque 0 es la profundidad legitima del caso inicio == meta (D-20).

    `corte` solo tiene sentido en la busqueda limitada: True significa "no
    hay solucion DENTRO DEL LIMITE, pero algun nodo quedo sin explorar por
    el limite"; False con encontrado=False es un fracaso DEFINITIVO (D-25).
    """
    encontrado: bool
    camino: tuple
    acciones: tuple
    costo: float
    profundidad: int
    expandidos: int
    generados: int
    repetidos_descartados: int
    frontera_maxima: int
    tiempo_ms: float
    algoritmo: str = ""
    corte: bool = False


class Nodo:
    """Nodo del arbol de busqueda: un estado MAS la forma de llegar a el.

    Guarda solo una REFERENCIA al padre, no el camino completo. Los nodos
    comparten sus prefijos como las ramas de un arbol: cada nodo cuesta
    memoria constante y el camino se reconstruye una sola vez, al final.
    Copiar el camino en cada hijo costaria O(d) por nodo.

    `__slots__` evita un diccionario por instancia: una busqueda crea miles.
    """
    __slots__ = ("estado", "padre", "accion", "g", "profundidad")

    def __init__(self, estado, padre=None, accion=None, g=0):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.g = g
        self.profundidad = 0 if padre is None else padre.profundidad + 1

    def hijo(self, problema, accion):
        """Construye el sucesor. Aqui se aplican RESULTADO y c(s, a, s')."""
        siguiente = problema.resultado(self.estado, accion)
        costo = problema.costo_paso(self.estado, accion, siguiente)
        return Nodo(siguiente, self, accion, self.g + costo)

    def __repr__(self):
        return "Nodo({0}, g={1}, prof={2})".format(self.estado, self.g,
                                                   self.profundidad)


def reconstruir(nodo):
    """Sigue los padres desde `nodo` hasta la raiz: O(d), una sola vez."""
    estados, acciones = [], []
    while nodo is not None:
        estados.append(nodo.estado)
        if nodo.accion is not None:
            acciones.append(nodo.accion)
        nodo = nodo.padre
    estados.reverse()
    acciones.reverse()
    return tuple(estados), tuple(acciones)


class Metricas:
    """Contadores de una busqueda, separados del algoritmo que los usa."""
    __slots__ = ("expandidos", "generados", "repetidos_descartados",
                 "frontera_maxima", "_inicio")

    def __init__(self):
        self.expandidos = 0
        self.generados = 0
        self.repetidos_descartados = 0
        self.frontera_maxima = 0
        self._inicio = time.perf_counter()

    def observar_frontera(self, tamano):
        if tamano > self.frontera_maxima:
            self.frontera_maxima = tamano

    def resultado(self, nodo_meta, algoritmo, corte=False):
        """Cierra el cronometro y arma el ResultadoBusqueda."""
        if nodo_meta is None:
            return self.resultado_de_camino((), (), math.inf, algoritmo, corte)
        camino, acciones = reconstruir(nodo_meta)
        return self.resultado_de_camino(camino, acciones, nodo_meta.g, algoritmo)

    def resultado_de_camino(self, camino, acciones, costo, algoritmo,
                            corte=False):
        """Variante para caminos que no salen de un unico nodo: la busqueda
        bidireccional une dos mitades."""
        tiempo_ms = (time.perf_counter() - self._inicio) * 1000.0
        encontrado = len(camino) > 0
        return ResultadoBusqueda(encontrado, tuple(camino), tuple(acciones),
                                 costo, len(camino) - 1 if encontrado else -1,
                                 self.expandidos, self.generados,
                                 self.repetidos_descartados,
                                 self.frontera_maxima, tiempo_ms, algoritmo,
                                 corte and not encontrado)


def verificar_resultado(problema, r):
    """Coherencia interna de un ResultadoBusqueda frente al problema.

    La seccion 8 la aplica a todos los algoritmos de las tres versiones.
    """
    if not r.encontrado:
        assert r.camino == () and r.acciones == (), "fracaso con camino"
        assert r.costo == math.inf and r.profundidad == -1
    else:
        costo = validar_solucion(problema, r.camino, r.acciones)
        assert r.costo == costo, "costo reportado {0} != {1}".format(r.costo, costo)
        assert r.profundidad == len(r.camino) - 1
        assert not r.corte, "una solucion no puede ser a la vez un corte"
    # No se acota `expandidos` por |V|: IDDFS re-expande estados entre
    # iteraciones de forma legitima. Esa cota es propia de la busqueda en
    # grafo y se comprueba alli.
    assert 0 <= r.repetidos_descartados <= r.generados
    assert r.frontera_maxima >= 1, "la frontera arranca con el nodo inicial"
    return True

## 5. Pseudocódigo base: búsqueda en grafo

El siguiente esquema no especifica la estructura de la frontera ni resuelve el manejo de costos. Debe especializarlo y justificar cada decisión.

```text
BUSQUEDA-GRAFO(problema, frontera):
    nodo_inicial ← crear_nodo(problema.estado_inicial)
    insertar(frente, nodo_inicial)
    alcanzados ← estructura apropiada

    mientras frontera no esté vacía:
        nodo ← extraer_segun_politica(frontera)

        si ES_META(nodo.estado):
            devolver RECONSTRUIR(nodo)

        si nodo debe expandirse:
            para cada acción aplicable:
                hijo ← construir sucesor
                decidir si insertar, reemplazar o descartar hijo

        actualizar métricas

    devolver FRACASO
```

### Preguntas de diseño

1. ¿Cuándo debe marcarse un estado: al generarlo o al expandirlo?
2. ¿La respuesta cambia entre BFS, DFS y UCS?
3. ¿Qué información mínima debe almacenar cada nodo?
4. ¿Cómo se reconstruye el camino sin copiar listas completas en cada nodo?
5. ¿Cómo se resuelve un camino más barato hacia un estado ya descubierto?

### 5.1 Respuestas a las preguntas de diseño

**1. ¿Cuándo marcar un estado: al generarlo o al expandirlo?**
Son dos políticas válidas con costos distintos:

| | Marcar **al generar** | Marcar **al expandir** |
|---|---|---|
| Un estado entra a la frontera | a lo sumo una vez | varias veces (una por cada padre que lo genere) |
| Tamaño de la frontera | ≤ \|V\| | hasta \|E\| |
| Qué nodo se queda con el estado | el **primero** que lo descubre | el primero que se **extrae** |

En BFS y DFS se marca **al generar** (D-19). La prueba de objetivo se hace **al extraer**, que es la convención del enunciado, en todos los algoritmos.

**2. ¿Cambia entre BFS, DFS y UCS?** Sí, y la razón es distinta en cada caso:

- **BFS:** marcar al generar es **seguro**. La cola saca los nodos en orden no decreciente de profundidad, así que el primer camino que descubre un estado ya es de profundidad mínima. Descartar los posteriores no pierde nada.
- **DFS:** marcar al generar mantiene la frontera en ≤ |V|. El precio es que, **en un grafo con ciclos**, el recorrido deja de coincidir con el de una DFS recursiva: un estado descubierto como hermano de un nodo poco profundo ya no se visita por un camino más hondo. No afecta la completitud, y DFS no es óptima de todos modos. **En el laberinto perfecto las dos políticas coinciden**, porque cada estado tiene un único camino desde el inicio.
- **UCS:** marcar al generar **rompe la optimalidad**. El primer camino que descubre un estado no tiene por qué ser el más barato. UCS lleva un `mejor_costo[s]` y acepta reinsertar un estado si encuentra un camino más barato (etapa 4).

**3. Información mínima de un nodo:** `estado`, `padre` y `g`. Con eso se reconstruye el camino y se ordena UCS. También se guarda la `accion`, porque el contrato exige la secuencia de acciones y recuperarla desde dos estados costaría recorrer las acciones del padre. Además se guarda la `profundidad`, que la búsqueda limitada consulta en cada nodo. Con costo unitario es igual a `g`, pero con costos variables no.

**4. Reconstruir el camino sin copiar listas:** cada nodo guarda una **referencia** a su padre. Los nodos forman un árbol que comparte prefijos: los 1 000 nodos que cuelgan de un mismo antecesor no copian su camino, lo comparten. Al encontrar la meta se siguen los padres hasta la raíz y se invierte la lista. Cuesta O(d) una sola vez. Copiar el camino en cada hijo costaría O(d) **por nodo** en tiempo y en memoria.

**5. Un camino más barato hacia un estado ya descubierto:** solo importa en UCS. Si `g(nodo) + c < mejor_costo[s]`, se actualiza `mejor_costo[s]` y se **inserta una entrada nueva** en la cola de prioridad. La entrada vieja no se busca ni se borra: `heapq` no ofrece *decrease-key*, y buscarla costaría O(n). Queda en la cola como **entrada obsoleta** y, cuando sale, se reconoce porque su `g` es mayor que `mejor_costo[s]`, así que se descarta sin expandirla. Se implementa en la etapa 4.

# Versión 1 — Implementación desde cero

No puede utilizar bibliotecas de búsqueda o grafos. Se permiten únicamente estructuras estándar como listas, diccionarios, conjuntos, `deque` y colas de prioridad.

Debe implementar:

1. DFS iterativa;
2. BFS;
3. búsqueda de costo uniforme;
4. búsqueda limitada en profundidad;
5. profundización iterativa;
6. búsqueda bidireccional;
7. Lee o expansión por frente de onda.

La versión debe separar, como mínimo:

- representación del problema;
- representación de nodos;
- política de frontera;
- control de repetidos;
- reconstrucción del camino;
- recolección de métricas.

## 6. Pseudocódigo que debe traducirse

### DFS y BFS

```text
inicializar frontera con el nodo inicial
inicializar alcanzados

mientras haya nodos:
    extraer según disciplina LIFO o FIFO
    comprobar objetivo
    expandir
    insertar estados nuevos
```

La diferencia no debe quedar dispersa por todo el programa. Diseñe una abstracción que permita cambiar la política de frontera.

### Costo uniforme

```text
frontera ← cola de prioridad ordenada por g(n)
mejor_costo[inicial] ← 0

mientras frontera no esté vacía:
    extraer nodo con menor g
    ignorar entradas obsoletas
    comprobar objetivo
    para cada sucesor:
        nuevo_costo ← g(nodo) + costo_de_paso
        si mejora el costo conocido:
            actualizar costo, padre y frontera
```

### Profundización iterativa

```text
para límite = 0, 1, 2, ...:
    resultado ← BUSQUEDA-LIMITADA(problema, límite)
    si resultado es solución: devolverlo
    si resultado es fracaso definitivo: terminar
```

Debe distinguir **corte** de **fracaso**.

### Bidireccional

```text
crear una frontera desde inicio y otra desde meta
expandir de forma equilibrada
detectar intersección entre regiones alcanzadas
unir los dos caminos respetando su orientación
```

No se acepta ejecutar dos búsquedas completas y comparar al final.

In [ ]:
# ===========================================================================
# VERSION 1 - Desde cero.  Parte 1: politica de frontera, nucleo, DFS y BFS
#
# Separacion exigida por el enunciado, y donde vive cada cosa:
#   representacion del problema ... ProblemaLaberinto        (seccion 3)
#   representacion de nodos ....... Nodo                     (seccion 4)
#   politica de frontera .......... FronteraFIFO / FronteraLIFO   (aqui)
#   control de repetidos .......... `alcanzados` en busqueda_en_grafo (aqui)
#   reconstruccion del camino ..... reconstruir              (seccion 4)
#   recoleccion de metricas ....... Metricas                 (seccion 4)
#
# BFS y DFS son EL MISMO algoritmo con distinta frontera (D-17). La
# diferencia entre ambos esta en estas dos clases y en ningun otro sitio.
# ===========================================================================


class FronteraFIFO:
    """Cola: sale primero el nodo que entro primero -> BFS."""
    nombre = "FIFO"

    def __init__(self):
        self._nodos = deque()

    def agregar_hijos(self, hijos):
        self._nodos.extend(hijos)

    def extraer(self):
        return self._nodos.popleft()

    def __len__(self):
        return len(self._nodos)

    def estados(self):
        """En orden de salida. Solo para trazas."""
        return [n.estado for n in self._nodos]


class FronteraLIFO:
    """Pila: sale primero el nodo que entro ultimo -> DFS.

    Los hijos se apilan en orden INVERSO para que el primero en salir sea el
    de la primera accion (N). Asi DFS explora en el orden declarado N-E-S-O,
    igual que lo haria una DFS recursiva (D-18). Sin invertir, una pila
    explora O primero y el orden de ACCIONES dejaria de significar nada.
    """
    nombre = "LIFO"

    def __init__(self):
        self._nodos = []

    def agregar_hijos(self, hijos):
        self._nodos.extend(reversed(hijos))

    def extraer(self):
        return self._nodos.pop()

    def __len__(self):
        return len(self._nodos)

    def estados(self):
        """En orden de salida (tope primero). Solo para trazas."""
        return [n.estado for n in reversed(self._nodos)]


def busqueda_en_grafo(problema, frontera, algoritmo, traza=None):
    """Nucleo comun de BFS y DFS: busqueda en grafo con frontera inyectada.

    Politica de repetidos (D-19): un estado se MARCA AL GENERARLO. Todo hijo
    cuyo estado ya fue alcanzado se descarta y cuenta en
    `repetidos_descartados`. En consecuencia cada estado entra a la frontera
    a lo sumo una vez y se expande a lo sumo una vez.

    Prueba de objetivo AL EXTRAER, no al generar: es la convencion del
    enunciado y la que permite comparar con UCS en igualdad de condiciones.

    `traza`: si es una lista, se le agrega una fotografia por iteracion.
    """
    metricas = Metricas()
    raiz = Nodo(problema.estado_inicial)
    frontera.agregar_hijos([raiz])
    alcanzados = {raiz.estado}
    metricas.observar_frontera(len(frontera))

    while len(frontera):
        nodo = frontera.extraer()

        if problema.es_meta(nodo.estado):
            # La meta retirada NO se cuenta como expandida: no se generan
            # sus sucesores (convencion del enunciado).
            if traza is not None:
                traza.append({"extraido": nodo.estado, "meta": True,
                              "nuevos": [], "repetidos": [],
                              "frontera": frontera.estados()})
            return metricas.resultado(nodo, algoritmo)

        metricas.expandidos += 1
        nuevos, repetidos = [], []
        for accion in problema.acciones(nodo.estado):
            hijo = nodo.hijo(problema, accion)
            metricas.generados += 1
            if hijo.estado in alcanzados:
                metricas.repetidos_descartados += 1
                repetidos.append(hijo.estado)
            else:
                alcanzados.add(hijo.estado)
                nuevos.append(hijo)
        frontera.agregar_hijos(nuevos)
        metricas.observar_frontera(len(frontera))

        if traza is not None:
            traza.append({"extraido": nodo.estado, "meta": False,
                          "nuevos": [h.estado for h in nuevos],
                          "repetidos": repetidos,
                          "frontera": frontera.estados()})

    return metricas.resultado(None, algoritmo)


def bfs(problema, traza=None):
    return busqueda_en_grafo(problema, FronteraFIFO(), "BFS", traza)


def dfs(problema, traza=None):
    """DFS ITERATIVA, como exige el enunciado: pila explicita, sin recursion.

    Una DFS recursiva sobre la instancia podria bajar a profundidad ~500 y
    rozar el limite de recursion de Python (1000 por omision). Con una pila
    propia la profundidad solo esta limitada por la memoria.
    """
    return busqueda_en_grafo(problema, FronteraLIFO(), "DFS", traza)


# ---------------------------------------------------------------------------
# Ejecucion sobre la instancia individual
# ---------------------------------------------------------------------------

def imprimir_resultados(resultados):
    """Tabla de metricas. Recibe ResultadoBusqueda, no recalcula nada."""
    print("{0:<9} {1:>5} {2:>6} {3:>6} {4:>10} {5:>10} {6:>10} {7:>12} {8:>9}".format(
        "alg.", "hallo", "prof.", "costo", "expandidos", "generados",
        "repetidos", "frontera_max", "tiempo_ms"))
    for r in resultados:
        print("{0:<9} {1:>5} {2:>6} {3:>6} {4:>10} {5:>10} {6:>10} {7:>12} {8:>9.3f}".format(
            r.algoritmo, "si" if r.encontrado else "no", r.profundidad,
            r.costo, r.expandidos, r.generados, r.repetidos_descartados,
            r.frontera_maxima, r.tiempo_ms))


RESULTADO_BFS = bfs(PROBLEMA)
RESULTADO_DFS = dfs(PROBLEMA)

print("INSTANCIA INDIVIDUAL: {0} -> {1}".format(PROBLEMA.estado_inicial,
                                               PROBLEMA.meta))
imprimir_resultados([RESULTADO_BFS, RESULTADO_DFS])

for _r in (RESULTADO_BFS, RESULTADO_DFS):
    verificar_resultado(PROBLEMA, _r)
    # En un arbol hay un unico camino simple (D-16): los dos DEBEN devolver
    # el camino de referencia, obtenido sin buscadores.
    assert _r.camino == CAMINO_REFERENCIA, _r.algoritmo + " se aparto del camino unico"
    # Busqueda en grafo: ningun estado se expande dos veces.
    assert _r.expandidos <= len(PROBLEMA.grafo)
    # Prediccion propia de un ARBOL con marcado al generar: cada nodo
    # expandido, salvo la raiz, genera exactamente un repetido (su padre).
    # Si esto fallara, o el laberinto tiene un ciclo, o el control de
    # repetidos esta mal.
    assert _r.repetidos_descartados == _r.expandidos - 1, _r.algoritmo
print()
print("Ambos devuelven el camino de referencia ({0} estados) y pasan "
      "verificar_resultado.".format(len(CAMINO_REFERENCIA)))
print("En un arbol la solucion no distingue a BFS de DFS; lo que cambia es el "
      "trabajo: BFS expande {0} estados y DFS {1}.".format(
          RESULTADO_BFS.expandidos, RESULTADO_DFS.expandidos))

# ---------------------------------------------------------------------------
# Casos limite (la seccion 8 los formaliza; aqui se comprueba el nucleo)
# ---------------------------------------------------------------------------
print()
print("CASOS LIMITE")

# 1) inicio == meta: solucion de profundidad 0 sin expandir nada.
_trivial = ProblemaLaberinto(INSTANCIA.grafo, INSTANCIA.meta, INSTANCIA.meta)
for _buscar in (bfs, dfs):
    _r = _buscar(_trivial)
    verificar_resultado(_trivial, _r)
    assert (_r.encontrado, _r.camino, _r.profundidad, _r.expandidos,
            _r.generados) == (True, (INSTANCIA.meta,), 0, 0, 0)
print("  inicio == meta : camino de 1 estado, 0 expandidos, 0 generados")

# 2) meta inalcanzable: se corta UN corredor del camino unico, en una copia.
#    El grafo compartido esta congelado (D-13); se construye otro. El corte
#    se ubica en la mitad del camino, sea cual sea su longitud, para que la
#    celda siga funcionando si en la defensa se cambian inicio o meta.
_k_corte = min(len(CAMINO_REFERENCIA) // 2, len(CAMINO_REFERENCIA) - 2)
CORREDOR_DE_CORTE = (CAMINO_REFERENCIA[_k_corte], CAMINO_REFERENCIA[_k_corte + 1])
_u, _v = CORREDOR_DE_CORTE
_cortado = {c: set(vs) for c, vs in INSTANCIA.grafo.items()}
_cortado[_u].discard(_v)
_cortado[_v].discard(_u)
_cortado = MappingProxyType({c: frozenset(vs) for c, vs in _cortado.items()})
_sin_salida = ProblemaLaberinto(_cortado, INSTANCIA.inicio, INSTANCIA.meta)
_componente = inundar(_cortado, INSTANCIA.inicio)
for _buscar in (bfs, dfs):
    _r = _buscar(_sin_salida)
    verificar_resultado(_sin_salida, _r)
    assert not _r.encontrado and _r.camino == ()
    # Sin solucion, la busqueda en grafo agota la componente del inicio:
    # expande exactamente los estados alcanzables, ni uno mas ni uno menos.
    assert _r.expandidos == len(_componente)
print("  meta inalcanzable (corredor {0}-{1} cortado): no hallado, expande "
      "los {2} estados de la componente del inicio".format(
          _u, _v, len(_componente)))

# ---------------------------------------------------------------------------
# Traza sobre un laberinto pequeno (evidencia de la traza manual)
# ---------------------------------------------------------------------------
# Rejilla 3x3 de la misma semilla, de (0, 0) a (2, 2). La traza es una LISTA
# DE DICCIONARIOS, no un texto: es evidencia verificable, y la traza hecha a
# mano en papel se contrasta contra ella.
_INST_3x3 = construir_instancia(INSTANCIA.semilla, 3, 3, (0, 0), (2, 2))
PROBLEMA_3x3 = ProblemaLaberinto.desde_instancia(_INST_3x3)
TRAZA_BFS_3x3, TRAZA_DFS_3x3 = [], []
_r_bfs_3x3 = bfs(PROBLEMA_3x3, TRAZA_BFS_3x3)
_r_dfs_3x3 = dfs(PROBLEMA_3x3, TRAZA_DFS_3x3)


def imprimir_traza(traza):
    print("  paso  extraido  nuevos (a la frontera)       repetidos     "
          "frontera tras el paso (proximo a salir primero)")
    for paso, t in enumerate(traza, 1):
        if t["meta"]:
            print("  {0:>4}  {1}    META: se detiene sin expandir".format(
                paso, t["extraido"]))
            continue
        print("  {0:>4}  {1}    {2:<28} {3:<13} {4}".format(
            paso, t["extraido"], " ".join(map(str, t["nuevos"])) or "-",
            " ".join(map(str, t["repetidos"])) or "-",
            " ".join(map(str, t["frontera"])) or "(vacia)"))


print()
print("LABERINTO 3x3 PARA LA TRAZA MANUAL (semilla {0})".format(INSTANCIA.semilla))
print(dibujar_laberinto(_INST_3x3.grafo, 3, 3))
for _celda in sorted(_INST_3x3.grafo):
    print("  ACCIONES{0} = {1}".format(_celda, PROBLEMA_3x3.acciones(_celda)))
print()
print("TRAZA BFS (frontera FIFO)")
imprimir_traza(TRAZA_BFS_3x3)
print()
print("TRAZA DFS (frontera LIFO)")
imprimir_traza(TRAZA_DFS_3x3)
print()
imprimir_resultados([_r_bfs_3x3, _r_dfs_3x3])

# ---------------------------------------------------------------------------
# Contra-experimento (D-11): la inversion de la pila NO es decorativa
# ---------------------------------------------------------------------------
# Misma DFS con una pila que apila los hijos en el orden de ACCIONES, sin
# invertir. Si la inversion no importara, la traza seria identica.
class _PilaSinInvertir(FronteraLIFO):
    def agregar_hijos(self, hijos):
        self._nodos.extend(hijos)


_traza_sin_invertir = []
busqueda_en_grafo(PROBLEMA_3x3, _PilaSinInvertir(), "DFS-sin-invertir",
                  _traza_sin_invertir)
_orden_con = [t["extraido"] for t in TRAZA_DFS_3x3]
_orden_sin = [t["extraido"] for t in _traza_sin_invertir]
print()
print("Orden de extraccion de DFS en la 3x3:")
print("  pila invertida (N-E-S-O primero): " + " ".join(map(str, _orden_con)))
print("  pila sin invertir (O-S-E-N)     : " + " ".join(map(str, _orden_sin)))
assert _orden_con != _orden_sin, "la inversion deberia cambiar el recorrido"
assert _orden_con[1] == PROBLEMA_3x3.resultado((0, 0), PROBLEMA_3x3.acciones((0, 0))[0])

### 6.2 Costo uniforme: eliminación perezosa de entradas obsoletas

UCS extrae siempre el nodo de menor `g(n)`. Con costos positivos, cuando un estado sale de la frontera ya no existe un camino más barato hacia él. Por eso la prueba de objetivo **tiene** que hacerse al extraer: cuando se genera la meta, todavía puede haber un camino más barato hacia ella en la frontera.

Cuando se encuentra un camino más barato hacia un estado que ya está en la frontera, `heapq` no permite rebajar su prioridad. Se **inserta una entrada nueva** y la vieja queda **obsoleta**. Al extraerla se reconoce porque su `g` es mayor que `mejor_costo[s]`, y se descarta sin expandirla (D-21). La cola puede llegar a tener O(|E|) entradas, y el costo total es O(|E| log |E|).

La celda siguiente comprueba tres cosas:

1. **Costo unitario, en la instancia individual:** UCS expande **los mismos estados y en el mismo orden** que BFS (D-22). En un árbol nunca aparece una entrada obsoleta.
2. **Un grafo ponderado de 2×4 con ciclos**, construido a mano: UCS encuentra el costo 12 y **extrae y descarta dos entradas obsoletas**. BFS, que minimiza pasos y no costo, se queda con un camino de costo 17.
3. **Contra-experimento:** una frontera ordenada por `g` con el control de repetidos de BFS (marcar al generar) devuelve una solución válida pero de costo 17. Ordenar la frontera no basta; hay que aceptar mejoras.

In [ ]:
# ===========================================================================
# VERSION 1 - Parte 2: busqueda de costo uniforme (UCS)
#
# UCS NO reutiliza `busqueda_en_grafo` (D-17). Alli un estado se descarta si
# ya fue alcanzado; aqui se acepta de nuevo si llega por un camino MAS
# BARATO. El contra-experimento del final muestra que pasa si se ignora
# esa diferencia.
# ===========================================================================

import heapq
import itertools


def ucs(problema, traza=None):
    """Busqueda de costo uniforme con eliminacion perezosa (D-21).

    - La frontera es un monticulo de tuplas (g, orden, nodo). `orden` es un
      contador creciente: desempata en orden de insercion (FIFO), es
      determinista y evita que heapq intente comparar dos Nodo.
    - `mejor_costo[s]` es el menor g conocido para s. Un hijo entra a la
      frontera solo si MEJORA ese valor.
    - Mejorar un estado que ya esta en la frontera no la modifica: se inserta
      una entrada nueva y la vieja queda OBSOLETA. Se reconoce al extraerla,
      porque su g supera a mejor_costo, y se descarta sin expandir.
    - Prueba de objetivo AL EXTRAER. Es obligatorio para la optimalidad: al
      generar la meta todavia puede existir un camino mas barato hacia ella.

    Metricas: una entrada obsoleta descartada cuenta en
    `repetidos_descartados`, igual que un hijo que no mejora. Cada hijo
    generado se descarta a lo sumo una vez, asi que repetidos <= generados.
    `frontera_maxima` incluye las entradas obsoletas: ocupan memoria real.
    """
    metricas = Metricas()
    orden = itertools.count()
    raiz = Nodo(problema.estado_inicial)
    frontera = [(raiz.g, next(orden), raiz)]
    mejor_costo = {raiz.estado: raiz.g}
    metricas.observar_frontera(len(frontera))

    while frontera:
        g, _, nodo = heapq.heappop(frontera)

        if g > mejor_costo[nodo.estado]:
            metricas.repetidos_descartados += 1
            if traza is not None:
                traza.append({"extraido": nodo.estado, "g": g,
                              "evento": "obsoleta", "empujados": []})
            continue

        if problema.es_meta(nodo.estado):
            if traza is not None:
                traza.append({"extraido": nodo.estado, "g": g,
                              "evento": "meta", "empujados": []})
            return metricas.resultado(nodo, "UCS")

        metricas.expandidos += 1
        empujados = []
        for accion in problema.acciones(nodo.estado):
            hijo = nodo.hijo(problema, accion)
            metricas.generados += 1
            if hijo.g < mejor_costo.get(hijo.estado, math.inf):
                mejor_costo[hijo.estado] = hijo.g
                heapq.heappush(frontera, (hijo.g, next(orden), hijo))
                empujados.append((hijo.estado, hijo.g))
            else:
                metricas.repetidos_descartados += 1
        metricas.observar_frontera(len(frontera))

        if traza is not None:
            traza.append({"extraido": nodo.estado, "g": g,
                          "evento": "expandido", "empujados": empujados})

    return metricas.resultado(None, "UCS")


# ---------------------------------------------------------------------------
# 1) Instancia individual, costo unitario
# ---------------------------------------------------------------------------
RESULTADO_UCS = ucs(PROBLEMA)
print("INSTANCIA INDIVIDUAL, costo unitario")
imprimir_resultados([RESULTADO_BFS, RESULTADO_UCS])
verificar_resultado(PROBLEMA, RESULTADO_UCS)
assert RESULTADO_UCS.camino == CAMINO_REFERENCIA

# Prediccion (D-22): con costo unitario, g coincide con la profundidad, y el
# desempate FIFO del contador reproduce el orden de la cola de BFS. UCS debe
# entonces expandir EXACTAMENTE los mismos estados, en el mismo orden.
_traza_bfs, _traza_ucs = [], []
bfs(PROBLEMA, _traza_bfs)
ucs(PROBLEMA, _traza_ucs)
_orden_bfs = [t["extraido"] for t in _traza_bfs]
_orden_ucs = [t["extraido"] for t in _traza_ucs if t["evento"] != "obsoleta"]
assert _orden_bfs == _orden_ucs, "UCS unitaria deberia recorrer como BFS"
assert RESULTADO_UCS.expandidos == RESULTADO_BFS.expandidos
# En un arbol no hay dos caminos a un mismo estado: nunca hay mejora, asi que
# nunca hay entradas obsoletas.
assert not any(t["evento"] == "obsoleta" for t in _traza_ucs)
print("UCS unitaria recorre los mismos {0} estados que BFS y en el mismo "
      "orden; 0 entradas obsoletas (en un arbol no hay mejoras).".format(
          RESULTADO_UCS.expandidos))

# ---------------------------------------------------------------------------
# 2) Grafo ponderado pequeno con ciclos: aqui UCS se distingue
# ---------------------------------------------------------------------------
# Construido a mano para producir DOS entradas obsoletas que llegan a
# extraerse. Rejilla 2x4; A = inicio, G = meta. Costos en las aristas.
#
#     A(0,0) -1- B(0,1) -1- C(0,2) -1- H(0,3)
#       |          |          |
#       5          7         10
#       |          |          |
#     D(1,0) -1- E(1,1) -1- F(1,2) -5- G(1,3)
#
# Camino barato: A-D-E-F-G = 5+1+1+5 = 12.  El de arriba: A-B-C-F-G = 17.
# Los costos se asocian a ARISTAS no dirigidas, por eso la clave es un
# frozenset: ir y volver por el mismo corredor cuesta lo mismo.
_COSTOS_PONDERADO = {
    frozenset({(0, 0), (0, 1)}): 1, frozenset({(0, 1), (0, 2)}): 1,
    frozenset({(0, 2), (0, 3)}): 1, frozenset({(0, 0), (1, 0)}): 5,
    frozenset({(0, 1), (1, 1)}): 7, frozenset({(0, 2), (1, 2)}): 10,
    frozenset({(1, 0), (1, 1)}): 1, frozenset({(1, 1), (1, 2)}): 1,
    frozenset({(1, 2), (1, 3)}): 5,
}
_adyacencia = {(f, c): set() for f in range(2) for c in range(4)}
for _arista in _COSTOS_PONDERADO:
    _p, _q = tuple(_arista)
    _adyacencia[_p].add(_q)
    _adyacencia[_q].add(_p)
GRAFO_PONDERADO = MappingProxyType({c: frozenset(v) for c, v in _adyacencia.items()})


def costo_por_arista(costos):
    """Fabrica una funcion c(s, a, s') a partir de costos por arista."""
    def costo(estado, accion, siguiente):
        return costos[frozenset((estado, siguiente))]
    return costo


PROBLEMA_PONDERADO = ProblemaLaberinto(GRAFO_PONDERADO, (0, 0), (1, 3),
                                       costo_por_arista(_COSTOS_PONDERADO))
TRAZA_UCS_PONDERADO = []
_r_ucs_p = ucs(PROBLEMA_PONDERADO, TRAZA_UCS_PONDERADO)
_r_bfs_p = bfs(PROBLEMA_PONDERADO)
for _r in (_r_ucs_p, _r_bfs_p):
    verificar_resultado(PROBLEMA_PONDERADO, _r)

print()
print("GRAFO PONDERADO 2x4 CON CICLOS: (0, 0) -> (1, 3)")
print("  paso  extraido  g     evento      empujados a la frontera (estado, g)")
for _paso, _t in enumerate(TRAZA_UCS_PONDERADO, 1):
    print("  {0:>4}  {1}   {2:<4}  {3:<10}  {4}".format(
        _paso, _t["extraido"], _t["g"], _t["evento"],
        " ".join("{0}:{1}".format(e, g) for e, g in _t["empujados"]) or "-"))
print()
imprimir_resultados([_r_bfs_p, _r_ucs_p])

_obsoletas = [t for t in TRAZA_UCS_PONDERADO if t["evento"] == "obsoleta"]
assert _r_ucs_p.costo == 12 and len(_obsoletas) == 2
# Ningun estado se expande dos veces, ni siquiera con entradas repetidas.
_expandidos = [t["extraido"] for t in TRAZA_UCS_PONDERADO if t["evento"] == "expandido"]
assert len(_expandidos) == len(set(_expandidos))
# Concordancia exigida por la seccion 8: costo(UCS) <= costo(BFS).
assert _r_ucs_p.costo <= _r_bfs_p.costo
print("UCS: costo {0} por {1}; {2} entradas obsoletas extraidas y descartadas "
      "({3}).".format(_r_ucs_p.costo, " ".join(map(str, _r_ucs_p.camino)),
                      len(_obsoletas),
                      ", ".join("{0} con g={1}".format(t["extraido"], t["g"])
                                for t in _obsoletas)))
print("BFS: costo {0}. Minimiza PASOS, no costo; aqui los dos caminos tienen "
      "4 pasos y BFS se queda con el primero que descubre.".format(_r_bfs_p.costo))

# ---------------------------------------------------------------------------
# 3) Contra-experimento (D-11): UCS con el control de repetidos de BFS
# ---------------------------------------------------------------------------
# Una frontera con prioridad por g, pero metida en `busqueda_en_grafo`, que
# marca al generar. Tiene el orden de extraccion de UCS y la politica de
# repetidos de BFS. Si el marcado al generar fuera inocuo, el costo seria 12.
class _FronteraPrioridad:
    def __init__(self):
        self._orden = itertools.count()
        self._monticulo = []

    def agregar_hijos(self, hijos):
        for hijo in hijos:
            heapq.heappush(self._monticulo, (hijo.g, next(self._orden), hijo))

    def extraer(self):
        return heapq.heappop(self._monticulo)[2]

    def __len__(self):
        return len(self._monticulo)

    def estados(self):
        return [n.estado for _, _, n in sorted(self._monticulo)]


_r_mal = busqueda_en_grafo(PROBLEMA_PONDERADO, _FronteraPrioridad(),
                           "UCS-marcando-al-generar")
verificar_resultado(PROBLEMA_PONDERADO, _r_mal)   # la solucion es VALIDA...
assert _r_mal.costo > _r_ucs_p.costo              # ...pero NO optima
print()
print("Contra-experimento: prioridad por g + marcado al generar devuelve "
      "costo {0} ({1}), no {2}.".format(
          _r_mal.costo, " ".join(map(str, _r_mal.camino)), _r_ucs_p.costo))
print("La frontera ordenada no basta: sin aceptar mejoras, (1,1) y (1,2) quedan "
      "fijados con el costo con que se descubren primero (8 y 12), y el camino "
      "barato por (1,0) nunca se aprovecha.")

### 6.3 Búsqueda limitada y profundización iterativa: corte frente a fracaso

La búsqueda limitada (DLS) es una DFS que no desciende más allá de `límite` pasos. Puede terminar de **tres** maneras, y confundir las dos últimas es el error que el enunciado pide evitar:

| Resultado | Significado | `encontrado` | `corte` |
|---|---|---|---|
| **Solución** | halló la meta a profundidad ≤ límite | `True` | `False` |
| **Corte** | no hay solución *dentro del límite*, pero algún nodo quedó sin explorar por el límite: con un límite mayor **podría** haberla | `False` | `True` |
| **Fracaso** | no hay solución y ningún nodo alcanzó el límite: la búsqueda agotó todo lo alcanzable y un límite mayor no cambiaría nada | `False` | `False` |

IDDFS repite DLS con límite 0, 1, 2, … y se detiene ante una **solución** o un **fracaso**. Un corte significa «sigue probando». Así IDDFS termina sin ningún tope externo aunque la meta sea inalcanzable (D-25).

**Control de repetidos por camino, no global (D-24).** DLS solo descarta los estados que están en el **camino actual** y los libera al retroceder. Un conjunto de alcanzados global, como el de BFS, sería **incorrecto**: un estado alcanzado primero por una rama profunda quedaría marcado, y cuando luego se llegara a él por un camino más corto ya no podría usarse. Las soluciones que pasan por él y que sí caben en el límite se perderían. La celda siguiente lo demuestra en una rejilla abierta de 2×4 con límite exacto 3.

**El costo de IDDFS en un laberinto.** El argumento clásico dice que re-expandir los niveles superiores es barato: con factor de ramificación *b* el último nivel domina y el sobrecosto es del orden de *b/(b−1)*. Pero un laberinto perfecto es casi un pasillo, con ramificación efectiva cercana a 1: el número de estados a profundidad ≤ L crece más o menos **linealmente** con L, y la suma de todas las iteraciones crece como L². En la instancia individual IDDFS expande unas **22 veces** lo que expande BFS. El argumento de *b/(b−1)* supone *b* > 1 holgado, y este laberinto no lo cumple.

In [ ]:
# ===========================================================================
# VERSION 1 - Parte 3: busqueda limitada en profundidad (DLS) y
#                      profundizacion iterativa (IDDFS)
#
# DLS es una busqueda en ARBOL con control de ciclos sobre el camino
# actual, no una busqueda en grafo (D-24). Devuelve tres resultados
# distintos: solucion, CORTE (quedo algo sin explorar por el limite) o
# FRACASO (no quedo nada: no hay solucion a ninguna profundidad) (D-25).
# ===========================================================================


def busqueda_limitada(problema, limite, control="camino", traza=None):
    """DLS iterativa con una pila de (nodo, acciones pendientes).

    Cada nivel de la pila es un nodo del camino actual junto con el
    iterador de sus acciones aun no probadas. Los hijos se generan de a uno,
    al descender, asi que la memoria es O(limite): la pila ES el camino.

    Repetidos: `en_camino` contiene solo los estados del camino actual. Un
    estado se quita al retroceder, de modo que puede volver a visitarse por
    otra rama. `control="global"` NO lo quita, y existe unicamente para el
    contra-experimento del final: muestra por que no sirve un conjunto de
    alcanzados global en una busqueda con limite.

    Metricas (D-24):
      expandidos  nodos que pasan a la pila para generar sus sucesores; un
                  nodo en el limite no se expande.
      frontera    la pila. Un nodo en el limite no entra en ella, asi que
                  frontera_maxima <= limite (o 1 si limite == 0).
    La prueba de objetivo se hace al VISITAR el nodo, que en esta
    formulacion coincide con retirarlo: se visita al generarlo, antes de
    generar a sus hermanos.
    """
    metricas = Metricas()
    raiz = Nodo(problema.estado_inicial)
    metricas.observar_frontera(1)
    nombre = "DLS({0})".format(limite)

    if problema.es_meta(raiz.estado):
        return metricas.resultado(raiz, nombre)
    if limite == 0:
        return metricas.resultado(None, nombre, corte=True)

    hubo_corte = False
    metricas.expandidos += 1
    pila = [(raiz, iter(problema.acciones(raiz.estado)))]
    en_camino = {raiz.estado}

    while pila:
        nodo, pendientes = pila[-1]
        accion = next(pendientes, None)
        if accion is None:
            # Todas las acciones de `nodo` probadas: se retrocede.
            pila.pop()
            if control == "camino":
                en_camino.discard(nodo.estado)
            continue

        hijo = nodo.hijo(problema, accion)
        metricas.generados += 1
        if hijo.estado in en_camino:
            metricas.repetidos_descartados += 1
            continue
        if traza is not None:
            traza.append(hijo.estado)
        if problema.es_meta(hijo.estado):
            return metricas.resultado(hijo, nombre)
        if hijo.profundidad == limite:
            # El limite impide mirar mas alla: podria haber solucion abajo.
            hubo_corte = True
            if control == "global":
                en_camino.add(hijo.estado)
            continue

        metricas.expandidos += 1
        pila.append((hijo, iter(problema.acciones(hijo.estado))))
        en_camino.add(hijo.estado)
        metricas.observar_frontera(len(pila))

    return metricas.resultado(None, nombre, corte=hubo_corte)


def profundizacion_iterativa(problema, iteraciones=None):
    """IDDFS: DLS con limite 0, 1, 2, ... hasta hallar solucion o fracaso.

    Se detiene en un fracaso DEFINITIVO (DLS sin corte), no en un tope
    arbitrario: si una DLS no corto nada, un limite mayor no veria nada
    nuevo. Termina siempre en un grafo finito, porque un camino simple no
    puede tener mas de |V| - 1 pasos.

    Las metricas ACUMULAN todas las iteraciones: re-expandir los niveles
    superiores es el costo real de IDDFS y debe verse en la tabla.
    """
    inicio = time.perf_counter()
    expandidos = generados = repetidos = frontera_maxima = 0
    for limite in itertools.count():
        r = busqueda_limitada(problema, limite)
        expandidos += r.expandidos
        generados += r.generados
        repetidos += r.repetidos_descartados
        frontera_maxima = max(frontera_maxima, r.frontera_maxima)
        if iteraciones is not None:
            iteraciones.append({"limite": limite, "expandidos": r.expandidos,
                                "encontrado": r.encontrado, "corte": r.corte})
        if r.encontrado or not r.corte:
            return ResultadoBusqueda(
                r.encontrado, r.camino, r.acciones, r.costo, r.profundidad,
                expandidos, generados, repetidos, frontera_maxima,
                (time.perf_counter() - inicio) * 1000.0, "IDDFS")
        assert limite <= len(problema.grafo), "IDDFS deberia haber terminado"


# ---------------------------------------------------------------------------
# 1) Limite insuficiente, exacto y excesivo sobre la instancia (d = 59)
# ---------------------------------------------------------------------------
_d = len(CAMINO_REFERENCIA) - 1
_dls_corto = busqueda_limitada(PROBLEMA, _d - 1)
_dls_exacto = busqueda_limitada(PROBLEMA, _d)
_dls_largo = busqueda_limitada(PROBLEMA, 2 * _d)
print("INSTANCIA INDIVIDUAL: DLS con limite insuficiente, exacto y excesivo "
      "(d = {0})".format(_d))
imprimir_resultados([_dls_corto, _dls_exacto, _dls_largo])
for _r in (_dls_corto, _dls_exacto, _dls_largo):
    verificar_resultado(PROBLEMA, _r)
assert not _dls_corto.encontrado and _dls_corto.corte      # CORTE, no fracaso
assert _dls_exacto.camino == CAMINO_REFERENCIA
# En un arbol el unico camino simple es el de referencia: un limite de
# sobra no puede producir uno mas largo (en un grafo con ciclos si, ver 4).
assert _dls_largo.camino == CAMINO_REFERENCIA
assert _dls_exacto.frontera_maxima <= _d and _dls_largo.frontera_maxima <= 2 * _d
# Prediccion: con un limite que no se alcanza nunca, DLS es DFS. En un arbol
# visita en el mismo preorden N-E-S-O, asi que debe expandir exactamente los
# mismos estados que la DFS de la parte 1, aunque son dos implementaciones
# distintas. Genera MENOS, porque produce los hijos de a uno y no llega a
# construir los hermanos que DFS apilo y nunca visito. La condicion "no se
# alcanza" se comprueba: si la meta esta cerca, 2d puede quedarse corto.
_limite_libre = _dls_largo.frontera_maxima < 2 * _d
if _limite_libre:
    assert _dls_largo.expandidos == RESULTADO_DFS.expandidos
    assert _dls_largo.generados <= RESULTADO_DFS.generados
print("limite {0}: corte={1} (no hay solucion DENTRO del limite, pero se "
      "podo algo)".format(_d - 1, _dls_corto.corte))
if _limite_libre:
    print("limite {0}: nunca se alcanza; expande {1}, igual que DFS; genera {2} "
          "frente a {3} de DFS.".format(2 * _d, _dls_largo.expandidos,
                                        _dls_largo.generados, RESULTADO_DFS.generados))
else:
    print("limite {0}: se alcanza en esta instancia, asi que DLS no equivale a "
          "DFS.".format(2 * _d))

# ---------------------------------------------------------------------------
# 2) IDDFS sobre la instancia
# ---------------------------------------------------------------------------
ITERACIONES_IDDFS = []
RESULTADO_IDDFS = profundizacion_iterativa(PROBLEMA, ITERACIONES_IDDFS)
verificar_resultado(PROBLEMA, RESULTADO_IDDFS)
assert RESULTADO_IDDFS.camino == CAMINO_REFERENCIA
# Con costo unitario IDDFS es optima: encuentra la profundidad de BFS.
assert RESULTADO_IDDFS.profundidad == RESULTADO_BFS.profundidad
assert [it["limite"] for it in ITERACIONES_IDDFS] == list(range(_d + 1))
assert all(it["corte"] for it in ITERACIONES_IDDFS[:-1])

print()
print("IDDFS: {0} iteraciones (limites 0..{1}); la ultima halla la meta.".format(
    len(ITERACIONES_IDDFS), _d))
print("Expandidos por iteracion (las anteriores se repiten en cada una):")
print("  " + " ".join("{0}:{1}".format(it["limite"], it["expandidos"])
                      for it in ITERACIONES_IDDFS[::6] + ITERACIONES_IDDFS[-1:]))
print()
imprimir_resultados([RESULTADO_BFS, RESULTADO_DFS, RESULTADO_UCS, _dls_exacto,
                     RESULTADO_IDDFS])
print("IDDFS paga {0:.1f} veces los expandidos de BFS por re-expandir los "
      "niveles superiores en cada iteracion.".format(
          RESULTADO_IDDFS.expandidos / RESULTADO_BFS.expandidos))

# ---------------------------------------------------------------------------
# 3) Meta inalcanzable: IDDFS debe terminar por FRACASO, no por un tope
# ---------------------------------------------------------------------------
def grafo_sin_corredor(grafo, u, v):
    """Copia congelada del grafo sin la arista {u, v}. No toca el original."""
    copia = {c: set(vs) for c, vs in grafo.items()}
    copia[u].discard(v)
    copia[v].discard(u)
    return MappingProxyType({c: frozenset(vs) for c, vs in copia.items()})


_cortado = grafo_sin_corredor(INSTANCIA.grafo, *CORREDOR_DE_CORTE)
_sin_salida = ProblemaLaberinto(_cortado, INSTANCIA.inicio, INSTANCIA.meta)
# Profundidad maxima de la componente del inicio, medida por niveles con una
# utilidad independiente de los buscadores (D-07).
_nivel, _vistos, _prof_max = {INSTANCIA.inicio}, {INSTANCIA.inicio}, 0
while True:
    _nivel = {v for u in _nivel for v in _cortado[u]} - _vistos
    if not _nivel:
        break
    _vistos |= _nivel
    _prof_max += 1
_iter_fracaso = []
_r = profundizacion_iterativa(_sin_salida, _iter_fracaso)
verificar_resultado(_sin_salida, _r)
assert not _r.encontrado and not _r.corte
# Prediccion: con limite L <= prof_max siempre hay un nodo en el limite
# (corte); con L = prof_max + 1 ya no hay ninguno (fracaso). Son exactamente
# prof_max + 2 iteraciones.
assert len(_iter_fracaso) == _prof_max + 2
assert not _iter_fracaso[-1]["corte"] and all(it["corte"] for it in _iter_fracaso[:-1])
print()
print("Meta inalcanzable: la componente del inicio llega a profundidad {0}. "
      "IDDFS corta en los limites 0..{0} y declara FRACASO en el limite {1} "
      "({2} iteraciones), sin ningun tope externo.".format(
          _prof_max, _prof_max + 1, len(_iter_fracaso)))

# ---------------------------------------------------------------------------
# 4) Grafo con ciclos: rejilla 2x4 SIN paredes
# ---------------------------------------------------------------------------
def rejilla_abierta(filas, columnas):
    """Todas las celdas conectadas con sus vecinas ortogonales."""
    return MappingProxyType({
        (f, c): frozenset(desplazar((f, c), a) for a in ACCIONES
                          if 0 <= desplazar((f, c), a)[0] < filas
                          and 0 <= desplazar((f, c), a)[1] < columnas)
        for f in range(filas) for c in range(columnas)})


GRAFO_ABIERTO_2x4 = rejilla_abierta(2, 4)
PROBLEMA_ABIERTO = ProblemaLaberinto(GRAFO_ABIERTO_2x4, (1, 0), (1, 3))
print()
print("REJILLA ABIERTA 2x4 (con ciclos), de (1, 0) a (1, 3); distancia 3")
print(dibujar_laberinto(GRAFO_ABIERTO_2x4, 2, 4))

_r_bfs_ab = bfs(PROBLEMA_ABIERTO)
_r_dls_ab = busqueda_limitada(PROBLEMA_ABIERTO, 10)
_r_iddfs_ab = profundizacion_iterativa(PROBLEMA_ABIERTO)
for _r in (_r_bfs_ab, _r_dls_ab, _r_iddfs_ab):
    verificar_resultado(PROBLEMA_ABIERTO, _r)
imprimir_resultados([_r_bfs_ab, _r_dls_ab, _r_iddfs_ab])
# DLS con limite de sobra no es optima: sigue la primera rama (N primero).
assert _r_dls_ab.profundidad == 5 and _r_iddfs_ab.profundidad == 3
print("DLS(10) devuelve {0} pasos: {1}. IDDFS y BFS devuelven 3.".format(
    _r_dls_ab.profundidad, " ".join(map(str, _r_dls_ab.camino))))

# --- Contra-experimento (D-24): alcanzados GLOBAL en lugar de por camino ----
# Con limite exacto 3, la version global pierde la solucion: (1,1) se
# alcanza primero a profundidad 3 por arriba, queda marcado, y cuando se
# llega a el a profundidad 1 ya no se puede usar.
_traza_camino, _traza_global = [], []
_r_camino = busqueda_limitada(PROBLEMA_ABIERTO, 3, "camino", _traza_camino)
_r_global = busqueda_limitada(PROBLEMA_ABIERTO, 3, "global", _traza_global)
assert _r_camino.encontrado and not _r_global.encontrado
print()
print("Contra-experimento, limite exacto 3:")
print("  control por camino : halla {0}".format(" ".join(map(str, _r_camino.camino))))
print("  control global     : {0}; visita {1} y ya no puede volver a (1, 1)".format(
    "corte" if _r_global.corte else "fracaso",
    " ".join(map(str, _traza_global))))

### 6.4 Búsqueda bidireccional: capas, encuentro y unión de caminos

Se lanzan dos BFS, una desde el inicio (**adelante**) y otra desde la meta (**atrás**). En cada paso se expande **la capa completa** del lado cuya frontera es más pequeña (D-27). Cada hijo nuevo se busca de inmediato en lo alcanzado por el otro lado; si está, las dos regiones se tocaron. No se ejecutan dos búsquedas completas para comparar al final.

**Unir las dos mitades.** La mitad de atrás se reconstruye desde la meta (`meta → … → x`). Para recorrerla hacia adelante se invierte la lista de estados y **se invierte cada acción**: si desde la meta se fue al `N` para llegar a `u`, desde `u` hay que ir al `S`. Como los costos pertenecen a aristas simétricas (D-23), el costo total es `g_adelante(x) + g_atrás(x)`. `verificar_resultado` lo recalcula paso a paso hacia adelante, así que cualquier error de orientación o de inversión haría fallar la celda.

**Por qué el camino es el más corto (D-28).** Supongamos que el lado de adelante expande su capa *k* (todos los estados a distancia *k* del inicio), que el de atrás tiene completas sus capas hasta *j*, y que no ha habido ningún encuentro.

1. *Ningún camino es más corto que k + j + 1.* En un camino más corto de longitud *D*, el estado que está a distancia *k* del inicio está a distancia *D − k* de la meta. Si *D − k ≤ j*, ese estado ya habría sido alcanzado por los dos lados y el encuentro se habría detectado antes.
2. *Todo encuentro en esta capa mide exactamente k + j + 1.* Un encuentro `x` tiene distancia *k + 1* desde el inicio y ≤ *j* desde la meta. Si fuera *< j*, su padre `p`, a distancia *k*, estaría a ≤ *j* de la meta y también habría sido un encuentro anterior.

Por (1) y (2), el primer encuentro ya es óptimo. Aun así se termina la capa y se reúnen todos los encuentros. El costo es una sola capa, y así la celda siguiente puede **comprobar** el lema: en 300 grafos aleatorios con ciclos, todos los encuentros de la capa final miden lo mismo que BFS.

**Qué muestra la instancia.** La ventaja teórica es O(b^(d/2)) frente a O(b^d), pero un laberinto casi sin bifurcaciones apenas la aprovecha: la búsqueda bidireccional expande el 88 % de lo que expande BFS. Con la meta inalcanzable basta con que **una** frontera se vacíe, pero mientras tanto el otro lado también trabajó en vano.

**Con costos no unitarios** este algoritmo minimiza pasos y no costo, igual que BFS. Una versión óptima en costo necesitaría dos UCS y un criterio de parada distinto; queda fuera del alcance.

In [ ]:
# ===========================================================================
# VERSION 1 - Parte 4: busqueda bidireccional
#
# Dos BFS simultaneas, una desde el inicio y otra desde la meta, que avanzan
# por CAPAS completas y se detienen en cuanto sus regiones alcanzadas se
# tocan. No se ejecutan dos busquedas completas para comparar al final: cada
# hijo generado se contrasta de inmediato con lo alcanzado por el otro lado.
# ===========================================================================

ACCION_INVERSA = {"N": "S", "S": "N", "E": "O", "O": "E"}


def unir_caminos(nodo_adelante, nodo_atras):
    """Une las dos mitades en el estado comun x.

    La mitad de atras se reconstruyo desde la meta: meta -> ... -> x, con
    acciones que van en ese sentido. Para recorrerla hacia adelante se
    invierte la lista de estados y se invierte CADA accion (si desde la meta
    se fue al N para llegar a u, desde u hay que ir al S). Con costos por
    arista simetricos (D-23), el costo de la mitad de atras es su g.
    """
    estados_ad, acciones_ad = reconstruir(nodo_adelante)
    estados_at, acciones_at = reconstruir(nodo_atras)
    assert estados_ad[-1] == estados_at[-1], "las mitades no se tocan"
    camino = estados_ad + tuple(reversed(estados_at))[1:]
    acciones = acciones_ad + tuple(ACCION_INVERSA[a]
                                   for a in reversed(acciones_at))
    return camino, acciones, nodo_adelante.g + nodo_atras.g


def bidireccional(problema, traza=None):
    """BFS bidireccional por capas, equilibrada por tamano de frontera (D-27).

    - En cada paso se expande la CAPA COMPLETA del lado con la frontera mas
      pequena (empate: adelante). Expandir capas enteras, y no nodos sueltos,
      es lo que permite razonar sobre la optimalidad (D-28).
    - Cada hijo nuevo se busca en `alcanzados` del OTRO lado. Si esta, hay
      un punto de encuentro. Se termina la capa en curso, se reunen todos
      los puntos de encuentro y se elige el de menor longitud total (en
      empate, el primero descubierto).
    - Si cualquiera de las dos fronteras se vacia, no hay camino: ese lado
      agoto su componente sin tocar al otro. No hace falta agotar ambos.

    La meta se reconoce por el encuentro, al GENERAR, y no al extraer como
    en los demas algoritmos: en una busqueda bidireccional el objetivo no es
    un estado sino que las dos regiones se toquen (D-28).
    Requiere un grafo simetrico: los predecesores de s son sus vecinos, que
    es lo que garantiza la comprobacion 1 de la auditoria.
    """
    metricas = Metricas()
    raiz_ad = Nodo(problema.estado_inicial)
    raiz_at = Nodo(problema.meta)
    metricas.observar_frontera(2)
    if problema.es_meta(raiz_ad.estado):
        return metricas.resultado(raiz_ad, "BIDIR")

    alcanzados = {"adelante": {raiz_ad.estado: raiz_ad},
                  "atras": {raiz_at.estado: raiz_at}}
    frontera = {"adelante": [raiz_ad], "atras": [raiz_at]}

    while frontera["adelante"] and frontera["atras"]:
        lado = ("adelante" if len(frontera["adelante"]) <= len(frontera["atras"])
                else "atras")
        otro = "atras" if lado == "adelante" else "adelante"

        siguiente, encuentros = [], []
        antes = metricas.expandidos
        for nodo in frontera[lado]:
            metricas.expandidos += 1
            for accion in problema.acciones(nodo.estado):
                hijo = nodo.hijo(problema, accion)
                metricas.generados += 1
                if hijo.estado in alcanzados[lado]:
                    metricas.repetidos_descartados += 1
                    continue
                alcanzados[lado][hijo.estado] = hijo
                siguiente.append(hijo)
                if hijo.estado in alcanzados[otro]:
                    encuentros.append((hijo, alcanzados[otro][hijo.estado]))
        frontera[lado] = siguiente
        metricas.observar_frontera(len(frontera["adelante"]) + len(frontera["atras"]))

        if traza is not None:
            traza.append({"lado": lado, "expandidos": metricas.expandidos - antes,
                          "capa_nueva": [h.estado for h in siguiente],
                          "encuentros": [(h.estado, h.profundidad + o.profundidad)
                                         for h, o in encuentros]})
        if encuentros:
            hijo, del_otro = min(encuentros,
                                 key=lambda par: par[0].profundidad + par[1].profundidad)
            ad, at = (hijo, del_otro) if lado == "adelante" else (del_otro, hijo)
            camino, acciones, costo = unir_caminos(ad, at)
            return metricas.resultado_de_camino(camino, acciones, costo, "BIDIR")

    return metricas.resultado(None, "BIDIR")


# ---------------------------------------------------------------------------
# 1) Instancia individual
# ---------------------------------------------------------------------------
TRAZA_BIDIR = []
RESULTADO_BIDIR = bidireccional(PROBLEMA, TRAZA_BIDIR)
verificar_resultado(PROBLEMA, RESULTADO_BIDIR)
assert RESULTADO_BIDIR.camino == CAMINO_REFERENCIA
assert RESULTADO_BIDIR.profundidad == RESULTADO_BFS.profundidad

_capas = {"adelante": 0, "atras": 0}
for _t in TRAZA_BIDIR:
    _capas[_t["lado"]] += 1
_punto = TRAZA_BIDIR[-1]["encuentros"][0][0]
print("INSTANCIA INDIVIDUAL")
imprimir_resultados([RESULTADO_BFS, RESULTADO_BIDIR])
print("Capas expandidas: {0} desde el inicio y {1} desde la meta ({2} en total, "
      "d = {3}).".format(_capas["adelante"], _capas["atras"],
                         len(TRAZA_BIDIR), RESULTADO_BIDIR.profundidad))
print("Punto de encuentro: {0}, posicion {1} de las {2} celdas del camino.".format(
    _punto, CAMINO_REFERENCIA.index(_punto), len(CAMINO_REFERENCIA)))
# Cada capa avanza un paso en un lado: se necesitan d capas para cubrir los
# d pasos del camino.
assert len(TRAZA_BIDIR) == RESULTADO_BIDIR.profundidad

print("BIDIR expande {0} estados frente a {1} de BFS ({2:.0%}).".format(
    RESULTADO_BIDIR.expandidos, RESULTADO_BFS.expandidos,
    RESULTADO_BIDIR.expandidos / RESULTADO_BFS.expandidos))

# ---------------------------------------------------------------------------
# 2) Dos puntos de encuentro posibles (caso limite de la seccion 8)
# ---------------------------------------------------------------------------
# Rejilla abierta 2x2, de (0,0) a (1,1). Hay dos caminos de 2 pasos, uno por
# (0,1) y otro por (1,0), y los dos se descubren en la MISMA capa.
_p_2x2 = ProblemaLaberinto(rejilla_abierta(2, 2), (0, 0), (1, 1))
_traza_2x2 = []
_r_2x2 = bidireccional(_p_2x2, _traza_2x2)
verificar_resultado(_p_2x2, _r_2x2)
_encuentros = _traza_2x2[-1]["encuentros"]
assert sorted(e for e, _ in _encuentros) == [(0, 1), (1, 0)]
assert _r_2x2.profundidad == 2 and _r_2x2.camino[1] == _encuentros[0][0]
print()
print("REJILLA ABIERTA 2x2, (0, 0) -> (1, 1)")
print("  puntos de encuentro en la ultima capa: {0}".format(
    ", ".join("{0} (longitud {1})".format(e, l) for e, l in _encuentros)))
print("  elegido: {0} -> camino {1}, acciones {2}".format(
    _r_2x2.camino[1], " ".join(map(str, _r_2x2.camino)), "".join(_r_2x2.acciones)))

# ---------------------------------------------------------------------------
# 3) Lema de la capa (D-28), comprobado en muchos grafos con ciclos
# ---------------------------------------------------------------------------
# Afirmacion: con capas completas y encuentro detectado al generar, TODOS
# los puntos de encuentro de una misma capa tienen la misma longitud, y esa
# longitud es la de BFS. Se prueba en 300 subgrafos aleatorios de rejillas
# pequenas (con ciclos), con una semilla fija.
_rng = random.Random(INSTANCIA.semilla)
_probados = _con_varios = 0
for _ in range(300):
    _f, _c = _rng.choice([(2, 3), (3, 3), (3, 4), (4, 4)])
    _abierta = rejilla_abierta(_f, _c)
    _aristas = sorted({tuple(sorted((u, v))) for u in _abierta for v in _abierta[u]})
    _quitadas = {a for a in _aristas if _rng.random() < 0.25}
    _g = {c: set() for c in _abierta}
    for _u, _v in _aristas:
        if (_u, _v) not in _quitadas:
            _g[_u].add(_v)
            _g[_v].add(_u)
    _g = MappingProxyType({c: frozenset(v) for c, v in _g.items()})
    _ini, _fin = _rng.sample(sorted(_g), 2)
    _p = ProblemaLaberinto(_g, _ini, _fin)
    _tr = []
    _rb, _rbfs = bidireccional(_p, _tr), bfs(_p)
    verificar_resultado(_p, _rb)
    assert _rb.encontrado == _rbfs.encontrado
    if _rb.encontrado:
        _probados += 1
        _longitudes = {l for _, l in _tr[-1]["encuentros"]}
        assert _longitudes == {_rbfs.profundidad}, (_ini, _fin, _longitudes)
        _con_varios += len(_tr[-1]["encuentros"]) > 1
print()
print("Lema de la capa: {0} grafos con solucion; en {1} hubo varios puntos de "
      "encuentro y en todos tenian la longitud de BFS.".format(_probados, _con_varios))

# ---------------------------------------------------------------------------
# 4) Meta inalcanzable: basta agotar el lado mas pequeno
# ---------------------------------------------------------------------------
_sin_salida = ProblemaLaberinto(
    grafo_sin_corredor(INSTANCIA.grafo, *CORREDOR_DE_CORTE),
    INSTANCIA.inicio, INSTANCIA.meta)
_traza_sin = []
_r = bidireccional(_sin_salida, _traza_sin)
verificar_resultado(_sin_salida, _r)
_r_bfs_sin = bfs(_sin_salida)
assert not _r.encontrado and not _r.corte
_por_lado = {"adelante": 0, "atras": 0}
for _t in _traza_sin:
    _por_lado[_t["lado"]] += _t["expandidos"]
# El lado cuya frontera se vacia agota su componente ENTERA; el otro solo
# avanzo mientras su frontera era la mas pequena, y ese trabajo es inutil.
_comp = {"adelante": len(inundar(_sin_salida.grafo, _sin_salida.estado_inicial)),
         "atras": len(inundar(_sin_salida.grafo, _sin_salida.meta))}
_agotado = next(l for l in ("adelante", "atras") if _por_lado[l] == _comp[l])
_otro = "atras" if _agotado == "adelante" else "adelante"
assert _por_lado[_otro] < _comp[_otro]
print()
print("Meta inalcanzable: BFS expande {0} estados (la componente del inicio). "
      "BIDIR expande {1}: el lado de {2} agota su componente ({3}) y el de {4} "
      "expande {5} en vano.".format(
          _r_bfs_sin.expandidos, _r.expandidos, _agotado, _por_lado[_agotado],
          _otro, _por_lado[_otro]))

# ---------------------------------------------------------------------------
# 5) Con costos: BIDIR minimiza pasos, no costo
# ---------------------------------------------------------------------------
_r_bidir_p = bidireccional(PROBLEMA_PONDERADO)
verificar_resultado(PROBLEMA_PONDERADO, _r_bidir_p)   # costo = g_ad + g_at
print()
print("Grafo ponderado 2x4: BIDIR costo {0}, UCS costo {1}. Como BFS, la "
      "busqueda bidireccional por capas no es optima en costo.".format(
          _r_bidir_p.costo, ucs(PROBLEMA_PONDERADO).costo))

## 7. Lee: propagación de un frente luminoso

No se modela un único rayo que rebota. Se modela una onda que avanza simultáneamente por todos los corredores accesibles.

```text
LEE(grafo, inicio, meta):
    etiqueta[inicio] ← 0
    frente_actual ← {inicio}

    mientras frente_actual no esté vacío y meta no tenga etiqueta:
        frente_siguiente ← vacío
        para cada celda del frente_actual:
            para cada vecino transitable sin etiqueta:
                etiqueta[vecino] ← etiqueta[celda] + 1
                registrar procedencia o dirección
                agregar vecino al frente_siguiente
        frente_actual ← frente_siguiente

    si meta no tiene etiqueta: devolver fracaso
    reconstruir desde meta siguiendo etiquetas decrecientes
```

### Requisitos adicionales

- Dibuje al menos ocho instantes del frente de onda.
- Use una escala de color que represente el tiempo de llegada.
- Compruebe que la etiqueta de la meta coincide con la profundidad de BFS.
- Explique formalmente por qué Lee es BFS en un grafo no ponderado.
- Analice qué deja de funcionar cuando los costos no son unitarios.

### 7.1 Lee es BFS por frentes: argumento formal

Sea *L_k* el conjunto de celdas a distancia exactamente *k* del inicio, medida en número de corredores. Se afirma que **el frente *k* de Lee es *L_k*** y que **`etiqueta[v] = dist(inicio, v)`**. Se demuestra por inducción sobre *k*:

- **Base.** El frente 0 es `{inicio}` = *L_0*, con etiqueta 0.
- **Paso.** Por hipótesis, los frentes 0…*k* son *L_0*…*L_k*, así que al procesar el frente *k* ya tienen etiqueta **todas** las celdas a distancia ≤ *k* y ninguna otra. Una celda `v` recibe la etiqueta *k + 1* si y solo si (a) no tenía etiqueta, es decir, está a distancia > *k*, y (b) es vecina de una celda de *L_k*, es decir, está a distancia ≤ *k + 1*. Luego `v` ∈ *L_{k+1}*. Recíprocamente, toda celda de *L_{k+1}* tiene un vecino en *L_k*, que es su anterior en un camino más corto, y por eso recibe la etiqueta en este frente.

BFS calcula exactamente las mismas distancias. Su cola FIFO contiene en cada momento nodos de a lo sumo dos niveles consecutivos, y los extrae en orden de nivel. Lee hace **explícita** la frontera entre niveles: en lugar de una cola, procesa *L_k* entero y construye *L_{k+1}* aparte. **Es BFS sincronizada por niveles.**

La **reconstrucción** usa solo las etiquetas. Una celda con etiqueta *k* > 0 tiene siempre un vecino con etiqueta *k − 1*, así que bajar por etiquetas decrecientes desde la meta llega al inicio en exactamente *k* pasos: es un camino más corto. No hace falta guardar padres. Esa economía de memoria era el objetivo original de Lee (1961) en el trazado de pistas de circuitos.

La celda siguiente lo comprueba en la instancia individual. La etiqueta de la meta es 59, la profundidad de BFS. Y además **cada una de las 253 etiquetas** coincide con la profundidad que BFS obtiene hasta esa celda. Cada frente es exactamente el conjunto de celdas con esa etiqueta.

**Diferencia de métricas con BFS.** El pseudocódigo evalúa la parada *entre* frentes: termina el frente en el que la meta recibe su etiqueta, pero nunca procesa el frente de la meta. BFS, en cambio, prueba el objetivo al extraer, y antes de sacar la meta expande las celdas de su mismo nivel que entraron antes. Por eso Lee expande a lo sumo lo que BFS: 244 frente a 249 en la instancia.

### 7.2 Qué deja de funcionar con costos no unitarios

La etiqueta de Lee cuenta **frentes**, es decir **pasos**. Toda la demostración anterior usa que un corredor mide 1: «vecino de *L_k*» implica «distancia *k + 1*». Con costos distintos eso deja de ser cierto:

1. **Lee sin cambios minimiza pasos, no costo.** En el grafo ponderado de 2×4 devuelve un camino de 4 pasos y costo 17; UCS encuentra costo 12.
2. **Poner el costo en la etiqueta tampoco basta.** Si la etiqueta es `etiqueta[u] + c(u, v)` y se asigna al primer contacto, como hace Lee, queda congelada aunque después, en un frente posterior, llegue un camino más largo en pasos pero más barato. En el grafo de prueba, 3 de las 8 etiquetas salen erróneas: `(1,1)` recibe 8 y su costo mínimo es 6.

Para arreglarlo hacen falta dos cambios. Hay que permitir **rebajar** una etiqueta y procesar las celdas **en orden de costo**, no de frente. Eso ya no es Lee: es Dijkstra, que es la UCS de la sección 6.2. La alternativa clásica que conserva los frentes sirve para costos enteros pequeños: se parte cada corredor de costo *c* en *c* tramos unitarios y se aplica Lee al grafo subdividido. El número de frentes crece entonces con el costo óptimo *C\**.

In [13]:
# ===========================================================================
# Seccion 7 - Algoritmo de Lee: propagacion de un frente de onda
#
# Traduccion directa del pseudocodigo del enunciado. No usa Nodo ni padres:
# la unica memoria es `etiqueta[celda]`, el instante en que la onda llego a
# ella. El camino se reconstruye DESDE LA META bajando por etiquetas
# decrecientes, que es la idea original de Lee (1961) (D-29).
# ===========================================================================

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
import numpy as np


def lee(problema, registro=None):
    """Lee sobre `problema`. Devuelve un ResultadoBusqueda.

    `registro`, si es un dict, recibe:
      "etiqueta": {celda: instante de llegada}
      "frentes":  [frente_0, frente_1, ...], cada uno una lista de celdas

    Metricas (D-29): cada celda del frente que se procesa cuenta como
    expandida; cada vecino examinado, como generado; un vecino que ya tenia
    etiqueta, como repetido. `frontera_maxima` es el frente mas grande.
    Como dice el pseudocodigo, la parada se evalua ENTRE frentes: se termina
    el frente en el que la meta recibe su etiqueta.
    """
    metricas = Metricas()
    inicio, meta = problema.estado_inicial, problema.meta
    etiqueta = {inicio: 0}
    frente_actual = [inicio]
    frentes = [list(frente_actual)]
    metricas.observar_frontera(1)

    while frente_actual and meta not in etiqueta:
        frente_siguiente = []
        for celda in frente_actual:
            metricas.expandidos += 1
            for accion in problema.acciones(celda):
                vecino = problema.resultado(celda, accion)
                metricas.generados += 1
                if vecino in etiqueta:
                    metricas.repetidos_descartados += 1
                    continue
                etiqueta[vecino] = etiqueta[celda] + 1
                frente_siguiente.append(vecino)
        frente_actual = frente_siguiente
        if frente_actual:
            frentes.append(list(frente_actual))
        metricas.observar_frontera(len(frente_actual))

    if registro is not None:
        registro["etiqueta"] = etiqueta
        registro["frentes"] = frentes
    if meta not in etiqueta:
        return metricas.resultado(None, "LEE")

    # Reconstruccion hacia atras: desde una celda con etiqueta k siempre hay
    # un vecino con etiqueta k - 1 (el anterior en un camino mas corto). Se
    # toma el primero en orden N-E-S-O, y la accion hacia adelante es la
    # inversa de la que se uso para retroceder.
    camino, acciones, costo = [meta], [], 0
    celda = meta
    while celda != inicio:
        for accion in problema.acciones(celda):
            anterior = problema.resultado(celda, accion)
            if etiqueta.get(anterior) == etiqueta[celda] - 1:
                break
        else:
            raise AssertionError("sin vecino con etiqueta {0}".format(etiqueta[celda] - 1))
        camino.append(anterior)
        acciones.append(ACCION_INVERSA[accion])
        costo += problema.costo_paso(anterior, ACCION_INVERSA[accion], celda)
        celda = anterior
    camino.reverse()
    acciones.reverse()
    return metricas.resultado_de_camino(camino, acciones, costo, "LEE")


# ---------------------------------------------------------------------------
# 1) La instancia individual
# ---------------------------------------------------------------------------
REGISTRO_LEE = {}
RESULTADO_LEE = lee(PROBLEMA, REGISTRO_LEE)
ETIQUETA_LEE = REGISTRO_LEE["etiqueta"]
FRENTES_LEE = REGISTRO_LEE["frentes"]
verificar_resultado(PROBLEMA, RESULTADO_LEE)
assert RESULTADO_LEE.camino == CAMINO_REFERENCIA

# Exigencia del enunciado: la etiqueta de la meta es la profundidad de BFS.
assert ETIQUETA_LEE[PROBLEMA.meta] == RESULTADO_BFS.profundidad
# Mas fuerte: TODA etiqueta coincide con la profundidad de BFS hasta esa
# celda. No se contrastan las 500 celdas, solo las que Lee alcanzo.
for _celda, _k in ETIQUETA_LEE.items():
    assert bfs(ProblemaLaberinto(INSTANCIA.grafo, INSTANCIA.inicio, _celda)).profundidad == _k
# Cada frente es exactamente el conjunto de celdas con esa etiqueta.
for _k, _frente in enumerate(FRENTES_LEE):
    assert set(_frente) == {c for c, e in ETIQUETA_LEE.items() if e == _k}
# Lee termina el frente de la meta y no prueba el objetivo al extraer, pero
# no llega a procesar el frente d: expande a lo sumo lo que BFS.
assert RESULTADO_LEE.expandidos == sum(len(f) for f in FRENTES_LEE[:-1])
assert RESULTADO_LEE.expandidos <= RESULTADO_BFS.expandidos

print("LEE sobre la instancia individual")
imprimir_resultados([RESULTADO_BFS, RESULTADO_LEE])
print("Etiqueta de la meta = {0} = profundidad de BFS. Las {1} etiquetas "
      "coinciden con la profundidad de BFS hasta cada celda.".format(
          ETIQUETA_LEE[PROBLEMA.meta], len(ETIQUETA_LEE)))
print("Tamano de cada frente (t: celdas):")
print("  " + " ".join("{0}:{1}".format(k, len(f)) for k, f in enumerate(FRENTES_LEE)))

# El mapa de etiquetas como texto: la firma con `marcas` de la seccion 1.7 se
# diseno para esto. Es la evidencia en datos; la figura es solo lectura.
print()
print("Mapa de etiquetas (instante de llegada de la onda); I = inicio, M = meta:")
_marcas = {c: e for c, e in ETIQUETA_LEE.items()}
_marcas[PROBLEMA.estado_inicial] = "I"
_marcas[PROBLEMA.meta] = "M"
print(dibujar_laberinto(INSTANCIA.grafo, INSTANCIA.filas, INSTANCIA.columnas, _marcas))

# ---------------------------------------------------------------------------
# 2) Ocho instantes del frente de onda
# ---------------------------------------------------------------------------
# Escala secuencial de un solo tono (azul, claro -> oscuro): mas oscuro =
# llego mas tarde. Las celdas aun no alcanzadas van en gris neutro, fuera de
# la escala. El camino final, en otro tono (naranja), solo en el ultimo
# instante. La escala es la MISMA en los ocho paneles (0..d), asi que un
# mismo color significa el mismo instante en todos.
_RAMPA_AZUL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
               "#256abf", "#184f95", "#0d366b"]
_MAPA_TIEMPO = LinearSegmentedColormap.from_list("tiempo", _RAMPA_AZUL)
_MAPA_TIEMPO.set_bad("#ebeae6")
_TINTA, _TINTA_2, _SUPERFICIE, _CAMINO = "#0b0b0b", "#52514e", "#fcfcfb", "#eb6834"


def _paredes(grafo, filas, columnas):
    """Segmentos de pared: una pared es la AUSENCIA de arista (seccion 1.7)."""
    segmentos = [[(0, 0), (columnas, 0)], [(0, 0), (0, filas)]]
    for (f, c), vecinos in grafo.items():
        if (f, c + 1) not in vecinos:
            segmentos.append([(c + 1, f), (c + 1, f + 1)])
        if (f + 1, c) not in vecinos:
            segmentos.append([(c, f + 1), (c + 1, f + 1)])
    return segmentos


def dibujar_frentes(instancia, etiqueta, instantes, camino=None, archivo=None):
    filas, columnas = instancia.filas, instancia.columnas
    t_max = max(etiqueta.values())
    paredes = _paredes(instancia.grafo, filas, columnas)
    fig, ejes = plt.subplots(2, 4, figsize=(16, 7.4), facecolor=_SUPERFICIE,
                             layout="constrained")
    for eje, t in zip(ejes.flat, instantes):
        matriz = np.full((filas, columnas), np.nan)
        for (f, c), e in etiqueta.items():
            if e <= t:
                matriz[f, c] = e
        imagen = eje.imshow(matriz, cmap=_MAPA_TIEMPO, vmin=0, vmax=t_max,
                            extent=(0, columnas, filas, 0), interpolation="nearest")
        eje.add_collection(LineCollection(paredes, colors=_TINTA_2, linewidths=0.8))
        if camino is not None and t == instantes[-1]:
            eje.plot([c + 0.5 for _, c in camino], [f + 0.5 for f, _ in camino],
                     color=_CAMINO, linewidth=2, solid_capstyle="round")
        for (f, c), letra in ((instancia.inicio, "I"), (instancia.meta, "M")):
            eje.text(c + 0.5, f + 0.5, letra, ha="center", va="center",
                     fontsize=8, fontweight="bold", color=_TINTA,
                     bbox=dict(boxstyle="circle,pad=0.15", fc=_SUPERFICIE,
                               ec="none"))
        alcanzadas = int(np.sum(~np.isnan(matriz)))
        eje.set_title("t = {0}  ·  {1} celda{2}".format(
                          t, alcanzadas, "" if alcanzadas == 1 else "s"),
                      fontsize=10, color=_TINTA, loc="left")
        eje.set_xlim(0, columnas)
        eje.set_ylim(filas, 0)
        eje.set_aspect("equal")
        eje.set_xticks([])
        eje.set_yticks([])
        for borde in eje.spines.values():
            borde.set_visible(False)
    barra = fig.colorbar(imagen, ax=ejes, shrink=0.8, pad=0.01)
    barra.set_label("instante de llegada de la onda (numero de frente)",
                    color=_TINTA_2, fontsize=9)
    barra.outline.set_visible(False)
    fig.suptitle("Frente de onda de Lee desde {0} hasta {1}: ocho instantes; "
                 "en el ultimo, el camino reconstruido".format(
                     instancia.inicio, instancia.meta),
                 color=_TINTA, fontsize=12, x=0.01, ha="left")
    if archivo:
        fig.savefig(archivo, dpi=120, facecolor=_SUPERFICIE)
    plt.show()


_d = ETIQUETA_LEE[PROBLEMA.meta]
INSTANTES_LEE = sorted({round(i * _d / 7) for i in range(8)})
assert len(INSTANTES_LEE) >= min(8, _d + 1), "se exigen al menos ocho instantes"
os.makedirs("resultados", exist_ok=True)
dibujar_frentes(INSTANCIA, ETIQUETA_LEE, INSTANTES_LEE, RESULTADO_LEE.camino,
                "resultados/lee_frente_de_onda.png")

# ---------------------------------------------------------------------------
# 3) Traza en el laberinto 3x3 (para la traza manual)
# ---------------------------------------------------------------------------
_reg_3x3 = {}
_r_lee_3x3 = lee(PROBLEMA_3x3, _reg_3x3)
verificar_resultado(PROBLEMA_3x3, _r_lee_3x3)
print("LEE en la 3x3: frentes", " | ".join(" ".join(map(str, f)) for f in _reg_3x3["frentes"]))
print(dibujar_laberinto(PROBLEMA_3x3.grafo, 3, 3, _reg_3x3["etiqueta"]))

# ---------------------------------------------------------------------------
# 4) Costos no unitarios: que deja de funcionar
# ---------------------------------------------------------------------------
# a) Lee tal cual ignora los costos: cuenta FRENTES (pasos), no costo.
_r_lee_p = lee(PROBLEMA_PONDERADO)
verificar_resultado(PROBLEMA_PONDERADO, _r_lee_p)
_r_ucs_p = ucs(PROBLEMA_PONDERADO)
assert _r_lee_p.costo > _r_ucs_p.costo
print()
print("Grafo ponderado 2x4: LEE devuelve costo {0} ({1} pasos); UCS, costo {2}.".format(
    _r_lee_p.costo, _r_lee_p.profundidad, _r_ucs_p.costo))

# b) Contra-experimento: usar el COSTO como etiqueta, asignada la primera vez
#    que la onda toca la celda, como hace Lee. La etiqueta queda congelada
#    aunque despues llegue un camino mas barato por un frente posterior.
def _lee_con_costos_ingenuo(problema):
    etiqueta = {problema.estado_inicial: 0}
    frente = [problema.estado_inicial]
    while frente:
        siguiente = []
        for celda in frente:
            for accion in problema.acciones(celda):
                vecino = problema.resultado(celda, accion)
                if vecino not in etiqueta:
                    etiqueta[vecino] = etiqueta[celda] + problema.costo_paso(
                        celda, accion, vecino)
                    siguiente.append(vecino)
        frente = siguiente
    return etiqueta


_ingenua = _lee_con_costos_ingenuo(PROBLEMA_PONDERADO)
_verdadera = {c: ucs(ProblemaLaberinto(GRAFO_PONDERADO, (0, 0), c,
                                       costo_por_arista(_COSTOS_PONDERADO))).costo
              for c in _ingenua}
_malas = {c: (_ingenua[c], _verdadera[c]) for c in sorted(_ingenua)
          if _ingenua[c] != _verdadera[c]}
assert _malas, "el contra-experimento deberia mostrar etiquetas erroneas"
print("Lee con etiqueta = costo, asignada al primer contacto: {0} de {1} "
      "etiquetas erroneas:".format(len(_malas), len(_ingenua)))
for _c, (_e, _v) in _malas.items():
    print("  {0}: etiqueta {1}, costo minimo real {2}".format(_c, _e, _v))

# Versión 2 — SimpleAI

Instalación orientativa:

```text
pip install simpleai
```

Consulte la documentación oficial. Debe crear una subclase de `SearchProblem` y definir, como mínimo:

| Método de SimpleAI | Responsabilidad en el laberinto |
|---|---|
| `actions(state)` | Producir únicamente movimientos legales |
| `result(state, action)` | Calcular el estado sucesor sin mutar el original |
| `is_goal(state)` | Reconocer la celda meta |
| `cost(state, action, state2)` | Devolver el costo del corredor |

Instancie el problema con un estado inicial inmutable y use `graph_search=True`.

Debe ejecutar y estudiar:

- `depth_first`;
- `breadth_first`;
- `uniform_cost`;
- `limited_depth_first`;
- `iterative_limited_depth_first`.

### Exigencia avanzada

SimpleAI devuelve un nodo solución, pero las métricas del contrato no aparecen automáticamente con la misma definición. Investigue su mecanismo de *viewer* o diseñe instrumentación externa **sin modificar el código fuente instalado de la biblioteca**. Documente cualquier diferencia entre las métricas de SimpleAI y las de su versión.

No copie la estructura de su buscador desde cero dentro de esta versión: el propósito es construir correctamente el adaptador del problema y comprender la biblioteca.

### V2.1 Adaptador, instrumentación y diferencias con la Versión 1

**Adaptador.** `LaberintoSimpleAI` hereda de `SearchProblem` y **delega** en `ProblemaLaberinto`: `actions` devuelve las mismas acciones en el mismo orden N-E-S-O, `result` devuelve una tupla nueva sin mutar nada, `is_goal` compara con la meta y `cost` devuelve el costo del corredor. No hay una segunda formulación escrita a mano que pueda discrepar (D-31). Se usa siempre `graph_search=True`.

**Instrumentación sin tocar la biblioteca (D-32).** SimpleAI invoca `viewer.event(nombre, …)` en puntos fijos de su bucle `_search`. `ContadorSimpleAI` es un visor propio que traduce esos eventos al contrato:

| Métrica del contrato | Cómo se obtiene en SimpleAI |
|---|---|
| `expandidos` | eventos `expanded` |
| `generados` | tamaño de la lista de hijos de cada `expanded` |
| `repetidos_descartados` | no hay evento: *generados − insertados*, y los insertados se deducen del tamaño de la frontera entre dos iteraciones |
| `frontera_maxima` | máximo de `len(frontera)` en `new_iteration`, antes de extraer |
| `corte` | no existe en SimpleAI: se deduce de si se extrajo un nodo no meta a profundidad = límite |
| `tiempo_ms` | una **segunda** ejecución **sin** visor. Con visor, SimpleAI ordena la frontera en cada iteración para entregársela, y ese costo contaminaría la medición |

**Diferencias medidas y su causa:**

- **BFS:** coincide con la Versión 1 en las **cuatro** métricas. La política de SimpleAI («descartar si el estado ya se extrajo o está en la frontera») equivale a nuestro «alcanzados», marcado al generar.
- **DFS:** SimpleAI apila los hijos en el orden de `actions` y su pila devuelve primero el último: explora **O-S-E-N**. Coincide exactamente con nuestra DFS con la pila **sin invertir**, y difiere de nuestra DFS (D-18). La diferencia se debe al orden de sucesores, no al algoritmo.
- **UCS:** el mismo costo. SimpleAI no deja entradas obsoletas: **reemplaza** el nodo de la frontera (`remove` + `heapify`, O(n)). Además desempata los `g` iguales según el montículo, no por orden de inserción, así que las métricas pueden diferir en unas pocas unidades.
- **DLS e IDDFS:** la pila de SimpleAI guarda todos los hermanos, así que su `frontera_maxima` mide otra cosa que la de nuestra DLS, que solo guarda el camino. Las iteraciones de IDDFS sin solución expanden lo mismo en ambas versiones, y la celda comprueba que toda la diferencia de IDDFS está en la última iteración.

**Dos defectos de SimpleAI en búsqueda limitada:**

1. Con `graph_search=True`, `limited_depth_first` usa un conjunto `memory` **global**: es el error del contra-experimento de la etapa 5 (D-24). En la rejilla abierta de 2×4, de `(0,0)` a `(0,3)`, **encuentra la meta con límite 3 y la pierde con límite 4**. Un límite mayor produce un fracaso.
2. `iterative_limited_depth_first` itera `while not solution`: **no distingue corte de fracaso**, así que con la meta inalcanzable no termina. Se comprobó deteniéndolo desde el visor después de |V| + 2 ejecuciones. La Versión 1 declara el fracaso tras 34.

In [ ]:
# ===========================================================================
# VERSION 2 - SimpleAI
#
# Solo se construye el ADAPTADOR del problema y la instrumentacion; los
# algoritmos son los de la biblioteca, sin tocar su codigo (D-31). El
# adaptador delega en ProblemaLaberinto: una sola formulacion para las tres
# versiones (seccion 3), asi que las acciones, su orden N-E-S-O y los costos
# son identicos por construccion.
# ===========================================================================

import simpleai
from simpleai.search import SearchProblem
from simpleai.search.traditional import (breadth_first, depth_first,
                                         uniform_cost, limited_depth_first,
                                         iterative_limited_depth_first)


class LaberintoSimpleAI(SearchProblem):
    """Adaptador: `SearchProblem` de SimpleAI sobre `ProblemaLaberinto`."""

    def __init__(self, problema):
        # El estado inicial es la tupla (fila, columna): inmutable y hashable.
        super().__init__(initial_state=problema.estado_inicial)
        self.problema = problema

    def actions(self, state):
        return self.problema.acciones(state)      # tupla en orden N-E-S-O

    def result(self, state, action):
        return self.problema.resultado(state, action)   # tupla nueva

    def is_goal(self, state):
        return self.problema.es_meta(state)

    def cost(self, state, action, state2):
        return self.problema.costo_paso(state, action, state2)


class ContadorSimpleAI:
    """Visor de SimpleAI que traduce sus eventos a las metricas del contrato.

    SimpleAI llama a `viewer.event(nombre, *parametros)` en puntos fijos de
    su bucle `_search`. No se hereda de `BaseViewer` porque este convierte la
    frontera entera a texto en cada iteracion, un costo que no aporta nada
    aqui. Correspondencia con el contrato (D-32):

      'expanded'        -> expandidos += 1, generados += hijos construidos
      'new_iteration'   -> tamano de la frontera ANTES de extraer; el maximo
                           es frontera_maxima
      insertados        -> no hay evento. Se deduce del tamano de la frontera
                           entre dos iteraciones: nuevo = viejo - 1 + insertados.
                           repetidos = generados - insertados.
      'chosen_node'     -> profundidad de lo extraido, para saber si hubo un
                           CORTE en la busqueda limitada (SimpleAI no lo dice)
      'started'         -> una ejecucion nueva (IDDFS lanza una por limite)
    """

    def __init__(self, limite=None, max_ejecuciones=None):
        self.limite = limite
        self.max_ejecuciones = max_ejecuciones
        self.ejecuciones = 0
        self.expandidos = self.generados = self.repetidos = 0
        self.frontera_maxima = 0
        self.hubo_corte = False
        self._pendiente = None        # (tamano antes de extraer, hijos generados)
        self._tamano = 0

    def event(self, nombre, *parametros):
        if nombre == "started":
            self.ejecuciones += 1
            self._pendiente = None
            if self.max_ejecuciones and self.ejecuciones > self.max_ejecuciones:
                raise RuntimeError("SimpleAI supero {0} ejecuciones".format(
                    self.max_ejecuciones))
        elif nombre in ("new_iteration", "finished"):
            tamano = len(parametros[0])
            self._cerrar_expansion(tamano)
            if nombre == "new_iteration":
                self._tamano = tamano
                self.frontera_maxima = max(self.frontera_maxima, tamano)
        elif nombre == "chosen_node":
            nodo, es_meta = parametros
            if (not es_meta and self.limite is not None
                    and nodo.depth >= self.limite):
                self.hubo_corte = True
        elif nombre == "expanded":
            hijos = parametros[1][0]
            self.expandidos += 1
            self.generados += len(hijos)
            self._pendiente = (self._tamano, len(hijos))

    def _cerrar_expansion(self, tamano_nuevo):
        if self._pendiente is None:
            return
        tamano_viejo, generados = self._pendiente
        insertados = tamano_nuevo - (tamano_viejo - 1)
        self.repetidos += generados - insertados
        self._pendiente = None


_ALGORITMOS_SIMPLEAI = {
    "BFS": breadth_first,
    "DFS": depth_first,
    "UCS": uniform_cost,
    "DLS": limited_depth_first,
    "IDDFS": iterative_limited_depth_first,
}


def resolver_simpleai(problema, algoritmo, limite=None, max_ejecuciones=None):
    """Ejecuta un algoritmo de SimpleAI y devuelve un ResultadoBusqueda.

    Se ejecuta DOS veces (D-32): una sin visor para medir el tiempo, porque
    SimpleAI ordena la frontera para el visor en cada iteracion aunque el
    visor no la use, y otra con el contador para las metricas. Las dos
    recorren lo mismo: SimpleAI es determinista con este adaptador.
    """
    adaptador = LaberintoSimpleAI(problema)
    buscar = _ALGORITMOS_SIMPLEAI[algoritmo]
    argumentos = {"graph_search": True}
    if algoritmo == "DLS":
        argumentos["depth_limit"] = limite

    tiempo_ms = math.nan
    if max_ejecuciones is None:   # con tope, la ejecucion libre podria no terminar
        inicio = time.perf_counter()
        buscar(adaptador, **argumentos)
        tiempo_ms = (time.perf_counter() - inicio) * 1000.0

    contador = ContadorSimpleAI(limite if algoritmo == "DLS" else None,
                                max_ejecuciones)
    nodo = buscar(adaptador, viewer=contador, **argumentos)

    nombre = "SAI-" + (algoritmo if limite is None else "DLS({0})".format(limite))
    if nodo is None:
        camino, acciones, costo = (), (), math.inf
    else:
        pasos = nodo.path()                      # [(accion, estado), ...]
        camino = tuple(estado for _, estado in pasos)
        acciones = tuple(accion for accion, _ in pasos[1:])
        costo = nodo.cost
    encontrado = nodo is not None
    return ResultadoBusqueda(
        encontrado, camino, acciones, costo,
        len(camino) - 1 if encontrado else -1,
        contador.expandidos, contador.generados, contador.repetidos,
        contador.frontera_maxima, tiempo_ms, nombre,
        contador.hubo_corte and not encontrado)


# ---------------------------------------------------------------------------
# 1) Los cinco algoritmos sobre la instancia individual
# ---------------------------------------------------------------------------
print("SimpleAI {0}, graph_search=True".format(
    getattr(simpleai, "__version__", "0.8.3")))
SAI = {alg: resolver_simpleai(PROBLEMA, alg) for alg in ("BFS", "DFS", "UCS", "IDDFS")}
SAI["DLS"] = resolver_simpleai(PROBLEMA, "DLS", limite=RESULTADO_BFS.profundidad)
_dls_v1 = busqueda_limitada(PROBLEMA, RESULTADO_BFS.profundidad)
for _r in SAI.values():
    verificar_resultado(PROBLEMA, _r)
    assert _r.camino == CAMINO_REFERENCIA       # arbol: un unico camino
imprimir_resultados([SAI["BFS"], RESULTADO_BFS, SAI["DFS"], RESULTADO_DFS,
                     SAI["UCS"], RESULTADO_UCS, SAI["DLS"], _dls_v1,
                     SAI["IDDFS"], RESULTADO_IDDFS])

# ---------------------------------------------------------------------------
# 2) Diagnostico de las diferencias de metricas
# ---------------------------------------------------------------------------
# BFS: misma cola FIFO, mismo orden de acciones y la misma politica de
# repetidos. SimpleAI descarta un hijo si su estado esta en `memory`
# (extraidos) o en la frontera, que es exactamente nuestro "alcanzados".
# Prediccion: TODAS las metricas iguales.
_m = lambda r: (r.expandidos, r.generados, r.repetidos_descartados, r.frontera_maxima)
assert _m(SAI["BFS"]) == _m(RESULTADO_BFS), "BFS de SimpleAI deberia coincidir"

# DFS: SimpleAI apila los hijos en el orden de `actions` y la pila devuelve
# primero el ULTIMO: explora O-S-E-N. Nuestra DFS invierte la pila (D-18).
# Prediccion: SimpleAI coincide con NUESTRA DFS SIN INVERTIR, no con la DFS.
_dfs_sin_invertir = busqueda_en_grafo(PROBLEMA, _PilaSinInvertir(), "DFS-O-S-E-N")
assert _m(SAI["DFS"]) == _m(_dfs_sin_invertir)
print()
print("BFS : SimpleAI y la Version 1 coinciden en las cuatro metricas.")
print("DFS : SimpleAI explora O-S-E-N (pila sin invertir) y coincide exactamente "
      "con la DFS de la Version 1 con la pila sin invertir: {0} expandidos.".format(
          _dfs_sin_invertir.expandidos))

# UCS: SimpleAI no deja entradas obsoletas; si un hijo mejora a un nodo de la
# frontera, lo REEMPLAZA (remove + heapify, O(n)). Y desempata por el __lt__
# del nodo (solo el costo): entre costos iguales el orden lo decide el
# monticulo, no el orden de insercion. Con costo unitario casi todo empata.
print("UCS : mismo costo. Expandidos {0} y {1}; generados {2} y {3}. SimpleAI "
      "desempata los g iguales segun el monticulo, no por orden de insercion, "
      "y la meta sale en otro momento.".format(
          SAI["UCS"].expandidos, RESULTADO_UCS.expandidos,
          SAI["UCS"].generados, RESULTADO_UCS.generados))
# Las iteraciones sin solucion expanden todo lo que esta a profundidad menor
# que el limite, en cualquier orden. Toda la diferencia de IDDFS deberia
# estar entonces en la ULTIMA iteracion, que es la DLS con limite d.
assert (SAI["IDDFS"].expandidos - RESULTADO_IDDFS.expandidos
        == SAI["DLS"].expandidos - _dls_v1.expandidos)
print("IDDFS: {0} frente a {1}. Las iteraciones sin solucion expanden lo mismo "
      "(todo lo que hay a profundidad < limite); la ultima difiere por el orden "
      "O-S-E-N.".format(SAI["IDDFS"].expandidos, RESULTADO_IDDFS.expandidos))

# ---------------------------------------------------------------------------
# 3) Dos defectos de SimpleAI en busqueda limitada, medidos
# ---------------------------------------------------------------------------
_trivial = ProblemaLaberinto(INSTANCIA.grafo, INSTANCIA.meta, INSTANCIA.meta)
_sin_salida = ProblemaLaberinto(
    grafo_sin_corredor(INSTANCIA.grafo, *CORREDOR_DE_CORTE),
    INSTANCIA.inicio, INSTANCIA.meta)
_iter_fracaso = []
profundizacion_iterativa(_sin_salida, _iter_fracaso)
# a) limited_depth_first(graph_search=True) usa un `memory` GLOBAL: el
#    mismo defecto que el contra-experimento de la etapa 5 (D-24). Con su
#    orden O-S-E-N el caso de la etapa 5 no lo dispara; una busqueda
#    exhaustiva en rejillas abiertas encontro este: de (0,0) a (0,3) en la
#    2x4 abierta (distancia 3), SimpleAI halla la meta con limite 3 y la
#    PIERDE con limite 4. Un limite MAYOR produce un fracaso.
_p_sai = ProblemaLaberinto(GRAFO_ABIERTO_2x4, (0, 0), (0, 3))
_sai_l3 = resolver_simpleai(_p_sai, "DLS", limite=3)
_sai_l4 = resolver_simpleai(_p_sai, "DLS", limite=4)
assert _sai_l3.encontrado and not _sai_l4.encontrado
assert busqueda_limitada(_p_sai, 4).encontrado
print()
print("Rejilla abierta 2x4, (0,0) -> (0,3), distancia 3:")
print("  SimpleAI DLS limite 3 -> {0} pasos; limite 4 -> NO encuentra "
      "(corte={1}).".format(_sai_l3.profundidad, _sai_l4.corte))
print("  Version 1 DLS limite 4 -> {0} pasos.".format(
    busqueda_limitada(_p_sai, 4).profundidad))

# b) iterative_limited_depth_first repite `while not solution`: no distingue
#    corte de fracaso, y sin solucion NO TERMINA. Se detiene desde fuera con
#    el visor tras |V| + 2 ejecuciones (la Version 1 declara fracaso en 34).
try:
    resolver_simpleai(_sin_salida, "IDDFS", max_ejecuciones=len(_sin_salida.grafo) + 2)
    _no_termina = False
except RuntimeError:
    _no_termina = True
assert _no_termina
print("Meta inalcanzable: SimpleAI IDDFS sigue despues de {0} ejecuciones (se "
      "corto desde el visor); la Version 1 declara fracaso tras {1}.".format(
          len(_sin_salida.grafo) + 2, len(_iter_fracaso)))

# ---------------------------------------------------------------------------
# 4) Costos: UCS de SimpleAI en el grafo ponderado
# ---------------------------------------------------------------------------
_sai_ucs_p = resolver_simpleai(PROBLEMA_PONDERADO, "UCS")
verificar_resultado(PROBLEMA_PONDERADO, _sai_ucs_p)
assert _sai_ucs_p.costo == ucs(PROBLEMA_PONDERADO).costo == 12
print("Grafo ponderado 2x4: UCS de SimpleAI costo {0}, igual que la Version 1.".format(
    _sai_ucs_p.costo))

# ---------------------------------------------------------------------------
# 5) Casos limite
# ---------------------------------------------------------------------------
for _alg in ("BFS", "DFS", "UCS", "IDDFS"):
    _r = resolver_simpleai(_trivial, _alg)
    verificar_resultado(_trivial, _r)
    assert _r.camino == (INSTANCIA.meta,) and _r.expandidos == 0
for _alg in ("BFS", "DFS", "UCS"):
    _r = resolver_simpleai(_sin_salida, _alg)
    verificar_resultado(_sin_salida, _r)
    assert not _r.encontrado
print("Casos limite: inicio == meta (0 expandidos) y meta inalcanzable "
      "(BFS, DFS, UCS devuelven None -> fracaso) verificados.")

# Versión 3 — AIMA-Python

Repositorio de referencia: `aimacode/aima-python`.

Debe modelar el laberinto como una subclase de `Problem`. Investigue y documente los contratos de:

| Método de AIMA-Python | Decisión requerida |
|---|---|
| `actions(state)` | Formato de las acciones y orden determinista |
| `result(state, action)` | Transición entre celdas |
| `goal_test(state)` | Prueba de objetivo |
| `path_cost(c, state1, action, state2)` | Acumulación de costos |

Compare al menos:

- `depth_first_graph_search`;
- `breadth_first_graph_search`;
- `uniform_cost_search`;
- `depth_limited_search`;
- `iterative_deepening_search`;
- `bidirectional_search`.

### Exigencia avanzada

Instrumente los algoritmos sin alterar permanentemente el repositorio. Puede utilizar una subclase del problema, envoltorios o contadores cuidadosamente definidos. Explique por qué contar llamadas a `actions` no siempre coincide con contar nodos generados.

Compare la representación del nodo de AIMA-Python con su propia clase de nodo.

### V3.1 Contratos de AIMA-Python, instrumentación y comparación de nodos

**Procedencia.** `aima/search.py`, `aima/utils.py` y `aima/__init__.py` se copiaron **sin modificar** de `aimacode/aima-python`, rama `master`, commit `bbf6bc2` (2026-06-29), con su licencia MIT en `aima/LICENSE` (D-05, D-34).

**Decisiones sobre los contratos de `Problem`:**

| Método | Decisión |
|---|---|
| `actions(state)` | Devuelve la **tupla** de direcciones legales en el orden fijo N-E-S-O de `ProblemaLaberinto`, sin iterar un conjunto: es determinista. |
| `result(state, action)` | Delega en `ProblemaLaberinto.resultado`, que devuelve una tupla nueva y **falla** ante una acción con pared. |
| `goal_test(state)` | `state == meta`. Se redefine, aunque la versión por omisión de AIMA ya compara con `self.goal`, porque aquí vive un contador. |
| `path_cost(c, s, a, s′)` | AIMA **acumula**: recibe el costo hasta `s` y devuelve el total `c + c(s, a, s′)`. `bidirectional_search` la llama con `a = None`; es válido porque el costo es de la arista (D-23). |
| `h(node)` | `0`. `bidirectional_search` la exige, y con `h = 0` es la variante no informada MM0. |

**Instrumentación (D-35).** AIMA no tiene visor: sus funciones solo hablan con el problema. Por eso los contadores viven en la subclase: cada llamada a `actions` es una **expansión** (`Node.expand` la hace una vez), cada llamada a `result` es un **nodo generado** (`child_node`, una por hijo) y los estados distintos vistos son los **insertados**. En búsqueda en grafo, el tamaño de la frontera es *1 + insertados − expandidos*. En la DLS recursiva, la altura de la pila se reconstruye a partir de la secuencia de `goal_test` y `actions`.

**Por qué contar llamadas a `actions` no es contar nodos generados.** Una llamada a `actions` produce **varios** hijos. En la BFS de la instancia hubo 241 llamadas a `actions`, 489 a `result` y 250 a `goal_test`: tres números distintos para la misma búsqueda. Además, `actions` puede invocarse fuera de la búsqueda (un validador, un dibujo) e inflar la cuenta. Y si `actions` devolviera un generador, como sugiere la documentación de AIMA, una búsqueda que se detiene a mitad de una expansión no construiría todos los hijos anunciados.

**Nodo de AIMA frente a `Nodo` propio:**

| Aspecto | `aima.search.Node` | `Nodo` (Versión 1) |
|---|---|---|
| Campos | `state`, `parent`, `action`, `path_cost`, `depth` | `estado`, `padre`, `accion`, `g`, `profundidad`: los mismos |
| Igualdad | `__eq__` y `__hash__` **por estado**: dos nodos con el mismo estado son «iguales» | por **identidad**: dos nodos son distintos aunque compartan estado (sección 3.1) |
| Orden | `__lt__` compara **estados** | no define `<`; UCS desempata con un contador (D-21) |
| Memoria | `__dict__` por instancia | `__slots__` |
| Expansión | `expand` construye la lista completa de hijos | `hijo` construye uno; DLS genera de a uno |

La igualdad por estado le permite escribir `child not in frontier`, pero **confunde nodo y estado**, justo lo que la sección 3.1 separa. Y esa pertenencia sobre una lista o sobre la `PriorityQueue` es O(n) por hijo.

**Diferencias medidas y su causa:**

- **BFS:** AIMA prueba el objetivo **al generar** (figura 3.11 del libro) y expande 241 estados frente a 249. La celda comprueba que 241 es exactamente la posición, en nuestro orden de expansión, del padre de la meta.
- **DFS:** idéntica a SimpleAI. Apila en orden N-E-S-O y extrae el último, así que explora O-S-E-N.
- **UCS:** idéntica a la Versión 1 en las cuatro métricas. Su `PriorityQueue` desempata con un contador de inserción, igual que el nuestro, y en un árbol nunca reemplaza.
- **DLS e IDDFS:** `recursive_dls` es búsqueda en **árbol sin ningún control de ciclos**. En un grafo no dirigido recorre caminatas (ir y volver cuenta), así que con límite 16 genera 284 650 nodos donde la Versión 1 genera 82. Hasta la profundidad de la meta, 59, el árbol de caminatas tiene unos 4 × 10¹⁹ nodos: en la instancia **no se pueden ejecutar**. Se comparan en el 3×3, donde las tres versiones coinciden en el camino. A favor de AIMA: **sí** distingue corte (`'cutoff'`) de fracaso (`None`).
- **Bidireccional:** `bidirectional_search` implementa MM (Holte et al., 2016) y **devuelve solo el costo**, no un nodo ni un camino, así que no puede llenar el contrato. Se compara el costo. Con `h = 0` es óptima en **costo**: da 12 en el grafo ponderado, como UCS, mientras que nuestra bidireccional por capas da 17 porque es óptima en **pasos**.

In [ ]:
# ===========================================================================
# VERSION 3 - AIMA-Python
#
# aima/search.py y aima/utils.py estan VENDORIZADOS sin modificar (D-05):
# copia de aimacode/aima-python, rama master, commit bbf6bc2 (2026-06-29),
# licencia MIT en aima/LICENSE. La instrumentacion va en una subclase de
# Problem; los archivos de la biblioteca no se tocan (D-34).
# ===========================================================================

from aima.search import (Problem, Node as NodoAIMA, breadth_first_graph_search,
                         depth_first_graph_search, uniform_cost_search,
                         depth_limited_search, iterative_deepening_search,
                         bidirectional_search)


class LaberintoAIMA(Problem):
    """Adaptador `Problem` de AIMA sobre `ProblemaLaberinto`, con contadores.

    AIMA no ofrece visor ni ganchos: sus busquedas solo hablan con el
    problema. Por eso los contadores viven AQUI, en los metodos que la
    biblioteca invoca (D-35):

      actions(s)   una llamada por nodo EXPANDIDO (Node.expand la hace una vez)
      result(s,a)  una llamada por nodo GENERADO (Node.child_node, una por hijo)
      estados nuevos distintos -> insertados; repetidos = generados - insertados
      frontera     en busqueda en grafo, tamano = 1 + insertados - expandidos
      profundidad  en la DLS recursiva se reconstruye la pila de llamadas a
                   partir de la secuencia goal_test / actions (ver _entrar)
    """

    def __init__(self, problema):
        super().__init__(problema.estado_inicial, problema.meta)
        self.problema = problema
        self.reiniciar()

    def reiniciar(self):
        self.llamadas_actions = 0
        self.llamadas_result = 0
        self.llamadas_goal_test = 0
        self._vistos = {self.initial}
        self.frontera_maxima = 1
        self._pila = []                # [pendientes] por nivel de la DLS
        self.profundidad_maxima = 0

    # --- contrato de AIMA -------------------------------------------------
    def actions(self, state):
        acciones = self.problema.acciones(state)     # N-E-S-O, determinista
        self.llamadas_actions += 1
        self._pila.append(len(acciones))
        return acciones

    def result(self, state, action):
        siguiente = self.problema.resultado(state, action)
        self.llamadas_result += 1
        if siguiente not in self._vistos:
            self._vistos.add(siguiente)
        tamano = 1 + (len(self._vistos) - 1) - self.llamadas_actions
        self.frontera_maxima = max(self.frontera_maxima, tamano)
        return siguiente

    def goal_test(self, state):
        self.llamadas_goal_test += 1
        self._entrar()
        return self.problema.es_meta(state)

    def path_cost(self, c, state1, action, state2):
        # AIMA acumula: recibe el costo hasta state1 y devuelve el nuevo total.
        # bidirectional_search llama con action=None; c(s, a, s') por arista
        # no depende de la accion (D-23), asi que es valido.
        return c + self.problema.costo_paso(state1, action, state2)

    def h(self, node):
        return 0      # busqueda NO informada: bidirectional_search exige h

    # --- reconstruccion de la pila de la DLS recursiva --------------------
    def _entrar(self):
        """En recursive_dls, cada goal_test es la entrada a un nodo. Los
        niveles sin hijos pendientes ya terminaron; el nodo nuevo cuelga del
        nivel que queda arriba, y su profundidad es la altura de la pila."""
        while self._pila and self._pila[-1] == 0:
            self._pila.pop()
        if self._pila:
            self._pila[-1] -= 1
        self.profundidad_maxima = max(self.profundidad_maxima, len(self._pila))


_ALGORITMOS_AIMA = {
    "BFS": breadth_first_graph_search,
    "DFS": depth_first_graph_search,
    "UCS": uniform_cost_search,
    "DLS": depth_limited_search,
    "IDDFS": iterative_deepening_search,
}


def resolver_aima(problema, algoritmo, limite=None):
    """Ejecuta un algoritmo de AIMA y devuelve un ResultadoBusqueda."""
    adaptador = LaberintoAIMA(problema)
    buscar = _ALGORITMOS_AIMA[algoritmo]
    argumentos = {"limit": limite} if algoritmo == "DLS" else {}
    inicio = time.perf_counter()
    salida = buscar(adaptador, **argumentos)
    tiempo_ms = (time.perf_counter() - inicio) * 1000.0

    corte = salida == "cutoff"        # AIMA SI distingue corte de fracaso
    nodo = salida if isinstance(salida, NodoAIMA) else None
    if nodo is None:
        camino, acciones, costo = (), (), math.inf
    else:
        camino, acciones, costo = tuple(nodo.path_states()), tuple(nodo.solution()), nodo.path_cost
    encontrado = nodo is not None
    arbol = algoritmo in ("DLS", "IDDFS")
    generados = adaptador.llamadas_result
    return ResultadoBusqueda(
        encontrado, camino, acciones, costo,
        len(camino) - 1 if encontrado else -1,
        adaptador.llamadas_actions, generados,
        # En busqueda en arbol AIMA no descarta nada: no hay repetidos.
        0 if arbol else generados - (len(adaptador._vistos) - 1),
        adaptador.profundidad_maxima + 1 if arbol else adaptador.frontera_maxima,
        tiempo_ms,
        "AIMA-" + (algoritmo if limite is None else "DLS({0})".format(limite)),
        corte)


# ---------------------------------------------------------------------------
# 1) BFS, DFS y UCS en grafo sobre la instancia individual
# ---------------------------------------------------------------------------
AIMA = {alg: resolver_aima(PROBLEMA, alg) for alg in ("BFS", "DFS", "UCS")}
for _r in AIMA.values():
    verificar_resultado(PROBLEMA, _r)
    assert _r.camino == CAMINO_REFERENCIA
print("AIMA-Python (commit bbf6bc2), instancia individual")
imprimir_resultados([AIMA["BFS"], RESULTADO_BFS, SAI["BFS"],
                     AIMA["DFS"], RESULTADO_DFS, SAI["DFS"],
                     AIMA["UCS"], RESULTADO_UCS, SAI["UCS"]])

# a) BFS de AIMA prueba el objetivo AL GENERAR (figura 3.11 del libro): se
#    detiene al construir la meta, sin extraer el resto de su nivel.
#    Prediccion: sus expandidos son la posicion, en el orden de expansion de
#    nuestra BFS, del padre de la meta.
_orden = [t["extraido"] for t in _traza_bfs if not t["meta"]]
_padre_meta = CAMINO_REFERENCIA[-2]
assert AIMA["BFS"].expandidos == _orden.index(_padre_meta) + 1
# b) DFS de AIMA apila los hijos en orden y extrae el ultimo: O-S-E-N, igual
#    que SimpleAI. Prediccion: las cuatro metricas de SimpleAI.
assert _m(AIMA["DFS"]) == _m(SAI["DFS"]) == _m(_dfs_sin_invertir)
# c) UCS de AIMA desempata con un contador de insercion (PriorityQueue), como
#    la Version 1, y en un arbol nunca reemplaza. Prediccion: nuestras metricas.
assert _m(AIMA["UCS"]) == _m(RESULTADO_UCS)
print()
print("BFS : AIMA expande {0}: prueba la meta al GENERARLA y se detiene al "
      "expandir su padre, el {0}.o de nuestra BFS.".format(AIMA["BFS"].expandidos))
print("DFS : identica a SimpleAI y a nuestra DFS con pila sin invertir (O-S-E-N).")
print("UCS : identica a la Version 1 en las cuatro metricas (mismo desempate FIFO).")

# ---------------------------------------------------------------------------
# 2) DLS e IDDFS de AIMA: busqueda en ARBOL sin control de ciclos
# ---------------------------------------------------------------------------
# recursive_dls no mira ni el camino ni un conjunto de alcanzados: en un
# grafo no dirigido recorre CAMINATAS (ir y volver cuenta). Se mide el
# crecimiento en la instancia con limites pequenos y se cuenta, con
# programacion dinamica, cuantos nodos tendria el arbol hasta la profundidad
# de la meta.
print()
print("DLS de AIMA en la instancia (sin solucion dentro del limite):")
print("  limite   generados AIMA   generados Version 1   resultado AIMA")
for _L in [L for L in (4, 8, 12, 16) if L < RESULTADO_BFS.profundidad]:
    _ra, _rv = resolver_aima(PROBLEMA, "DLS", _L), busqueda_limitada(PROBLEMA, _L)
    assert _ra.corte and not _ra.encontrado
    print("  {0:>6}   {1:>14,}   {2:>19,}   corte".format(_L, _ra.generados, _rv.generados))

_caminatas, _nivel = 0, {INSTANCIA.inicio: 1}
for _ in range(RESULTADO_BFS.profundidad):
    _siguiente = {}
    for _e, _k in _nivel.items():
        for _v in INSTANCIA.grafo[_e]:
            _siguiente[_v] = _siguiente.get(_v, 0) + _k
    _nivel = _siguiente
    _caminatas += sum(_nivel.values())
print("  Hasta la profundidad {0} el arbol de caminatas tiene {1:.2e} nodos: "
      "DLS e IDDFS de AIMA no son ejecutables aqui.".format(
          RESULTADO_BFS.profundidad, _caminatas))

# Donde si caben: el laberinto 3x3. Las tres versiones coinciden en camino
# con limite EXACTO. Con un limite mayor, la DLS de AIMA puede devolver una
# caminata con idas y vueltas: no controla ciclos.
_d3 = _r_bfs_3x3.profundidad
_ra_it = resolver_aima(PROBLEMA_3x3, "IDDFS")
_ra_dls = resolver_aima(PROBLEMA_3x3, "DLS", _d3)
for _r in (_ra_it, _ra_dls):
    verificar_resultado(PROBLEMA_3x3, _r)
    assert _r.camino == _r_bfs_3x3.camino
imprimir_resultados([_ra_dls, busqueda_limitada(PROBLEMA_3x3, _d3),
                     _ra_it, profundizacion_iterativa(PROBLEMA_3x3)])
# AIMA SI distingue corte de fracaso ('cutoff' frente a None).
assert resolver_aima(PROBLEMA_3x3, "DLS", _d3 - 1).corte

# ---------------------------------------------------------------------------
# 3) bidirectional_search: MM (Holte et al., 2016), que devuelve SOLO el costo
# ---------------------------------------------------------------------------
# No devuelve nodo ni camino, asi que no puede llenar el contrato: se compara
# el costo optimo. Con h = 0 es MM0, una bidireccional optima EN COSTO.
_mm = bidirectional_search(LaberintoAIMA(PROBLEMA))
_mm_p = bidirectional_search(LaberintoAIMA(PROBLEMA_PONDERADO))
assert _mm == RESULTADO_BIDIR.costo == RESULTADO_BFS.costo
assert _mm_p == ucs(PROBLEMA_PONDERADO).costo
print()
print("bidirectional_search (MM, h=0): costo {0} en la instancia (igual que "
      "BIDIR y BFS); costo {1} en el grafo ponderado, igual que UCS. Nuestra "
      "BIDIR por capas da {2} ahi: MM es optima en costo, la nuestra en "
      "pasos.".format(_mm, _mm_p, bidireccional(PROBLEMA_PONDERADO).costo))

# ---------------------------------------------------------------------------
# 4) Por que contar llamadas a actions no es contar nodos generados
# ---------------------------------------------------------------------------
_c = LaberintoAIMA(PROBLEMA)
breadth_first_graph_search(_c)
print()
print("En BFS de AIMA: {0} llamadas a actions, {1} a result, {2} a goal_test.".format(
    _c.llamadas_actions, _c.llamadas_result, _c.llamadas_goal_test))
assert _c.llamadas_actions < _c.llamadas_result

# Casos limite en AIMA
for _alg in ("BFS", "DFS", "UCS", "IDDFS"):
    _r = resolver_aima(_trivial, _alg)
    verificar_resultado(_trivial, _r)
    assert _r.camino == (INSTANCIA.meta,)
for _alg in ("BFS", "DFS", "UCS"):
    assert not resolver_aima(_sin_salida, _alg).encontrado
print("Casos limite en AIMA: inicio == meta y meta inalcanzable verificados.")

## 8. Pruebas de aceptación

El estudiante debe implementar pruebas automáticas equivalentes a las siguientes especificaciones, sin recibir su solución:

### Validez de un camino

Para cada resultado exitoso:

1. el primer estado es el inicio;
2. el último estado es la meta;
3. cada pareja consecutiva aparece como arista en el grafo;
4. el costo reportado coincide con la suma de costos;
5. aplicar las acciones reproduce exactamente los estados.

### Concordancia entre versiones

- En costos unitarios: `longitud(BFS) = longitud(UCS) = etiqueta_meta(Lee)`.
- En un árbol: todas las soluciones válidas contienen la misma secuencia de estados.
- Con ciclos: `longitud(BFS) ≤ longitud(DFS)` para una DFS que encuentre solución.
- Con costos positivos: `costo(UCS) ≤ costo(cualquier solución BFS)`.
- Las tres versiones deben concordar en existencia de solución y costo óptimo.

### Casos límite obligatorios

- inicio igual a meta;
- laberinto de `1×1`;
- meta inalcanzable después de eliminar corredores;
- múltiples entradas obsoletas en UCS;
- dos posibles puntos de encuentro bidireccional;
- límite exacto, insuficiente y excesivo en búsqueda limitada.

### 8.1 Diseño de la batería

**Tres familias de instancias, todas con semilla fija:**
- **Árboles:** 12 laberintos Hunt-and-Kill de 4×4 a 8×8, más la instancia individual y el 3×3.
- **Con ciclos:** 15 rejillas abiertas a las que se les quita el 25 % de las aristas, más la rejilla abierta 2×4. Costo unitario.
- **Ponderados:** 15 rejillas con ciclos y costos por arista entre 1 y 9, más el grafo ponderado de 2×4.

Algunas instancias con ciclos quedan desconectadas. Así la concordancia en existencia de solución se prueba también con respuestas negativas.

**Qué se ejecuta.** `ejecutar_todas` corre los siete algoritmos de la Versión 1, los cinco de SimpleAI y los de AIMA-Python sobre cada instancia. Hay exclusiones **documentadas, no ocultas**:
- La IDDFS de SimpleAI solo se ejecuta si hay solución, y además acotada desde el visor, porque puede no terminar (D-33).
- La DLS y la IDDFS de AIMA solo se ejecutan si el árbol de caminatas cabe en 200 000 nodos (D-36).
- La bidireccional de AIMA se compara solo por su costo, que es lo único que devuelve.

**Cómo se prueba.** `condiciones_de_validez` comprueba **por separado** las cinco condiciones del enunciado y devuelve cuáles se incumplen. Así un fallo dice *qué* condición rompe. Cada prueba de validez o de concordancia lleva una **contra-prueba** que debe fallar (D-11):
- cinco mutaciones de un resultado válido, cada una pensada para romper una sola condición;
- la unicidad del camino, que debe **dejar** de cumplirse con ciclos;
- las desigualdades `BFS ≤ DFS` y `UCS ≤ BFS`, que además se exige que sean **estrictas** en al menos un caso, para no pasar por tratarse siempre de igualdades.

**Casos límite:** los seis que exige el enunciado, más la reproducción del defecto de la DLS de SimpleAI (D-33).

In [ ]:
# ===========================================================================
# Seccion 8 - Pruebas de aceptacion
#
# Una bateria automatica. Cada prueba DEVUELVE (aprobada, detalle) y el
# informe final hace `assert` sobre todas: si una falla, el cuaderno se
# detiene aqui. Toda prueba de validez o de concordancia se acompana de un
# caso que DEBE fallar (D-11): una prueba que no puede fallar no prueba nada.
# ===========================================================================

# ---------------------------------------------------------------------------
# 1) Validez de un camino: las cinco condiciones del enunciado, por separado
# ---------------------------------------------------------------------------
from dataclasses import replace

def condiciones_de_validez(problema, r):
    """Devuelve la lista de condiciones que r INCUMPLE (vacia = valido).

    Se comprueban por separado, y no con un unico assert, para que un
    resultado incorrecto diga QUE condicion rompe.
    """
    if not r.encontrado:
        return [] if (r.camino == () and r.acciones == ()) else ["fracaso con camino"]
    fallas = []
    camino, acciones = r.camino, r.acciones
    if camino[0] != problema.estado_inicial:
        fallas.append("1 inicio")
    if not problema.es_meta(camino[-1]):
        fallas.append("2 meta")
    if any(v not in problema.grafo.get(u, ()) for u, v in zip(camino, camino[1:])):
        fallas.append("3 aristas")
    try:
        costo = sum(problema.costo_paso(u, None, v) for u, v in zip(camino, camino[1:]))
        if costo != r.costo:
            fallas.append("4 costo")
    except KeyError:
        fallas.append("4 costo")
    estado, reproducido = problema.estado_inicial, [problema.estado_inicial]
    try:
        for accion in acciones:
            estado = problema.resultado(estado, accion)
            reproducido.append(estado)
    except (ValueError, KeyError):
        reproducido = None
    if reproducido is None or tuple(reproducido) != camino:
        fallas.append("5 acciones")
    return fallas


def _mutaciones(problema, r):
    """Cinco resultados corrompidos a partir de uno valido (con >= 3 estados),
    cada uno pensado para romper UNA condicion."""
    c, a = r.camino, r.acciones
    otra = next(x for x in sorted(problema.grafo) if x not in c)
    return {
        "1 inicio": replace(r, camino=(otra,) + c[1:]),
        "2 meta": replace(r, camino=c[:-1] + (otra,)),
        "3 aristas": replace(r, camino=(c[0],) + c[2:]),     # salta un estado
        "4 costo": replace(r, costo=r.costo + 1),
        "5 acciones": replace(r, acciones=(ACCION_INVERSA[a[0]],) + a[1:]),
    }


# ---------------------------------------------------------------------------
# 2) Ejecutar los algoritmos de las tres versiones sobre un problema
# ---------------------------------------------------------------------------
def nodos_arbol_de_caminatas(grafo, inicio, limite):
    """Nodos que generaria una busqueda en arbol SIN control de ciclos hasta
    `limite`. Decide si es viable ejecutar la DLS/IDDFS de AIMA (D-36)."""
    total, nivel = 0, {inicio: 1}
    for _ in range(limite):
        siguiente = {}
        for e, k in nivel.items():
            for v in grafo[e]:
                siguiente[v] = siguiente.get(v, 0) + k
        nivel = siguiente
        total += sum(nivel.values())
    return total


TOPE_CAMINATAS_AIMA = 200_000


def ejecutar_todas(problema):
    """{nombre: ResultadoBusqueda} para todo lo ejecutable en `problema`.

    Exclusiones deliberadas, documentadas en D-33 y D-36:
    - SimpleAI IDDFS solo si hay solucion (sin ella no termina).
    - AIMA DLS/IDDFS solo si el arbol de caminatas cabe en el tope.
    - AIMA bidirectional_search devuelve un costo, no un resultado: aparte.
    """
    r = {"BFS": bfs(problema), "DFS": dfs(problema), "UCS": ucs(problema),
         "IDDFS": profundizacion_iterativa(problema),
         "BIDIR": bidireccional(problema), "LEE": lee(problema)}
    hay = r["BFS"].encontrado
    limite = r["BFS"].profundidad if hay else len(problema.grafo)
    r["DLS"] = busqueda_limitada(problema, limite)
    for alg in ("BFS", "DFS", "UCS"):
        r["SAI-" + alg] = resolver_simpleai(problema, alg)
        r["AIMA-" + alg] = resolver_aima(problema, alg)
    r["SAI-DLS"] = resolver_simpleai(problema, "DLS", limite=limite)
    if hay:
        # Con ciclos, la DLS de SimpleAI puede fallar en todo limite (D-33) y
        # su IDDFS no terminaria: se acota desde el visor y se registra.
        try:
            r["SAI-IDDFS"] = resolver_simpleai(problema, "IDDFS",
                                               max_ejecuciones=len(problema.grafo) + 2)
        except RuntimeError:
            SAI_IDDFS_SIN_TERMINAR.append(problema)
    if nodos_arbol_de_caminatas(problema.grafo, problema.estado_inicial,
                                limite) <= TOPE_CAMINATAS_AIMA:
        r["AIMA-DLS"] = resolver_aima(problema, "DLS", limite)
        if hay:
            r["AIMA-IDDFS"] = resolver_aima(problema, "IDDFS")
    return r


SAI_IDDFS_SIN_TERMINAR = []


def costo_mm(problema):
    return bidirectional_search(LaberintoAIMA(problema))


# ---------------------------------------------------------------------------
# 3) Familias de instancias de prueba (todas con semilla fija)
# ---------------------------------------------------------------------------
_rng_pruebas = random.Random(INSTANCIA.semilla)


def _subgrafo_con_ciclos(filas, columnas, quitar, rng):
    """Rejilla abierta a la que se le quita una fraccion de aristas."""
    abierta = rejilla_abierta(filas, columnas)
    g = {c: set(v) for c, v in abierta.items()}
    for u in sorted(abierta):
        for v in sorted(abierta[u]):
            if u < v and rng.random() < quitar:
                g[u].discard(v)
                g[v].discard(u)
    return MappingProxyType({c: frozenset(v) for c, v in g.items()})


def _costos_aleatorios(grafo, rng, maximo=9):
    return {frozenset((u, v)): rng.randint(1, maximo)
            for u in sorted(grafo) for v in sorted(grafo[u]) if u < v}


def _extremos(grafo, rng):
    return tuple(rng.sample(sorted(grafo), 2))


ARBOLES, CICLICOS, PONDERADOS = [], [], []
for _k in range(12):                                      # laberintos perfectos
    _f, _c = _rng_pruebas.choice([(4, 4), (5, 6), (6, 6), (8, 8)])
    _g = MappingProxyType({c: frozenset(v) for c, v in
                           LaberintoHuntKill(_f, _c, _rng_pruebas.randrange(10**6)).generar().items()})
    ARBOLES.append(ProblemaLaberinto(_g, *_extremos(_g, _rng_pruebas)))
ARBOLES.append(PROBLEMA)
ARBOLES.append(PROBLEMA_3x3)
for _k in range(15):                                      # con ciclos, costo 1
    _g = _subgrafo_con_ciclos(*_rng_pruebas.choice([(3, 3), (3, 4), (4, 4), (4, 5)]),
                              0.25, _rng_pruebas)
    CICLICOS.append(ProblemaLaberinto(_g, *_extremos(_g, _rng_pruebas)))
CICLICOS.append(PROBLEMA_ABIERTO)
for _k in range(15):                                      # con ciclos y costos
    _g = _subgrafo_con_ciclos(*_rng_pruebas.choice([(3, 3), (3, 4), (4, 4), (4, 5)]),
                              0.2, _rng_pruebas)
    PONDERADOS.append(ProblemaLaberinto(_g, *_extremos(_g, _rng_pruebas),
                                        costo_por_arista(_costos_aleatorios(_g, _rng_pruebas))))
PONDERADOS.append(PROBLEMA_PONDERADO)

EJECUCIONES = {id(p): ejecutar_todas(p) for p in ARBOLES + CICLICOS + PONDERADOS}

# ---------------------------------------------------------------------------
# 4) La bateria
# ---------------------------------------------------------------------------
PRUEBAS = []


def prueba(nombre):
    def registrar(funcion):
        PRUEBAS.append((nombre, funcion))
        return funcion
    return registrar


def _todos():
    for familia, problemas in (("arbol", ARBOLES), ("ciclos", CICLICOS),
                               ("ponderado", PONDERADOS)):
        for p in problemas:
            yield familia, p, EJECUCIONES[id(p)]


@prueba("Validez: las 5 condiciones en todo resultado de las 3 versiones")
def _():
    n = 0
    for _, p, rs in _todos():
        for nombre, r in rs.items():
            fallas = condiciones_de_validez(p, r)
            if fallas:
                return False, "{0}: {1}".format(nombre, fallas)
            n += 1
    return True, "{0} resultados validos".format(n)


@prueba("Validez (contra-prueba): cada mutacion rompe SU condicion")
def _():
    n = 0
    for _, p, rs in _todos():
        r = rs["BFS"]
        if not r.encontrado or len(r.camino) < 3:
            continue
        for condicion, mutado in _mutaciones(p, r).items():
            if condicion not in condiciones_de_validez(p, mutado):
                return False, "la mutacion '{0}' no se detecto".format(condicion)
            n += 1
    return True, "{0} mutaciones detectadas, cada una por su condicion".format(n)


@prueba("Costo unitario: long(BFS) = long(UCS) = etiqueta_meta(Lee) = IDDFS = BIDIR")
def _():
    n = 0
    for familia, p, rs in _todos():
        if familia == "ponderado" or not rs["BFS"].encontrado:
            continue
        registro = {}
        lee(p, registro)
        longitudes = {rs[a].profundidad for a in ("BFS", "UCS", "IDDFS", "BIDIR",
                                                  "SAI-BFS", "SAI-UCS", "AIMA-BFS",
                                                  "AIMA-UCS")}
        if longitudes != {registro["etiqueta"][p.meta]}:
            return False, "longitudes {0}".format(longitudes)
        n += 1
    return True, "{0} problemas de costo unitario".format(n)


@prueba("Arbol: todas las soluciones validas tienen la misma secuencia de estados")
def _():
    n = 0
    for familia, p, rs in _todos():
        if familia != "arbol":
            continue
        caminos = {r.camino for r in rs.values() if r.encontrado}
        if len(caminos) != 1:
            return False, "{0} caminos distintos".format(len(caminos))
        n += len(rs)
    return True, "{0} soluciones, un unico camino por problema".format(n)


@prueba("Arbol (contra-prueba): con ciclos, al menos un problema tiene caminos distintos")
def _():
    distintos = sum(len({r.camino for r in rs.values() if r.encontrado}) > 1
                    for familia, p, rs in _todos() if familia == "ciclos")
    return distintos > 0, "{0} problemas con ciclos dan caminos distintos".format(distintos)


@prueba("Ciclos: long(BFS) <= long(DFS) para toda DFS que encuentra solucion")
def _():
    n = estricto = 0
    for familia, p, rs in _todos():
        if familia != "ciclos" or not rs["BFS"].encontrado:
            continue
        for a in ("DFS", "SAI-DFS", "AIMA-DFS", "DLS"):
            if rs[a].encontrado:
                if rs["BFS"].profundidad > rs[a].profundidad:
                    return False, a
                estricto += rs["BFS"].profundidad < rs[a].profundidad
                n += 1
    return estricto > 0, "{0} comparaciones; en {1} la DFS es estrictamente mas larga".format(
        n, estricto)


@prueba("Costos positivos: costo(UCS) <= costo(toda solucion BFS), en las 3 versiones")
def _():
    n = estricto = 0
    for familia, p, rs in _todos():
        if familia != "ponderado" or not rs["UCS"].encontrado:
            continue
        for u in ("UCS", "SAI-UCS", "AIMA-UCS"):
            for b in ("BFS", "SAI-BFS", "AIMA-BFS", "BIDIR", "LEE"):
                if rs[u].costo > rs[b].costo:
                    return False, "{0} > {1}".format(u, b)
                estricto += rs[u].costo < rs[b].costo
                n += 1
    return estricto > 0, "{0} comparaciones; en {1} UCS es estrictamente mas barata".format(
        n, estricto)


@prueba("Las 3 versiones concuerdan en existencia de solucion y costo optimo")
def _():
    n = 0
    for _, p, rs in _todos():
        existe = {rs[a].encontrado for a in ("BFS", "UCS", "SAI-BFS", "SAI-UCS",
                                             "AIMA-BFS", "AIMA-UCS")}
        costos = {rs[a].costo for a in ("UCS", "SAI-UCS", "AIMA-UCS")} | {costo_mm(p)}
        if len(existe) != 1 or len(costos) != 1:
            return False, "existencia {0}, costos {1}".format(existe, costos)
        n += 1
    return True, "{0} problemas; incluye el costo de bidirectional_search (MM)".format(n)


@prueba("Caso limite: inicio igual a meta")
def _():
    p = ProblemaLaberinto(INSTANCIA.grafo, INSTANCIA.meta, INSTANCIA.meta)
    rs = ejecutar_todas(p)
    malos = [a for a, r in rs.items()
             if not (r.encontrado and r.camino == (p.meta,) and r.costo == 0)]
    return not malos, "{0} algoritmos devuelven el camino de 1 estado".format(len(rs)) \
        if not malos else str(malos)


@prueba("Caso limite: laberinto 1x1")
def _():
    inst = construir_instancia(INSTANCIA.semilla, 1, 1, (0, 0), (0, 0),
                               exigir_cuadrantes_opuestos=False)
    p = ProblemaLaberinto.desde_instancia(inst)
    rs = ejecutar_todas(p)
    ok = all(r.encontrado and r.camino == ((0, 0),) and r.expandidos == 0
             for r in rs.values())
    return ok, "{0} algoritmos: camino ((0, 0),), 0 expandidos".format(len(rs))


@prueba("Caso limite: meta inalcanzable tras eliminar corredores")
def _():
    rs = ejecutar_todas(_sin_salida)
    ok = all(not r.encontrado and condiciones_de_validez(_sin_salida, r) == []
             for r in rs.values())
    ok = ok and not rs["IDDFS"].corte and costo_mm(_sin_salida) == math.inf
    return ok, "{0} algoritmos fracasan; IDDFS termina con fracaso, MM da inf".format(len(rs))


@prueba("Caso limite: multiples entradas obsoletas en UCS")
def _():
    traza = []
    r = ucs(PROBLEMA_PONDERADO, traza)
    obsoletas = [t for t in traza if t["evento"] == "obsoleta"]
    expandidos = [t["extraido"] for t in traza if t["evento"] == "expandido"]
    ok = (len(obsoletas) >= 2 and r.costo == 12
          and len(expandidos) == len(set(expandidos)))
    return ok, "{0} obsoletas descartadas, costo {1}, ningun estado expandido dos veces".format(
        len(obsoletas), r.costo)


@prueba("Caso limite: dos puntos de encuentro en la busqueda bidireccional")
def _():
    traza = []
    r = bidireccional(ProblemaLaberinto(rejilla_abierta(2, 2), (0, 0), (1, 1)), traza)
    encuentros = traza[-1]["encuentros"]
    ok = len(encuentros) == 2 and r.profundidad == 2 and r.camino[1] == encuentros[0][0]
    return ok, "encuentros {0}; elegido {1}".format([e for e, _ in encuentros], r.camino[1])


@prueba("Caso limite: limite exacto, insuficiente y excesivo en DLS (3 versiones)")
def _():
    p = PROBLEMA_3x3
    d = bfs(p).profundidad
    for nombre, dls_ in (("V1", lambda L: busqueda_limitada(p, L)),
                         ("SAI", lambda L: resolver_simpleai(p, "DLS", limite=L)),
                         ("AIMA", lambda L: resolver_aima(p, "DLS", L))):
        corto, exacto, largo = dls_(d - 1), dls_(d), dls_(d + 3)
        if corto.encontrado or not corto.corte or not exacto.encontrado or not largo.encontrado:
            return False, nombre
        if exacto.profundidad != d:
            return False, nombre + " exacto"
    return True, "limites {0}, {1} y {2} en la 3x3: corte, solucion, solucion".format(d - 1, d, d + 3)


@prueba("Caso limite (contra-prueba): SimpleAI DLS pierde la solucion con limite excesivo")
def _():
    r3 = resolver_simpleai(_p_sai, "DLS", limite=3)
    r4 = resolver_simpleai(_p_sai, "DLS", limite=4)
    return r3.encontrado and not r4.encontrado, "defecto D-33 reproducido"


# ---------------------------------------------------------------------------
# 5) Informe
# ---------------------------------------------------------------------------
RESULTADOS_PRUEBAS = []
for _nombre, _funcion in PRUEBAS:
    _aprobada, _detalle = _funcion()
    RESULTADOS_PRUEBAS.append((_nombre, _aprobada, _detalle))
_ancho = max(len(n) for n, _, _ in RESULTADOS_PRUEBAS)
print("PRUEBAS DE ACEPTACION: {0} arboles, {1} con ciclos, {2} ponderados".format(
    len(ARBOLES), len(CICLICOS), len(PONDERADOS)))
for _nombre, _aprobada, _detalle in RESULTADOS_PRUEBAS:
    print("[{0}] {1}  {2}".format(" OK " if _aprobada else "FALL", _nombre.ljust(_ancho), _detalle))
_fallidas = [n for n, a, _ in RESULTADOS_PRUEBAS if not a]
print("{0}/{1} pruebas aprobadas".format(len(RESULTADOS_PRUEBAS) - len(_fallidas),
                                        len(RESULTADOS_PRUEBAS)))
assert not _fallidas, _fallidas

## 9. Segunda familia de problemas: ciclos y costos

Hunt-and-Kill produce un árbol. Allí existe un único camino entre dos estados y no se aprecia plenamente la optimalidad. Diseñe, sin modificar el generador original, dos transformaciones:

### A. Laberinto con ciclos

Abra aleatoriamente entre 5% y 12% de las paredes internas que no sean corredores. Debe conservar:

- adyacencias ortogonales;
- simetría del grafo;
- reproducibilidad;
- al menos un ciclo comprobable.

### B. Laberinto ponderado

Asigne costos positivos a corredores o terrenos. Construya deliberadamente un caso donde:

```text
camino con menos pasos ≠ camino de menor costo
```

No use pesos negativos. Justifique si los costos pertenecen a celdas, acciones o aristas y mantenga esa decisión en las tres versiones.

### 9.1 Diseño de las dos transformaciones

**A. Ciclos.** `abrir_ciclos(instancia, fracción, semilla)` enumera las **paredes internas**: los pares de celdas contiguas, en la misma fila o columna, que no tienen corredor. Abre al azar una fracción de ellas, entre 5 % y 12 %; fuera de ese rango lanza un error. En la instancia hay 456 paredes internas: se abren 23 (5 %) o 46 (10 %). Las propiedades exigidas se comprueban con **las mismas funciones de la auditoría** de la sección 1:

- *Adyacencias ortogonales y dominio.* Solo se abren paredes entre vecinas ortogonales dentro de la rejilla, así que no puede aparecer una diagonal ni una celda fuera de rango.
- *Simetría.* Cada pared se abre en los dos sentidos.
- *Reproducibilidad.* La muestra depende solo de la semilla y se toma sobre una lista **ordenada**: la misma semilla da la misma huella SHA-256 y otra semilla da otra.
- *Ciclo comprobable.* `buscar_ciclo` encuentra una arista de retorno, y la comprobación de aciclicidad, que en el árbol aprobaba, ahora **falla**. Además |E| crece exactamente en el número de paredes abiertas.

El generador y el grafo de `INSTANCIA` no se tocan: se copia, se modifica la copia y se congela (D-38).

**Lo que el árbol ocultaba.** Con 10 % de ciclos, el camino más corto baja de 59 a 47 pasos y la DFS devuelve uno de 115. En un laberinto perfecto eso era imposible, porque solo hay un camino. La igualdad `repetidos = expandidos − 1` de D-19, que era propia de los árboles, deja de cumplirse. Y la IDDFS, que controla ciclos solo sobre el camino actual, pasa a recorrer **caminos simples**, cuyo número crece de forma exponencial con los ciclos: más de 550 000 expansiones frente a 425 de BFS (D-40).

**B. Costos.** Se mantiene la decisión D-23: **el costo pertenece a la arista no dirigida**, y las tres versiones lo leen del mismo `ProblemaLaberinto`. Para darle un sentido físico, el costo se deriva de un **terreno**: una zona de «barro» en forma de rombo alrededor del punto medio del camino más corto. Un corredor que toca el barro cuesta 9; los demás, 1. Es simétrico, positivo y reproducible.

El caso se construye **deliberadamente**. `construir_caso_pasos_vs_costo` busca el menor radio de barro con el que el camino de menor costo tiene **estrictamente más pasos** que el de menos pasos. Exigir solo «caminos distintos» no bastaba: con radio 0, UCS encontraba otro camino con el mismo número de pasos. En la instancia, BFS, la bidireccional y Lee siguen el camino de 47 pasos por el barro, con costo 143. UCS, en las tres versiones, y la MM de AIMA encuentran uno de 51 pasos con costo 51.

`costos_aleatorios(grafo, semilla)`, con costos uniformes entre 1 y 9 por arista, queda disponible para el protocolo experimental.

In [ ]:
# ===========================================================================
# Seccion 9 - Segunda familia: laberintos con ciclos y laberintos ponderados
#
# Ninguna transformacion toca LaberintoHuntKill ni el grafo de INSTANCIA,
# que esta congelado (D-13): cada una construye un grafo NUEVO a partir de
# una copia, y lo congela igual (D-38).
# ===========================================================================

# ---------------------------------------------------------------------------
# A. Abrir ciclos
# ---------------------------------------------------------------------------
from dataclasses import replace


def paredes_internas(grafo):
    """Pares de celdas CONTIGUAS que no tienen corredor, en forma canonica.

    Solo vecinas ortogonales dentro de la rejilla: abrir una de estas paredes
    no puede crear una arista diagonal ni salir del dominio.
    """
    return sorted({(u, v) for u in grafo for a in ACCIONES
                   for v in [desplazar(u, a)]
                   if v in grafo and u < v and v not in grafo[u]})


def abrir_ciclos(instancia, fraccion, semilla):
    """Nueva Instancia con una `fraccion` de las paredes internas abierta.

    Devuelve (instancia_nueva, paredes_abiertas). La eleccion depende SOLO
    de `semilla`, y sobre una lista ORDENADA de paredes, asi que es
    reproducible entre procesos (D-08).
    """
    if not 0.05 <= fraccion <= 0.12:
        raise ValueError("el enunciado pide abrir entre 5% y 12% de las paredes")
    paredes = paredes_internas(instancia.grafo)
    rng = random.Random(semilla)
    abiertas = sorted(rng.sample(paredes, max(1, round(fraccion * len(paredes)))))
    copia = {c: set(v) for c, v in instancia.grafo.items()}
    for u, v in abiertas:
        copia[u].add(v)
        copia[v].add(u)
    grafo = MappingProxyType({c: frozenset(v) for c, v in copia.items()})
    return replace(instancia, grafo=grafo), abiertas


SEMILLA_CICLOS = INSTANCIA.semilla                            # <-- PARAMETRO
INSTANCIA_C5, ABIERTAS_5 = abrir_ciclos(INSTANCIA, 0.05, SEMILLA_CICLOS)
INSTANCIA_C10, ABIERTAS_10 = abrir_ciclos(INSTANCIA, 0.10, SEMILLA_CICLOS)
_paredes_arbol = paredes_internas(INSTANCIA.grafo)

print("A. LABERINTO CON CICLOS")
print("  paredes internas del arbol: {0}; se abren {1} (5%) y {2} (10%)".format(
    len(_paredes_arbol), len(ABIERTAS_5), len(ABIERTAS_10)))

# Propiedades exigidas, con las MISMAS comprobaciones de la auditoria (s. 1).
for _nombre, _fraccion, _inst, _abiertas in (("5%", 0.05, INSTANCIA_C5, ABIERTAS_5),
                                             ("10%", 0.10, INSTANCIA_C10, ABIERTAS_10)):
    _g = _inst.grafo
    _ok = {
        "ortogonales": comprobar_ortogonalidad(_g).aprobada,
        "simetria": comprobar_simetria(_g).aprobada,
        "dominio": comprobar_dominio(_g, _inst.filas, _inst.columnas).aprobada,
        "conexo": comprobar_conectividad(_g).aprobada,
        "sin lazos": comprobar_sin_lazos(_g).aprobada,
    }
    assert all(_ok.values()), _ok
    # |E| crece exactamente en las paredes abiertas.
    assert len(aristas_canonicas(_g)) == len(aristas_canonicas(INSTANCIA.grafo)) + len(_abiertas)
    # Al menos un ciclo COMPROBABLE: la DFS de la auditoria encuentra una
    # arista de retorno, y la comprobacion 5 (aciclicidad) debe FALLAR.
    _ciclo = buscar_ciclo(_g)
    assert _ciclo is not None and not comprobar_aciclicidad(_g).aprobada
    # Reproducibilidad: misma semilla, mismo grafo; otra semilla, otro.
    assert huella(abrir_ciclos(INSTANCIA, _fraccion, SEMILLA_CICLOS)[0].grafo) == huella(_g)
    assert huella(abrir_ciclos(INSTANCIA, _fraccion, SEMILLA_CICLOS + 1)[0].grafo) != huella(_g)
    print("  {0:>3}: {1} aristas, ortogonal, simetrico, conexo, reproducible; "
          "ciclo detectado por la arista {2}".format(
              _nombre, len(aristas_canonicas(_g)), _ciclo))

# Contra-prueba: pedir fuera del rango del enunciado debe fallar.
try:
    abrir_ciclos(INSTANCIA, 0.30, SEMILLA_CICLOS)
    raise AssertionError("deberia rechazar 30%")
except ValueError:
    pass

# ---------------------------------------------------------------------------
# Que cambia con ciclos: la calidad de la ruta ya no es la misma
# ---------------------------------------------------------------------------
PROBLEMA_C10 = ProblemaLaberinto.desde_instancia(INSTANCIA_C10)
R_C10 = ejecutar_todas(PROBLEMA_C10)
for _r in R_C10.values():
    assert condiciones_de_validez(PROBLEMA_C10, _r) == []
print()
print("Instancia con 10% de ciclos, {0} -> {1}:".format(INSTANCIA.inicio, INSTANCIA.meta))
imprimir_resultados([R_C10[a] for a in ("BFS", "UCS", "IDDFS", "BIDIR", "LEE", "DLS",
                                         "DFS", "SAI-DFS", "AIMA-DFS")])
_d_arbol, _d_ciclos = RESULTADO_BFS.profundidad, R_C10["BFS"].profundidad
assert _d_ciclos <= _d_arbol
assert R_C10["DFS"].profundidad >= _d_ciclos
# La prediccion de D-19 (repetidos = expandidos - 1) valia SOLO en arboles:
# con ciclos debe dejar de cumplirse... si la busqueda llega a tocar un
# ciclo. Una BFS corta puede no tocar ninguno, asi que se comprueba con una
# BFS que recorre todo el grafo: hacia la celda mas lejana del inicio.
_nivel, _vistos = {INSTANCIA.inicio}, {INSTANCIA.inicio}
while _nivel:
    _ultimo = _nivel
    _nivel = {v for u in _nivel for v in INSTANCIA_C10.grafo[u]} - _vistos
    _vistos |= _nivel
_r_total = bfs(ProblemaLaberinto(INSTANCIA_C10.grafo, INSTANCIA.inicio, min(_ultimo)))
assert _r_total.repetidos_descartados != _r_total.expandidos - 1
print("El camino mas corto baja de {0} a {1} pasos. DFS devuelve {2}: en el "
      "arbol era imposible que DFS diera un camino peor que BFS.".format(
          _d_arbol, _d_ciclos, R_C10["DFS"].profundidad))
print("repetidos = expandidos - 1 ya no se cumple en una BFS que recorre todo el "
      "grafo ({0} frente a {1}): hay estados a los que se llega por dos "
      "caminos.".format(_r_total.repetidos_descartados, _r_total.expandidos))

_marcas = {}
for _c in R_C10["DFS"].camino:
    _marcas[_c] = "o"
for _c in R_C10["BFS"].camino:
    _marcas[_c] = "#" if _c in _marcas else "*"
_marcas[INSTANCIA.inicio], _marcas[INSTANCIA.meta] = "I", "M"
print("* = solo BFS, o = solo DFS, # = ambos")
print(dibujar_laberinto(INSTANCIA_C10.grafo, INSTANCIA.filas, INSTANCIA.columnas,
                        _marcas, ancho=1))

# ---------------------------------------------------------------------------
# B. Laberinto ponderado
# ---------------------------------------------------------------------------
# Decision D-23, mantenida: el costo pertenece a la ARISTA no dirigida. Aqui
# se deriva de un TERRENO: una zona de "barro" alrededor del punto medio
# del camino mas corto. Un corredor que toca el barro cuesta BARRO; los
# demas, 1. El costo sigue siendo de la arista (simetrico), pero tiene una
# interpretacion fisica.
COSTO_BARRO = 9                                               # <-- PARAMETRO


def costos_con_barro(grafo, centro, radio, barro=COSTO_BARRO):
    zona = {c for c in grafo if abs(c[0] - centro[0]) + abs(c[1] - centro[1]) <= radio}
    costos = {frozenset((u, v)): (barro if (u in zona or v in zona) else 1)
              for u, v in aristas_canonicas(grafo)}
    return costos, zona


def costos_aleatorios(grafo, semilla, minimo=1, maximo=9):
    """Para el protocolo experimental: costo entero uniforme por arista."""
    rng = random.Random(semilla)
    return {frozenset(a): rng.randint(minimo, maximo) for a in sorted(aristas_canonicas(grafo))}


def construir_caso_pasos_vs_costo(instancia):
    """Busca el MENOR radio de barro con el que el camino de menor costo
    tiene ESTRICTAMENTE MAS PASOS que el de menos pasos. Es deliberado: se
    pone barro sobre el camino mas corto. Exigir solo "caminos distintos"
    no basta: con radio 0 UCS encuentra otro camino de igual longitud (D-39).

    El barro nunca cubre el inicio ni la meta. Si con esos extremos no hay
    desvio posible (p. ej. extremos muy cercanos, cambiados en la defensa),
    se construye el caso entre las esquinas, y se informa.
    """
    candidatos = [(instancia.inicio, instancia.meta),
                  ((0, 0), (instancia.filas - 1, instancia.columnas - 1))]
    for inicio, meta in candidatos:
        base = ProblemaLaberinto(instancia.grafo, inicio, meta)
        corto = bfs(base).camino
        centro = corto[len(corto) // 2]
        for radio in range(0, max(instancia.filas, instancia.columnas)):
            costos, zona = costos_con_barro(instancia.grafo, centro, radio)
            zona -= {inicio, meta}
            costos = {a: (COSTO_BARRO if (set(a) & zona) else 1) for a in costos}
            p = ProblemaLaberinto(instancia.grafo, inicio, meta, costo_por_arista(costos))
            r_bfs, r_ucs = bfs(p), ucs(p)
            if r_ucs.profundidad > r_bfs.profundidad and r_ucs.costo < r_bfs.costo:
                return p, costos, zona, radio
    raise AssertionError("no se logro separar pasos de costo en esta instancia; "
                         "pruebe otra fraccion de ciclos o SEMILLA_CICLOS")


PROBLEMA_PONDERADO_C10, COSTOS_C10, ZONA_BARRO, RADIO_BARRO = \
    construir_caso_pasos_vs_costo(INSTANCIA_C10)
assert all(c > 0 for c in COSTOS_C10.values())                # sin pesos negativos
R_P = ejecutar_todas(PROBLEMA_PONDERADO_C10)
for _r in R_P.values():
    assert condiciones_de_validez(PROBLEMA_PONDERADO_C10, _r) == []

print()
print("B. LABERINTO PONDERADO (10% ciclos + barro de costo {0}, radio {1}, "
      "{2} celdas), de {3} a {4}".format(COSTO_BARRO, RADIO_BARRO, len(ZONA_BARRO),
                                         PROBLEMA_PONDERADO_C10.estado_inicial,
                                         PROBLEMA_PONDERADO_C10.meta))
imprimir_resultados([R_P[a] for a in ("BFS", "BIDIR", "LEE", "UCS", "SAI-UCS",
                                       "AIMA-UCS", "IDDFS", "DFS")])
_bfs_p, _ucs_p = R_P["BFS"], R_P["UCS"]
assert _ucs_p.profundidad > _bfs_p.profundidad and _ucs_p.costo < _bfs_p.costo
# Decision D-23 mantenida en las tres versiones: mismo costo optimo.
assert {R_P["UCS"].costo, R_P["SAI-UCS"].costo, R_P["AIMA-UCS"].costo,
        costo_mm(PROBLEMA_PONDERADO_C10)} == {_ucs_p.costo}
print("Camino con menos pasos  (BFS): {0} pasos, costo {1}".format(
    _bfs_p.profundidad, _bfs_p.costo))
print("Camino de menor costo   (UCS): {0} pasos, costo {1}   <- distintos".format(
    _ucs_p.profundidad, _ucs_p.costo))
print("UCS de SimpleAI, UCS de AIMA y la bidireccional MM de AIMA: costo {0}.".format(
    _ucs_p.costo))

_marcas = {c: "~" for c in ZONA_BARRO}
for _c in _bfs_p.camino:
    _marcas[_c] = "*"
for _c in _ucs_p.camino:
    _marcas[_c] = "#" if _marcas.get(_c) == "*" else "o"
_marcas[PROBLEMA_PONDERADO_C10.estado_inicial] = "I"
_marcas[PROBLEMA_PONDERADO_C10.meta] = "M"
print("~ = barro, * = solo BFS (menos pasos), o = solo UCS (menor costo), # = ambos")
print(dibujar_laberinto(INSTANCIA_C10.grafo, INSTANCIA.filas, INSTANCIA.columnas,
                        _marcas, ancho=1))

## 10. Diseño experimental

Ejecute como mínimo **30 instancias** por configuración y reporte mediana y rango intercuartílico, no solamente un caso.

| Factor | Niveles mínimos |
|---|---|
| Tamaño | 15×15, 25×25, 40×40 |
| Topología | árbol, 5% ciclos, 10% ciclos |
| Costos | unitarios, ponderados |
| Posición | esquinas, interior, cercanas |
| Algoritmos | todos los exigidos que sean aplicables |

Debe controlar semillas y separar tiempo de generación del tiempo de búsqueda.

### Gráficas obligatorias

1. expandidos frente a número de estados;
2. frontera máxima frente a profundidad de solución;
3. tiempo frente a tamaño;
4. costo y longitud de las soluciones;
5. mapa de calor de Lee;
6. comparación cruzada de las tres versiones.

### Discusión obligatoria

- ¿Qué métrica resulta más estable que el tiempo?
- ¿Cuándo la búsqueda bidireccional pierde su ventaja?
- ¿Por qué IDDFS repite trabajo y aun así puede ser conveniente?
- ¿Qué efecto tiene el orden de sucesores sobre DFS?
- ¿Qué parte de las diferencias se debe al algoritmo y cuál a la biblioteca?

### 10.1 Diseño del protocolo

| Factor | Niveles | Cómo se controla |
|---|---|---|
| Tamaño | 15×15, 25×25, 40×40 | 30 laberintos por tamaño, semillas `SEMILLA_BASE + i`, i = 0…29 |
| Topología | árbol, 5 % ciclos, 10 % ciclos | `abrir_ciclos` sobre **el mismo** laberinto base, con su misma semilla |
| Costos | unitarios; ponderados de 1 a 9 por arista | `costos_aleatorios(grafo, semilla)`: aristas no dirigidas (D-23) |
| Posición | esquinas; interior (cuadrantes opuestos, a ≥ n/5 del borde); cercanas (a 2–4 celdas en el plano) | generador pseudoaleatorio propio por (semilla, tamaño, posición) |
| Algoritmos | los siete de la Versión 1; BFS, DFS y UCS de SimpleAI y de AIMA | ver exclusiones |

Son 3 × 3 × 3 × 2 = **54 configuraciones con 30 réplicas cada una**, más de 10 000 búsquedas. El diseño es **pareado**: dentro de un tamaño y una réplica, todas las topologías, posiciones, costos y algoritmos trabajan sobre el mismo laberinto base. Las diferencias se deben al factor que cambia y no a que tocó otro laberinto.

**Tiempos separados.** El tiempo de **generación** (Hunt-and-Kill más la apertura de ciclos) se mide aparte. El tiempo de **búsqueda** es el `tiempo_ms` del contrato: no incluye construir el problema.

**Exclusiones, documentadas y no ocultas:**
- DLS e IDDFS con ciclos se ejecutan solo en 15×15, porque recorren caminos simples y su costo crece de forma exponencial (D-40).
- IDDFS lleva un tope de 250 000 expansiones. Una medición que lo supera queda **censurada**: se cuenta y se trata como «> tope» en la mediana, sin descartarla (D-42).
- Las tres versiones se comparan en la posición interior, que basta para la comparación cruzada y mantiene acotado el tiempo del cuaderno.

**Salidas.** Los datos crudos van en `resultados/experimentos.csv` y las seis figuras en `resultados/g*.png`; todo se regenera al ejecutar el cuaderno. Las tablas informan **mediana y rango intercuartílico [Q1–Q3]**, no promedios, porque las distribuciones son asimétricas (la de IDDFS abarca varios órdenes de magnitud).

In [14]:
# ===========================================================================
# Seccion 10 - Protocolo experimental
#
# Diseno (D-41): factorial completo de tamano x topologia x posicion x
# costos, con 30 replicas por configuracion. Las replicas son laberintos
# DISTINTOS (semillas SEMILLA_BASE + i), y la misma semilla se reutiliza en
# todas las configuraciones: las comparaciones entre algoritmos o topologias
# son pareadas, sobre los mismos laberintos. El tiempo de generacion se mide
# aparte del de busqueda.
# ===========================================================================

import csv

N_REPLICAS = 30
SEMILLA_BASE = INSTANCIA.semilla
TAMANOS = (15, 25, 40)
TOPOLOGIAS = (("arbol", None), ("ciclos 5%", 0.05), ("ciclos 10%", 0.10))
POSICIONES = ("esquinas", "interior", "cercanas")
TOPE_IDDFS = 250_000          # expansiones acumuladas; mas alla, se censura


def extremos(n, tipo, rng):
    """Inicio y meta segun el factor posicion, en una rejilla n x n.

    esquinas : (0, 0) y (n-1, n-1).
    interior : cuadrantes opuestos (NO y SE) pero a >= n/5 del borde.
    cercanas : a distancia de Manhattan 2..4 en la rejilla. Que esten cerca
               en el plano no implica que esten cerca en el laberinto.
    """
    if tipo == "esquinas":
        return (0, 0), (n - 1, n - 1)
    if tipo == "interior":
        m = max(1, n // 5)
        a = (rng.randint(m, n // 2 - 1), rng.randint(m, n // 2 - 1))
        b = (rng.randint(n // 2, n - 1 - m), rng.randint(n // 2, n - 1 - m))
        return a, b
    while True:
        a = (rng.randrange(n), rng.randrange(n))
        df, dc = rng.randint(-4, 4), rng.randint(-4, 4)
        b = (a[0] + df, a[1] + dc)
        if 2 <= abs(df) + abs(dc) <= 4 and 0 <= b[0] < n and 0 <= b[1] < n:
            return a, b


def iddfs_acotada(problema, tope=TOPE_IDDFS):
    """IDDFS de la Version 1 con un tope de expansiones acumuladas.

    Mismo bucle que profundizacion_iterativa sobre la misma busqueda_limitada;
    solo anade el tope. Devuelve None si se supera: la medicion queda
    CENSURADA y se informa como tal, no se descarta en silencio (D-42).
    """
    inicio = time.perf_counter()
    exp = gen = rep = fmax = 0
    for limite in itertools.count():
        r = busqueda_limitada(problema, limite)
        exp, gen, rep = exp + r.expandidos, gen + r.generados, rep + r.repetidos_descartados
        fmax = max(fmax, r.frontera_maxima)
        if exp > tope:
            return None
        if r.encontrado or not r.corte:
            return ResultadoBusqueda(r.encontrado, r.camino, r.acciones, r.costo,
                                     r.profundidad, exp, gen, rep, fmax,
                                     (time.perf_counter() - inicio) * 1000.0, "IDDFS")


def _fila(r, **config):
    return dict(config, algoritmo=r.algoritmo.replace("SAI-", "").replace("AIMA-", "")
                .split("(")[0], encontrado=r.encontrado, profundidad=r.profundidad,
                costo=r.costo, expandidos=r.expandidos, generados=r.generados,
                repetidos=r.repetidos_descartados, frontera=r.frontera_maxima,
                tiempo_ms=r.tiempo_ms)


FILAS, GENERACION, CENSURADAS = [], [], []
_t_total = time.perf_counter()
for n in TAMANOS:
    for i in range(N_REPLICAS):
        semilla = SEMILLA_BASE + i
        t0 = time.perf_counter()
        base = construir_instancia(semilla, n, n, (0, 0), (n - 1, n - 1))
        t_gen = (time.perf_counter() - t0) * 1000.0
        for nombre_top, fraccion in TOPOLOGIAS:
            t0 = time.perf_counter()
            inst = base if fraccion is None else abrir_ciclos(base, fraccion, semilla)[0]
            GENERACION.append({"tamano": n, "topologia": nombre_top, "replica": i,
                               "generacion_ms": t_gen + (time.perf_counter() - t0) * 1000.0})
            costos = costos_aleatorios(inst.grafo, semilla)
            for k, posicion in enumerate(POSICIONES):
                rng = random.Random(semilla * 1000 + n * 10 + k)
                ini, fin = extremos(n, posicion, rng)
                cfg = dict(tamano=n, estados=n * n, topologia=nombre_top,
                           posicion=posicion, replica=i)
                # --- costo unitario, Version 1 ---------------------------
                p = ProblemaLaberinto(inst.grafo, ini, fin)
                for buscar in (bfs, dfs, ucs, bidireccional, lee):
                    FILAS.append(_fila(buscar(p), version="V1", costos="unitario", **cfg))
                con_dls = fraccion is None or n == TAMANOS[0]
                if con_dls:            # D-40: DLS/IDDFS con ciclos solo en 15x15
                    d = FILAS[-5]["profundidad"]
                    FILAS.append(_fila(busqueda_limitada(p, d), version="V1",
                                       costos="unitario", **cfg))
                    r_id = iddfs_acotada(p)
                    if r_id is None:
                        CENSURADAS.append(cfg)
                    else:
                        FILAS.append(_fila(r_id, version="V1", costos="unitario", **cfg))
                # --- tres versiones: posicion interior -------------------
                if posicion == "interior":
                    for alg in ("BFS", "DFS", "UCS"):
                        FILAS.append(_fila(resolver_simpleai(p, alg), version="SimpleAI",
                                           costos="unitario", **cfg))
                        FILAS.append(_fila(resolver_aima(p, alg), version="AIMA",
                                           costos="unitario", **cfg))
                # --- costos ponderados (1..9 por arista), Version 1 ------
                pp = ProblemaLaberinto(inst.grafo, ini, fin, costo_por_arista(costos))
                for buscar in (bfs, dfs, ucs, bidireccional, lee):
                    FILAS.append(_fila(buscar(pp), version="V1", costos="ponderado", **cfg))
_t_total = time.perf_counter() - _t_total

# Todas las soluciones son validas (el laberinto es conexo: siempre hay).
assert all(f["encontrado"] for f in FILAS)

os.makedirs("resultados", exist_ok=True)
with open("resultados/experimentos.csv", "w", newline="", encoding="utf-8") as _f:
    _w = csv.DictWriter(_f, fieldnames=list(FILAS[0]))
    _w.writeheader()
    _w.writerows(FILAS)
print("{0} ejecuciones de busqueda sobre {1} laberintos x {2} topologias x {3} "
      "posiciones x 2 costos, en {4:.0f} s. Datos en resultados/experimentos.csv".format(
          len(FILAS), len(TAMANOS) * N_REPLICAS, len(TOPOLOGIAS), len(POSICIONES), _t_total))
print("IDDFS censurada (supero {0:,} expansiones) en {1} de {2} ejecuciones.".format(
    TOPE_IDDFS, len(CENSURADAS),
    len(CENSURADAS) + sum(f["algoritmo"] == "IDDFS" for f in FILAS)))

In [ ]:
# ===========================================================================
# Seccion 10 - Resumen del protocolo: mediana y rango intercuartilico
# ===========================================================================

def seleccionar(**filtro):
    return [f for f in FILAS if all(f[k] == v for k, v in filtro.items())]


def mediana_ric(valores):
    # Con valores censurados (inf), interpolar entre dos inf da nan: es inf.
    q1, q2, q3 = np.nan_to_num(np.percentile(valores, [25, 50, 75]), nan=math.inf)
    return q2, q1, q3


def formato(valores, decimales=0):
    """Mediana [Q1-Q3]. Un valor infinito es una medicion censurada (> tope):
    la mediana y los cuartiles siguen siendo validos mientras no caigan en
    la zona censurada, y si caen se escriben como '>tope' (D-42)."""
    q2, q1, q3 = mediana_ric(valores)
    txt = lambda v: ">{0:.0f}k".format(TOPE_IDDFS / 1000) if math.isinf(v) else         "{0:.{d}f}".format(v, d=decimales)
    return "{0} [{1}-{2}]".format(txt(q2), txt(q1), txt(q3))


def expandidos_con_censura(**cfg):
    valores = [f["expandidos"] for f in seleccionar(version="V1", costos="unitario", **cfg)]
    if cfg.get("algoritmo") == "IDDFS":
        valores += [math.inf for c in CENSURADAS
                    if all(c[k] == v for k, v in cfg.items() if k in c)]
    return valores


ALGORITMOS_V1 = ("BFS", "DFS", "UCS", "BIDIR", "LEE", "DLS", "IDDFS")
print()
print("EXPANDIDOS, costo unitario, Version 1: mediana [Q1-Q3] sobre 30 laberintos")
print("(DLS e IDDFS con ciclos solo en 15x15, D-40; '>250k' = censurado, D-42)")
for n in TAMANOS:
    print("  {0}x{0}".format(n))
    print("    {0:<22}".format("topologia / posicion") +
          "".join("{0:>22}".format(a) for a in ALGORITMOS_V1))
    for nombre_top, _ in TOPOLOGIAS:
        for posicion in POSICIONES:
            celdas = []
            for a in ALGORITMOS_V1:
                v = expandidos_con_censura(tamano=n, topologia=nombre_top,
                                           posicion=posicion, algoritmo=a)
                celdas.append(formato(v) if len(v) == N_REPLICAS else "no se ejecuta")
            print("    {0:<22}".format(nombre_top + " / " + posicion) +
                  "".join("{0:>22}".format(c) for c in celdas))

print()
print("TIEMPO: generacion del laberinto frente a busqueda (ms, mediana [Q1-Q3])")
for n in TAMANOS:
    gen = [g["generacion_ms"] for g in GENERACION if g["tamano"] == n]
    bus = [f["tiempo_ms"] for f in seleccionar(tamano=n, algoritmo="BFS", version="V1",
                                               costos="unitario")]
    print("  {0}x{0}: generacion {1:>22}   BFS {2:>20}".format(
        n, formato(gen, 1), formato(bus, 2)))

print()
print("COSTOS PONDERADOS (1..9 por arista): exceso de costo de cada algoritmo sobre UCS")
for nombre_top, _ in TOPOLOGIAS:
    linea = []
    for a in ("BFS", "BIDIR", "LEE", "DFS"):
        exceso = [100.0 * (f["costo"] / u["costo"] - 1.0)
                  for f, u in zip(seleccionar(topologia=nombre_top, algoritmo=a, costos="ponderado"),
                                  seleccionar(topologia=nombre_top, algoritmo="UCS", costos="ponderado"))]
        linea.append("{0} {1}%".format(a, formato(exceso, 1)))
    print("  {0:<11} ".format(nombre_top) + "   ".join(linea))
for nombre_top, _ in TOPOLOGIAS:
    _pares = list(zip(seleccionar(topologia=nombre_top, algoritmo="BFS", costos="ponderado"),
                      seleccionar(topologia=nombre_top, algoritmo="UCS", costos="ponderado")))
    print("  {0:<11} BFS paga mas que UCS en {1} de {2} busquedas".format(
        nombre_top, sum(f["costo"] > u["costo"] for f, u in _pares), len(_pares)))

# ---------------------------------------------------------------------------
# Estabilidad: la misma busqueda repetida 30 veces
# ---------------------------------------------------------------------------
_p_estab = ProblemaLaberinto(INSTANCIA_C10.grafo, INSTANCIA.inicio, INSTANCIA.meta)
_reps = [bfs(_p_estab) for _ in range(30)]
_tiempos = np.array([r.tiempo_ms for r in _reps])
assert len({r.expandidos for r in _reps}) == 1
CV_TIEMPO = float(np.std(_tiempos) / np.mean(_tiempos))
print()
print("Estabilidad (misma BFS repetida 30 veces): expandidos siempre {0}; tiempo "
      "entre {1:.3f} y {2:.3f} ms, coeficiente de variacion {3:.0%}.".format(
          _reps[0].expandidos, _tiempos.min(), _tiempos.max(), CV_TIEMPO))

# ---------------------------------------------------------------------------
# Tres versiones, 40x40, posicion interior (90 busquedas por celda)
# ---------------------------------------------------------------------------
print()
print("TRES VERSIONES, 40x40, posicion interior: mediana [Q1-Q3]")
print("  {0:<5} {1:<9} {2:>24} {3:>22}".format("alg.", "version", "tiempo_ms", "expandidos"))
for a in ("BFS", "DFS", "UCS"):
    for version in ("V1", "SimpleAI", "AIMA"):
        filas = seleccionar(tamano=40, algoritmo=a, version=version, posicion="interior",
                            costos="unitario")
        print("  {0:<5} {1:<9} {2:>24} {3:>22}".format(
            a, version, formato([f["tiempo_ms"] for f in filas], 2),
            formato([f["expandidos"] for f in filas])))

In [ ]:
# ===========================================================================
# Seccion 10 - Las seis graficas obligatorias
#
# Convenciones (D-43): cada ALGORITMO conserva su color en todas las
# graficas; las lineas son medianas y la banda sombreada es el rango
# intercuartilico; se usa escala logaritmica cuando los valores abarcan
# varios ordenes de magnitud. Las figuras se guardan en resultados/ y se
# regeneran al ejecutar el cuaderno. La evidencia son FILAS y el CSV.
# ===========================================================================

COLOR_ALG = {"BFS": "#2a78d6", "DFS": "#eb6834", "BIDIR": "#1baf7a",
             "IDDFS": "#eda100", "LEE": "#e87ba4", "UCS": "#008300", "DLS": "#4a3aa7"}
COLOR_VERSION = {"V1": "#2a78d6", "SimpleAI": "#eb6834", "AIMA": "#1baf7a"}
_GRIS, _REJILLA = "#8a8984", "#e6e5e0"


def _estilo(eje, titulo=None, x=None, y=None):
    for lado in ("top", "right"):
        eje.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        eje.spines[lado].set_color(_GRIS)
    eje.tick_params(colors=_TINTA_2, labelsize=8)
    eje.grid(axis="y", color=_REJILLA, linewidth=0.8)
    eje.set_axisbelow(True)
    if titulo:
        eje.set_title(titulo, loc="left", fontsize=10, color=_TINTA)
    if x:
        eje.set_xlabel(x, fontsize=9, color=_TINTA_2)
    if y:
        eje.set_ylabel(y, fontsize=9, color=_TINTA_2)


def _linea_mediana(eje, xs, grupos, color, etiqueta, marcador="o"):
    """Mediana con banda Q1-Q3. `grupos` = lista de listas de valores."""
    pares = [(x, np.percentile(v, [25, 50, 75])) for x, v in zip(xs, grupos) if len(v)]
    if not pares:
        return
    x = [p[0] for p in pares]
    q1, q2, q3 = zip(*[p[1] for p in pares])
    eje.fill_between(x, q1, q3, color=color, alpha=0.15, linewidth=0)
    eje.plot(x, q2, color=color, linewidth=2, marker=marcador,
             markersize=5, label=etiqueta)
    # Etiqueta directa al final de la linea, salvo que choque con otra ya
    # puesta (lineas que coinciden, p. ej. BFS, UCS y LEE): la leyenda basta.
    ocupadas = eje.__dict__.setdefault("_etiquetas_y", [])
    if all(abs(math.log10(q2[-1] / y)) > 0.06 for y in ocupadas if y > 0) and q2[-1] > 0:
        ocupadas.append(q2[-1])
        eje.annotate(etiqueta, (x[-1], q2[-1]), xytext=(4, 0), textcoords="offset points",
                     va="center", fontsize=8, color=_TINTA_2)


def _guardar(fig, nombre):
    fig.savefig("resultados/" + nombre, dpi=110, facecolor=_SUPERFICIE,
                bbox_inches="tight")
    plt.show()


ESTADOS = [n * n for n in TAMANOS]

# --- 1. Expandidos frente a numero de estados -------------------------------
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True, facecolor=_SUPERFICIE)
for eje, (top, _) in zip(ejes, TOPOLOGIAS):
    for a in ("BFS", "DFS", "BIDIR", "LEE", "IDDFS"):
        grupos = [expandidos_con_censura(tamano=n, topologia=top, algoritmo=a,
                                         posicion="interior") for n in TAMANOS]
        _linea_mediana(eje, ESTADOS, grupos, COLOR_ALG[a], a)
    eje.set_yscale("log")
    eje.set_xticks(ESTADOS)
    eje.set_xticklabels(["{0}\n({1}x{1})".format(e, n) for e, n in zip(ESTADOS, TAMANOS)])
    _estilo(eje, top, "numero de estados |V|", "nodos expandidos (log)" if eje is ejes[0] else None)
ejes[0].legend(fontsize=8, frameon=False, loc="upper left")
fig.suptitle("1. Expandidos frente a numero de estados (costo unitario, posicion interior; "
             "mediana y Q1-Q3 de 30 laberintos). UCS y LEE coinciden con BFS.", x=0.01,
             ha="left", fontsize=11, color=_TINTA)
_guardar(fig, "g1_expandidos_vs_estados.png")

# --- 2. Frontera maxima frente a profundidad de la solucion -----------------
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True, facecolor=_SUPERFICIE)
for eje, (top, _) in zip(ejes, TOPOLOGIAS):
    for a in ("BFS", "DFS", "BIDIR"):
        filas = seleccionar(topologia=top, algoritmo=a, version="V1", costos="unitario")
        eje.scatter([f["profundidad"] for f in filas], [f["frontera"] for f in filas],
                    s=12, color=COLOR_ALG[a], alpha=0.45, linewidths=0, label=a)
    _estilo(eje, top, "profundidad de la solucion (pasos)",
            "frontera maxima (nodos)" if eje is ejes[0] else None)
ejes[0].legend(fontsize=8, frameon=False, markerscale=1.6)
fig.suptitle("2. Frontera maxima frente a profundidad (costo unitario, todas las "
             "instancias). IDDFS no se dibuja: su frontera es exactamente la profundidad.",
             x=0.01, ha="left", fontsize=11, color=_TINTA)
_guardar(fig, "g2_frontera_vs_profundidad.png")

# --- 3. Tiempo frente a tamano ------------------------------------------------
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True, facecolor=_SUPERFICIE)
for eje, (top, _) in zip(ejes, TOPOLOGIAS):
    for a in ("BFS", "DFS", "UCS", "BIDIR", "LEE", "IDDFS"):
        grupos = [[f["tiempo_ms"] for f in seleccionar(tamano=n, topologia=top, algoritmo=a,
                                                       version="V1", costos="unitario",
                                                       posicion="interior")]
                  for n in TAMANOS]
        _linea_mediana(eje, ESTADOS, grupos, COLOR_ALG[a], a)
    gen = [[g["generacion_ms"] for g in GENERACION if g["tamano"] == n and g["topologia"] == top]
           for n in TAMANOS]
    _linea_mediana(eje, ESTADOS, gen, _GRIS, "generacion", marcador="s")
    eje.set_yscale("log")
    eje.set_xticks(ESTADOS)
    eje.set_xticklabels(["{0}x{0}".format(n) for n in TAMANOS])
    _estilo(eje, top, "tamano", "tiempo (ms, log)" if eje is ejes[0] else None)
ejes[0].legend(fontsize=8, frameon=False, loc="upper left", ncol=2)
fig.suptitle("3. Tiempo de busqueda frente a tamano (posicion interior, 30 laberintos); "
             "en gris, el tiempo de GENERAR el laberinto, medido aparte", x=0.01, ha="left",
             fontsize=11, color=_TINTA)
_guardar(fig, "g3_tiempo_vs_tamano.png")

# --- 4. Costo y longitud de las soluciones -----------------------------------
fig, (e1, e2) = plt.subplots(1, 2, figsize=(15, 4.6), facecolor=_SUPERFICIE)
posiciones_x = np.arange(len(TOPOLOGIAS))
for desplaz, a in ((-0.2, "BFS"), (0.2, "DFS")):
    datos = []
    for top, _ in TOPOLOGIAS:
        datos.append([100.0 * (f["costo"] / u["costo"] - 1.0)
                      for f, u in zip(seleccionar(topologia=top, algoritmo=a, costos="ponderado"),
                                      seleccionar(topologia=top, algoritmo="UCS", costos="ponderado"))])
    caja = e1.boxplot(datos, positions=posiciones_x + desplaz, widths=0.32, patch_artist=True,
                      showfliers=False, medianprops=dict(color=_TINTA))
    for parte in caja["boxes"]:
        parte.set(facecolor=COLOR_ALG[a], alpha=0.55, edgecolor=COLOR_ALG[a])
    e1.plot([], [], color=COLOR_ALG[a], linewidth=6, alpha=0.55, label=a)
e1.set_xticks(posiciones_x)
e1.set_xticklabels([t for t, _ in TOPOLOGIAS])
e1.axhline(0, color=_GRIS, linewidth=1)
_estilo(e1, "a) Costos 1..9: exceso de costo sobre el optimo (UCS)", None,
        "costo sobre UCS (%)")
e1.legend(fontsize=8, frameon=False)
for desplaz, (a, costos, color) in ((-0.2, ("DFS", "unitario", COLOR_ALG["DFS"])),
                                    (0.2, ("UCS", "ponderado", COLOR_ALG["UCS"]))):
    datos = []
    for top, _ in TOPOLOGIAS:
        datos.append([f["profundidad"] / b["profundidad"] if b["profundidad"] else 1.0
                      for f, b in zip(seleccionar(topologia=top, algoritmo=a, costos=costos,
                                                  version="V1"),
                                      seleccionar(topologia=top, algoritmo="BFS", costos=costos,
                                                  version="V1"))])
    caja = e2.boxplot(datos, positions=posiciones_x + desplaz, widths=0.32, patch_artist=True,
                      showfliers=False, medianprops=dict(color=_TINTA))
    for parte in caja["boxes"]:
        parte.set(facecolor=color, alpha=0.55, edgecolor=color)
    e2.plot([], [], color=color, linewidth=6, alpha=0.55,
            label="{0} ({1})".format(a, "costo 1" if costos == "unitario" else "costos 1..9"))
e2.set_xticks(posiciones_x)
e2.set_xticklabels([t for t, _ in TOPOLOGIAS])
e2.axhline(1, color=_GRIS, linewidth=1)
_estilo(e2, "b) Longitud de la solucion relativa a la de BFS (la minima)", None,
        "pasos / pasos de BFS")
e2.legend(fontsize=8, frameon=False)
fig.suptitle("4. Costo y longitud de las soluciones: en un arbol todas coinciden; con "
             "ciclos se separan", x=0.01, ha="left", fontsize=11, color=_TINTA)
_guardar(fig, "g4_costo_y_longitud.png")

# --- 5. Mapa de calor de Lee ---------------------------------------------------
# Laberinto de 40x40 (replica 0), onda completa desde (0, 0): la meta es la
# celda mas lejana, asi la onda etiqueta todo el laberinto.
_inst40 = construir_instancia(SEMILLA_BASE, 40, 40, (0, 0), (39, 39))
_dist = {(0, 0): 0}
_frente = [(0, 0)]
while _frente:                                   # utilidad independiente (D-07)
    _sig = []
    for _u in _frente:
        for _v in _inst40.grafo[_u]:
            if _v not in _dist:
                _dist[_v] = _dist[_u] + 1
                _sig.append(_v)
    _frente = _sig
_lejana = max(sorted(_dist), key=_dist.get)
_reg40 = {}
lee(ProblemaLaberinto(_inst40.grafo, (0, 0), _lejana), _reg40)
assert _reg40["etiqueta"] == _dist               # Lee coincide con la onda de control
_camino40 = lee(ProblemaLaberinto(_inst40.grafo, (0, 0), (39, 39))).camino
fig, eje = plt.subplots(figsize=(8.2, 7.4), facecolor=_SUPERFICIE)
_matriz40 = np.full((40, 40), np.nan)
for (f, c), e in _reg40["etiqueta"].items():
    _matriz40[f, c] = e
_img = eje.imshow(_matriz40, cmap=_MAPA_TIEMPO, extent=(0, 40, 40, 0), interpolation="nearest")
eje.add_collection(LineCollection(_paredes(_inst40.grafo, 40, 40), colors=_TINTA_2,
                                  linewidths=0.6))
eje.plot([c + 0.5 for _, c in _camino40], [f + 0.5 for f, _ in _camino40],
         color=_CAMINO, linewidth=1.8)
for (f, c), letra in (((0, 0), "I"), (_lejana, "L"), ((39, 39), "M")):
    eje.text(c + 0.5, f + 0.5, letra, ha="center", va="center", fontsize=7,
             fontweight="bold", color=_TINTA,
             bbox=dict(boxstyle="circle,pad=0.12", fc=_SUPERFICIE, ec="none"))
eje.set_xticks([])
eje.set_yticks([])
for borde in eje.spines.values():
    borde.set_visible(False)
_barra = fig.colorbar(_img, ax=eje, shrink=0.85, pad=0.02)
_barra.set_label("etiqueta de Lee = distancia al inicio (pasos)", color=_TINTA_2, fontsize=9)
_barra.outline.set_visible(False)
eje.set_title("5. Mapa de calor de Lee, 40x40: onda completa desde I; L = celda mas "
              "lejana ({0} pasos); en naranja, el camino a la esquina M ({1} pasos)".format(
                  _dist[_lejana], len(_camino40) - 1), loc="left", fontsize=10, color=_TINTA)
_guardar(fig, "g5_mapa_calor_lee.png")

# --- 6. Comparacion cruzada de las tres versiones ------------------------------
fig, ejes = plt.subplots(2, 3, figsize=(15, 8), facecolor=_SUPERFICIE)
for col, a in enumerate(("BFS", "DFS", "UCS")):
    for fila, (metrica, etiqueta_y) in enumerate((("tiempo_ms", "tiempo (ms, log)"),
                                                   ("expandidos", "nodos expandidos (log)"))):
        eje = ejes[fila, col]
        for version in ("V1", "SimpleAI", "AIMA"):
            grupos = [[f[metrica] for f in seleccionar(tamano=n, algoritmo=a, version=version,
                                                       posicion="interior", costos="unitario")]
                      for n in TAMANOS]
            _linea_mediana(eje, ESTADOS, grupos, COLOR_VERSION[version], version)
        eje.set_yscale("log")
        eje.set_xticks(ESTADOS)
        eje.set_xticklabels(["{0}x{0}".format(n) for n in TAMANOS])
        _estilo(eje, a if fila == 0 else None, "tamano" if fila == 1 else None,
                etiqueta_y if col == 0 else None)
ejes[0, 0].legend(fontsize=8, frameon=False, loc="upper left")
fig.suptitle("6. Las tres versiones sobre los mismos laberintos (posicion interior, las "
             "tres topologias; mediana y Q1-Q3 de 90 busquedas)", x=0.01, ha="left",
             fontsize=11, color=_TINTA)
_guardar(fig, "g6_tres_versiones.png")

### 10.2 Discusión

Las cifras citadas son medianas de las tablas anteriores. Los recuentos (expandidos, frontera, longitudes, costos) son deterministas con estas semillas. Los tiempos varían de una ejecución a otra y se citan de forma aproximada.

**¿Qué métrica resulta más estable que el tiempo?** Los **nodos expandidos**, y con ellos los generados y la frontera máxima. Son función exclusiva de la instancia, el algoritmo y el orden de sucesores: la misma BFS repetida 30 veces expande siempre los mismos 425 estados, mientras que su tiempo varía con un coeficiente de variación de entre 8 % y 15 % según la carga de la máquina. Además, el tiempo mezcla el algoritmo con la implementación: con **los mismos** 804 expandidos, la BFS de 40×40 tarda unos 4,4 ms en la Versión 1 y unos 6,2 ms en SimpleAI. Por eso las comparaciones de algoritmos se hacen sobre expandidos, y el tiempo se usa para comparar implementaciones.

**¿Cuándo pierde su ventaja la búsqueda bidireccional?**
1. **Cuando la ramificación es baja.** En los árboles de 40×40 con extremos en las esquinas expande 906 estados frente a 1 376 de BFS, apenas un 34 % menos y lejos de la raíz cuadrada que promete O(b^(d/2)). Con 10 % de ciclos la ventaja crece (409 frente a 796 en el interior, un 49 % menos), porque los ciclos aumentan la ramificación efectiva.
2. **Cuando la meta es inalcanzable.** Se detiene al vaciarse una frontera, pero el otro lado ya trabajó en vano: 95 estados frente a 89 de BFS (etapa 6).
3. **Con costos no unitarios.** Siendo por capas, es óptima en pasos y no en costo (gráfica 4a).

En el caso «cercanas» sigue ganando en proporción, pero la ganancia absoluta es de unas decenas de nodos.

**¿Por qué IDDFS repite trabajo y aun así puede ser conveniente?** Cada iteración vuelve a recorrer todos los niveles anteriores. En un árbol de 40×40 con extremos en las esquinas eso cuesta 123 697 expansiones frente a 1 376 de BFS, unas 90 veces más. Con ciclos es peor, porque DLS recorre **caminos simples** y el crecimiento es exponencial (D-40); por eso hay mediciones censuradas en 15×15 con 10 % de ciclos. Su ventaja es la **memoria**: guarda solo el camino actual, O(d), frente a una frontera que en BFS crece como O(b^d). Compensa cuando la ramificación es alta, que es cuando el último nivel domina la suma y el sobrecosto es del orden de b/(b−1), y cuando la frontera de BFS no cabe en memoria. En estos laberintos no se da ninguna de las dos cosas: la frontera de BFS nunca pasa de unas decenas de nodos (gráfica 2).

**¿Qué efecto tiene el orden de sucesores sobre DFS?** Decisivo, y en dos sentidos:
- **Trabajo:** en la instancia individual, DFS con orden N-E-S-O expande 297 estados y con O-S-E-N (SimpleAI y AIMA) expande 228, sobre el mismo laberinto.
- **Calidad:** con 10 % de ciclos, la DFS N-E-S-O devuelve un camino de 115 pasos y la O-S-E-N uno de 55, cuando el óptimo es 47.

En el protocolo, el rango intercuartílico de DFS es el más ancho de todos. En la posición «cercanas», DFS expande muchas veces más que BFS, porque a menudo sale en la dirección equivocada y agota esa rama antes de volver. Con ciclos, su solución es mediana entre 1,6 y 2,3 veces más larga que la de BFS y, con costos, entre 58 % y 135 % más cara que la óptima (gráfica 4).

**¿Qué parte de las diferencias se debe al algoritmo y cuál a la biblioteca?**
- **Del algoritmo:** qué camino se devuelve y su costo. Las tres versiones concuerdan en existencia de solución y costo óptimo en todas las pruebas de la sección 8.
- **De las convenciones de la biblioteca:** los recuentos. Con la misma política coinciden exactamente: la BFS de SimpleAI y la de la Versión 1 (804 y 804 en 40×40), y la UCS de AIMA y la de la Versión 1. Difieren cuando cambia una convención: la BFS de AIMA prueba la meta al generar (793), las DFS de las bibliotecas exploran O-S-E-N, y SimpleAI desempata UCS de otra forma.
- **De la implementación de la biblioteca:** los tiempos. Las dos bibliotecas comprueban si un hijo está en la frontera recorriéndola, O(n) por hijo, y reemplazan en la frontera en lugar de dejar entradas obsoletas. Por eso su UCS tarda entre 2 y 4 veces más que la de la Versión 1 (unos 10 y 16 ms frente a 4,5 ms en 40×40) con los mismos nodos expandidos.

**Generación frente a búsqueda.** En 40×40, generar el laberinto con Hunt-and-Kill cuesta una mediana de unos 160 ms y una BFS unos 4 ms. Si no se separaran, más del 95 % del tiempo medido sería del generador, no del algoritmo de búsqueda (gráfica 3). El coste viene de la fase *hunt*, que recorre todas las celdas no visitadas en cada cacería.

## 11. Análisis teórico

Complete y justifique esta tabla usando $b$ como factor de ramificación, $d$ como profundidad de la solución menos profunda y $m$ como profundidad máxima.

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | Condiciones |
|---|---|---|---|---|---|
| DFS | | | | | |
| BFS | | | | | |
| UCS | | | | | |
| DLS | | | | | |
| IDDFS | | | | | |
| Bidireccional | | | | | |
| Lee | | | | | |

Relacione después las cotas teóricas con las mediciones. Una tabla memorizada sin conexión con los experimentos no recibe puntaje completo.

### 11.1 Tabla justificada

*b*: factor de ramificación; *d*: profundidad de la solución menos profunda; *m*: profundidad máxima; *ℓ*: límite; *C\**: costo óptimo; *ε*: costo de paso mínimo. Entre corchetes, la cota de la **búsqueda en grafo** tal como se implementó, en función de |V| y |E|.

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | Condiciones |
|---|---|---|---|---|---|
| DFS | Sí, en espacios finitos | No | O(b^m) [O(\|V\|+\|E\|)] | O(b·m) en árbol [O(\|V\|) por *alcanzados*] | Completo gracias a la búsqueda en grafo; en árbol, un grafo no dirigido lo hace ciclar |
| BFS | Sí, si *b* es finito | Sí, con costo unitario | O(b^d) [O(\|V\|+\|E\|)] | O(b^d) | Óptimo en **pasos**; con costos variables, no |
| UCS | Sí, si todo costo ≥ ε > 0 | Sí | O(b^(1+⌊C\*/ε⌋)) [O(\|E\| log \|E\|) con entradas obsoletas] | ídem | Prueba de objetivo al extraer; costos positivos |
| DLS | Solo si d ≤ ℓ | No | O(b^ℓ) | O(ℓ) en esta implementación (hijos de a uno); O(b·ℓ) en el libro | Control de ciclos sobre el camino actual; distingue corte de fracaso |
| IDDFS | Sí | Sí, con costo unitario | O(b^d) | O(d) | Sobrecosto ≈ b/(b−1) con *b* holgado; **≈ d/2 si b → 1**; con ciclos recorre caminos simples |
| Bidireccional | Sí | Sí, con costo unitario (por capas) | O(b^(d/2)) | O(b^(d/2)) | Requiere predecesores (grafo simétrico) y meta explícita; óptima en **pasos** |
| Lee | Sí | Sí, en pasos | O(\|V\|+\|E\|) = O(b^d) | O(\|V\|) etiquetas | Solo costo unitario: la etiqueta cuenta frentes, no costo (7.2) |

**Justificaciones breves.**
- *DFS* no es óptima porque devuelve el primer camino que encuentra. Con *alcanzados* cada estado se expande una vez, y de ahí la cota en |V| + |E|.
- *BFS* es óptima con costo unitario porque la cola extrae en orden no decreciente de profundidad, y marcar al generar no pierde nada (5.1).
- *UCS* extrae en orden no decreciente de *g*. Con costos positivos, lo que sale ya no puede mejorar, y por eso la meta se prueba al extraer (6.2).
- *IDDFS* hereda la optimalidad de BFS porque prueba los límites en orden.
- La *bidireccional* es óptima en pasos por el lema de la capa (6.4).
- *Lee* es BFS sincronizada por niveles, demostrado por inducción en 7.1.

In [ ]:
# ===========================================================================
# Seccion 11 - Cotas teoricas frente a mediciones
#
# Se estima el factor de ramificacion EFECTIVO b* de cada busqueda BFS del
# protocolo (definicion de AIMA: el b de un arbol uniforme de profundidad d
# con N+1 nodos) y se contrastan dos predicciones que dependen de b.
# ===========================================================================

def ramificacion_efectiva(expandidos, d):
    """b* tal que 1 + b* + ... + b*^d = expandidos + 1 (biseccion)."""
    if d <= 0:
        return math.nan
    objetivo = expandidos + 1
    bajo, alto = 1.0, 10.0
    for _ in range(80):
        medio = (bajo + alto) / 2
        suma = (medio ** (d + 1) - 1) / (medio - 1)
        bajo, alto = (medio, alto) if suma < objetivo else (bajo, medio)
    return bajo


print("POSICION ESQUINAS, costo unitario (mediana sobre 30 laberintos)")
print("  {0:<11} {1:>5} {2:>6} {3:>8} {4:>12} {5:>12} {6:>14}".format(
    "topologia", "n", "d", "b*", "BIDIR/BFS", "IDDFS/BFS", "(IDDFS/BFS)/d"))
COTAS = []
for nombre_top, _ in TOPOLOGIAS:
    for n in TAMANOS:
        base = dict(tamano=n, topologia=nombre_top, posicion="esquinas",
                    version="V1", costos="unitario")
        bfs_ = {f["replica"]: f for f in seleccionar(algoritmo="BFS", **base)}
        bid_ = {f["replica"]: f for f in seleccionar(algoritmo="BIDIR", **base)}
        idd_ = {f["replica"]: f for f in seleccionar(algoritmo="IDDFS", **base)}
        b_est = np.median([ramificacion_efectiva(f["expandidos"], f["profundidad"])
                           for f in bfs_.values()])
        d_med = np.median([f["profundidad"] for f in bfs_.values()])
        r_bid = np.median([bid_[i]["expandidos"] / bfs_[i]["expandidos"] for i in bfs_])
        r_idd = [idd_[i]["expandidos"] / bfs_[i]["expandidos"] for i in idd_]
        r_idd_d = [idd_[i]["expandidos"] / bfs_[i]["expandidos"] / bfs_[i]["profundidad"]
                   for i in idd_]
        COTAS.append(dict(topologia=nombre_top, n=n, b=b_est, d=d_med, bidir=r_bid,
                          iddfs=np.median(r_idd) if r_idd else math.nan,
                          iddfs_d=np.median(r_idd_d) if r_idd_d else math.nan))
        print("  {0:<11} {1:>5} {2:>6.0f} {3:>8.4f} {4:>12.2f} {5:>12} {6:>14}".format(
            nombre_top, n, d_med, b_est, r_bid,
            "{0:.1f}".format(np.median(r_idd)) if r_idd else "-",
            "{0:.2f}".format(np.median(r_idd_d)) if r_idd_d else "-"))

_arboles = [c for c in COTAS if c["topologia"] == "arbol"]
# Prediccion 1: con b* ~ 1 el numero de estados a distancia <= L crece
# ~ linealmente, BFS expande ~ c*d y IDDFS ~ c*d^2/2: el cociente ~ d/2.
assert all(c["b"] < 1.1 for c in COTAS)
assert all(0.3 < c["iddfs_d"] < 0.6 for c in _arboles)
# Prediccion 2: la ventaja bidireccional crece con la ramificacion. Con 10%
# de ciclos (b* mayor), BIDIR/BFS debe ser menor que en el arbol. Se compara
# sobre las 90 busquedas de cada topologia (los tres tamanos): la mediana de
# un solo tamano, con 30 laberintos, es demasiado ruidosa para un assert.
def _cocientes(topologia):
    b_, r_ = [], []
    for n in TAMANOS:
        base = dict(tamano=n, topologia=topologia, posicion="esquinas",
                    version="V1", costos="unitario")
        for f, g in zip(seleccionar(algoritmo="BFS", **base),
                        seleccionar(algoritmo="BIDIR", **base)):
            b_.append(ramificacion_efectiva(f["expandidos"], f["profundidad"]))
            r_.append(g["expandidos"] / f["expandidos"])
    return np.median(b_), np.median(r_)


(_b_arbol, _r_arbol), (_b_ciclos, _r_ciclos) = _cocientes("arbol"), _cocientes("ciclos 10%")
assert _b_ciclos > _b_arbol and _r_ciclos < _r_arbol
print("Agrupando los tres tamanos: arbol b* = {0:.3f}, BIDIR/BFS = {1:.2f}; "
      "10% ciclos b* = {2:.3f}, BIDIR/BFS = {3:.2f}.".format(
          _b_arbol, _r_arbol, _b_ciclos, _r_ciclos))
print()
print("b* entre {0:.3f} y {1:.3f}: el laberinto es casi un pasillo.".format(
    min(c["b"] for c in COTAS), max(c["b"] for c in COTAS)))
print("En arboles IDDFS/BFS ~ {0:.2f}*d a {1:.2f}*d: la cota b/(b-1) no aplica; "
      "con b -> 1 la prediccion es d/2.".format(
          min(c["iddfs_d"] for c in _arboles), max(c["iddfs_d"] for c in _arboles)))

### 11.2 Relación de las cotas con las mediciones

La celda anterior estima el **factor de ramificación efectivo** *b\** de cada BFS del protocolo. Sale entre **1,01 y 1,07**: los laberintos de Hunt-and-Kill son casi pasillos. Casi todas las diferencias entre lo que dice el libro y lo que se mide se explican por ese número.

1. **O(b^d) con b ≈ 1 no es exponencial.** Si *b\** → 1, el número de estados a distancia ≤ *d* crece de forma aproximadamente **lineal** en *d*. Por eso los expandidos de BFS crecen casi en proporción a |V| en la gráfica 1, y no de forma explosiva. Además *d* crece más que linealmente con el lado *n* (mediana 53, 112 y 221 en árboles con extremos en las esquinas).
2. **IDDFS:** con *b* holgado, el sobrecosto frente a BFS es del orden de *b/(b−1)*, casi constante. Con *b* → 1 la suma de las *d* iteraciones es cuadrática y el cociente se aproxima a **d/2**. Lo medido en árboles es **0,40·d a 0,46·d**, unas 23, 45 y 91 veces BFS en 15×15, 25×25 y 40×40. La celda lo comprueba. Es el mismo fenómeno de la etapa 5, ahora cuantificado.
3. **Bidireccional:** O(b^(d/2)) frente a O(b^d) es una ganancia enorme para *b* grande y **ninguna** para *b* = 1: dos pasillos de *d/2* suman un pasillo de *d*. Lo medido está en el medio: la bidireccional expande el 61 %–77 % de BFS, y la ventaja **crece con los ciclos**, que aumentan *b\**. La celda comprueba esa tendencia en los tres tamaños.
4. **Espacio:** con *b* ≈ 1 la frontera de BFS, O(b^d), es pequeña (nunca pasa de unas decenas de nodos, gráfica 2). Por eso la ventaja de memoria de IDDFS, O(d), no compensa aquí, y su frontera, igual a *d*, suele ser **mayor** que la de BFS.
5. **DFS:** su cota O(b^m) es un peor caso. Lo medido es muy variable, y el rango intercuartílico de DFS es el más ancho de todos, porque su trabajo depende del orden de sucesores (sección 10.2). Su no optimalidad solo se hace visible con ciclos (gráfica 4).
6. **UCS y Lee:** con costo unitario las dos coinciden con BFS, que es lo que predice la teoría. UCS recorre exactamente el orden de BFS (D-22) y Lee es BFS por niveles (7.1). Con costos 1..9, UCS es la única óptima en costo: BFS paga más en 91 y 125 de 270 búsquedas con 5 % y 10 % de ciclos, y en ninguna en los árboles.

### 11.3 Conclusiones

1. **Formular bien precede a buscar bien.** Una sola formulación, `ProblemaLaberinto`, compartida por las tres versiones, hizo que toda diferencia medida fuera atribuible al algoritmo o a la biblioteca, nunca al modelo.
2. **El laberinto perfecto oculta la calidad de la ruta.** Con un único camino simple, los siete algoritmos devuelven la misma solución. La optimalidad solo se observa al abrir ciclos y poner costos: la DFS pasa a caminos 1,6 a 2,3 veces más largos y BFS paga más que UCS en casi la mitad de los casos con 10 % de ciclos.
3. **Las cotas asintóticas dependen de *b*, y aquí b ≈ 1.** Por eso IDDFS cuesta unas *d/2* veces BFS y la bidireccional apenas gana. Las predicciones del libro se cumplen al sustituir el *b* efectivo, no el nominal de 4.
4. **Los nodos expandidos son la métrica comparable; el tiempo mide implementaciones.** Las tres versiones coinciden exactamente en recuentos cuando comparten política, y difieren en tiempo por detalles de la biblioteca.
5. **Las bibliotecas tienen contratos que hay que leer.** SimpleAI no distingue corte de fracaso y su DLS con memoria global pierde soluciones. AIMA prueba la meta al generar en BFS, no controla ciclos en DLS y devuelve solo el costo en la bidireccional. Ninguna de estas diferencias es visible sin instrumentación y sin casos construidos para provocarlas.
6. **Toda prueba lleva su contra-prueba.** Varias de las comprobaciones de este cuaderno habrían pasado siempre si no se hubiera exigido también el caso en que deben fallar (D-11). Tres de los errores registrados en la bitácora (D-09, D-39, D-44) se detectaron así o ejecutando el cuaderno completo en limpio.

## 12. Preguntas para la defensa

El docente seleccionará algunas al azar:

1. Muestre en memoria la diferencia entre frontera y alcanzados.
2. Cambie BFS a DFS modificando solamente la política apropiada.
3. Explique un caso en el que marcar visitados demasiado tarde duplique trabajo.
4. Explique por qué terminar UCS al generar la meta puede ser incorrecto.
5. Reconstruya un camino usando padres sin almacenar caminos completos.
6. Muestre por qué dos BFS bidireccionales requieren orientar correctamente los padres.
7. Explique por qué Lee no es un rayo geométrico.
8. Adapte el problema a una meta múltiple.
9. Introduzca una celda bloqueada durante la ejecución y analice qué debe recalcularse.
10. Compare `SearchProblem` de SimpleAI con `Problem` de AIMA-Python.
11. Diagnostique una discrepancia de métricas entre dos versiones.
12. Argumente por qué un laberinto perfecto oculta diferencias de calidad de ruta.

### 12.1 Respuestas breves

La celda siguiente ejecuta una demostración por pregunta (P1–P9 y P11). Las variantes incorrectas que se muestran existen solo en esa celda.

1. **Frontera frente a alcanzados.** La frontera guarda lo descubierto y todavía no expandido. *Alcanzados* guarda todo lo descubierto alguna vez. La frontera es un subconjunto de alcanzados: un estado sale de la frontera al expandirse, pero sigue en alcanzados, y eso impide volver a generarlo.
2. **BFS → DFS.** Basta cambiar `FronteraFIFO()` por `FronteraLIFO()` en la llamada a `busqueda_en_grafo`; el núcleo no cambia (D-17).
3. **Marcar demasiado tarde.** Si se marca al expandir y no se revisa al extraer, un mismo estado entra varias veces en la frontera y se expande más de una vez. En la rejilla abierta de 4×4 se expanden 49 estados frente a 15, y uno de ellos 10 veces.
4. **UCS al generar la meta.** La primera vez que se genera la meta puede ser por un camino caro. Con la meta `(1,2)` del grafo ponderado, al generar se obtiene costo 12; al extraer, 7. Al extraer, todo lo que queda en la cola cuesta al menos lo mismo, y con costos positivos no puede mejorar.
5. **Reconstrucción con padres.** Cada nodo guarda una referencia a su padre. Se sigue la cadena desde la meta hasta la raíz y se invierte: O(d), una sola vez.
6. **Orientación en la bidireccional.** La mitad de atrás se construyó desde la meta. Hay que invertir el orden de sus estados **y** cada una de sus acciones. Sin la segunda inversión, la condición 5 de validez falla.
7. **Lee no es un rayo.** Es una onda que avanza a la vez por **todos** los corredores. En la instancia, el frente más ancho tiene 11 celdas en ramas distintas, y cada celda recibe su distancia mínima (7.1).
8. **Meta múltiple.** `ProblemaMultimeta` solo redefine `es_meta`. BFS, DFS, UCS, DLS e IDDFS funcionan sin cambios porque solo consultan la prueba de objetivo. La bidireccional y Lee usan la meta como *estado*: la bidireccional necesitaría una frontera de atrás inicializada con todas las metas.
9. **Celda bloqueada durante la ejecución.** Queda inválido todo lo que dependa de un camino por esa celda: los nodos de su subárbol, las entradas de la frontera que cuelgan de ella y, en Lee, las etiquetas posteriores. Lo más simple y correcto es replanificar desde la posición actual. Algoritmos incrementales como D\* Lite recalculan solo la parte afectada.
10. **`SearchProblem` frente a `Problem`.** Los dos piden acciones, resultado y prueba de objetivo. SimpleAI pide el costo **del paso** (`cost`) y ofrece un visor de eventos. AIMA pide el costo **acumulado** (`path_cost(c, …)`), no tiene visor y compara por defecto con `self.goal` (V2.1 y V3.1).
11. **Diagnosticar una discrepancia de métricas.** Primero `condiciones_de_validez`, para saber si algún resultado es inválido. Después, comparar los caminos. Por último, buscar la convención que difiere: momento de la prueba de objetivo, orden de sucesores o desempate. Ejemplo: la BFS de AIMA expande 241 estados frente a 249 porque prueba la meta al generar.
12. **Por qué un laberinto perfecto oculta la calidad de la ruta.** Entre dos celdas hay un único camino simple, así que todo algoritmo completo devuelve la misma solución. Con ciclos, la DFS da caminos 1,6 a 2,3 veces más largos; con costos, BFS paga más que UCS en el 46 % de los casos (secciones 9 y 10).

In [ ]:
# ===========================================================================
# Seccion 12 - Demostraciones ejecutables para las preguntas de la defensa
#
# Cada bloque responde una pregunta con codigo que se puede modificar en
# vivo. Las variantes "incorrectas" se escriben aqui, marcadas como tales,
# para MOSTRAR el fallo; ninguna se usa en el resto del cuaderno.
# ===========================================================================

def _titulo(numero, texto):
    print()
    print("-" * 78)
    print("P{0}. {1}".format(numero, texto))
    print("-" * 78)


# --- P1. Frontera frente a alcanzados, en memoria ----------------------------
_titulo(1, "Frontera frente a alcanzados (BFS en la 3x3, tras 3 extracciones)")
_traza = []
bfs(PROBLEMA_3x3, _traza)
_alcanzados = {PROBLEMA_3x3.estado_inicial}
for _t in _traza[:3]:
    _alcanzados |= set(_t["nuevos"])
print("frontera   (por explorar, en orden de salida):", _traza[2]["frontera"])
print("alcanzados (todo lo visto alguna vez)       :", sorted(_alcanzados))
print("La frontera es un SUBCONJUNTO de alcanzados: lo ya expandido sale de la "
      "frontera pero sigue en alcanzados, y eso impide volver a generarlo.")
assert set(_traza[2]["frontera"]) <= _alcanzados

# --- P2. Cambiar BFS por DFS tocando solo la politica ------------------------
_titulo(2, "BFS -> DFS cambiando SOLO la frontera")
_r_fifo = busqueda_en_grafo(PROBLEMA_C10, FronteraFIFO(), "FIFO")
_r_lifo = busqueda_en_grafo(PROBLEMA_C10, FronteraLIFO(), "LIFO")   # unica diferencia
print("busqueda_en_grafo(p, FronteraFIFO()) -> {0} pasos, {1} expandidos".format(
    _r_fifo.profundidad, _r_fifo.expandidos))
print("busqueda_en_grafo(p, FronteraLIFO()) -> {0} pasos, {1} expandidos".format(
    _r_lifo.profundidad, _r_lifo.expandidos))
assert (_r_fifo.profundidad, _r_lifo.profundidad) == (bfs(PROBLEMA_C10).profundidad,
                                                      dfs(PROBLEMA_C10).profundidad)


# --- P3. Marcar visitados demasiado tarde duplica trabajo --------------------
# VARIANTE INCORRECTA: marca al EXPANDIR y no revisa al extraer.
def _bfs_marca_tarde(problema):
    expandidos, generados = 0, 0
    frontera, expandidos_set = deque([Nodo(problema.estado_inicial)]), set()
    veces = {}
    while frontera:
        nodo = frontera.popleft()
        if problema.es_meta(nodo.estado):
            return expandidos, generados, veces
        expandidos += 1
        veces[nodo.estado] = veces.get(nodo.estado, 0) + 1
        expandidos_set.add(nodo.estado)                    # marca TARDE
        for accion in problema.acciones(nodo.estado):
            hijo = nodo.hijo(problema, accion)
            generados += 1
            if hijo.estado not in expandidos_set:          # no mira la frontera
                frontera.append(hijo)


_titulo(3, "Marcar demasiado tarde duplica trabajo (rejilla abierta 4x4)")
_p4 = ProblemaLaberinto(rejilla_abierta(4, 4), (0, 0), (3, 3))
_exp_t, _gen_t, _veces = _bfs_marca_tarde(_p4)
_r_bien = bfs(_p4)
print("BFS marcando al generar : {0} expandidos, {1} generados".format(
    _r_bien.expandidos, _r_bien.generados))
print("BFS marcando al expandir: {0} expandidos, {1} generados; {2} estados "
      "expandidos mas de una vez (el peor, {3} veces)".format(
          _exp_t, _gen_t, sum(v > 1 for v in _veces.values()), max(_veces.values())))
assert _exp_t > _r_bien.expandidos


# --- P4. Terminar UCS al GENERAR la meta es incorrecto ------------------------
# VARIANTE INCORRECTA: UCS que prueba la meta al generar.
def _ucs_meta_al_generar(problema):
    orden = itertools.count()
    frontera = [(0, next(orden), Nodo(problema.estado_inicial))]
    mejor = {problema.estado_inicial: 0}
    while frontera:
        g, _, nodo = heapq.heappop(frontera)
        if g > mejor[nodo.estado]:
            continue
        for accion in problema.acciones(nodo.estado):
            hijo = nodo.hijo(problema, accion)
            if problema.es_meta(hijo.estado):              # demasiado pronto
                return hijo.g
            if hijo.g < mejor.get(hijo.estado, math.inf):
                mejor[hijo.estado] = hijo.g
                heapq.heappush(frontera, (hijo.g, next(orden), hijo))


_titulo(4, "UCS terminando al GENERAR la meta (grafo ponderado 2x4, meta (1,2))")
_p_f = ProblemaLaberinto(GRAFO_PONDERADO, (0, 0), (1, 2), costo_por_arista(_COSTOS_PONDERADO))
print("UCS al extraer: costo {0}.  UCS al generar: costo {1}.".format(
    ucs(_p_f).costo, _ucs_meta_al_generar(_p_f)))
print("(1,2) se GENERA primero desde (0,2) con g = 12; el camino por abajo, de "
      "costo 7, se descubre despues. Al extraer, g = 7 ya es el menor posible.")
assert _ucs_meta_al_generar(_p_f) > ucs(_p_f).costo

# --- P5. Reconstruir un camino con padres ------------------------------------
_titulo(5, "Reconstruccion por padres (solucion de BFS en la 3x3)")
_nodo = Nodo((0, 0))
for _a in bfs(PROBLEMA_3x3).acciones:          # rearmar la cadena de nodos
    _nodo = _nodo.hijo(PROBLEMA_3x3, _a)
_cadena, _x = [], _nodo
while _x is not None:
    _cadena.append("{0} (padre {1}, accion {2})".format(
        _x.estado, _x.padre.estado if _x.padre else None, _x.accion))
    _x = _x.padre
print("desde la meta hacia atras:")
for _linea in _cadena:
    print("   " + _linea)
print("invertido:", reconstruir(_nodo)[0])

# --- P6. Orientar los padres en la bidireccional -----------------------------
_titulo(6, "Bidireccional: por que hay que invertir la mitad de atras")
_t6 = []
_r6 = bidireccional(PROBLEMA_3x3, _t6)
_camino6, _acc6 = _r6.camino, _r6.acciones
_punto = _t6[-1]["encuentros"][0][0]
_k = _camino6.index(_punto)                       # donde se tocaron
# Olvidar invertir las acciones de atras equivale a invertir las correctas.
_sin_invertir = _acc6[:_k] + tuple(ACCION_INVERSA[a] for a in _acc6[_k:])
print("punto de encuentro:", _punto)
print("acciones correctas     :", "".join(_acc6))
print("sin invertir la mitad  :", "".join(_sin_invertir))
print("validar_solucion con las acciones sin invertir ->",
      condiciones_de_validez(PROBLEMA_3x3, replace(_r6, acciones=_sin_invertir)))

# --- P7. Lee no es un rayo geometrico -----------------------------------------
_titulo(7, "Lee es una onda por TODOS los corredores, no un rayo")
_mas_ancho = max(range(len(FRENTES_LEE)), key=lambda k: len(FRENTES_LEE[k]))
print("En la instancia, el frente {0} tiene {1} celdas a la vez, en corredores "
      "distintos: {2}".format(_mas_ancho, len(FRENTES_LEE[_mas_ancho]),
                              sorted(FRENTES_LEE[_mas_ancho])))
print("Un rayo seguiria UNA direccion y rebotaria; la onda avanza por todas las "
      "ramas a la vez y llega a cada celda por el camino mas corto (7.1).")


# --- P8. Meta multiple ---------------------------------------------------------
class ProblemaMultimeta(ProblemaLaberinto):
    """Varias metas: solo cambia la prueba de objetivo."""

    def __init__(self, grafo, inicio, metas, costo_paso=costo_unitario):
        super().__init__(grafo, inicio, min(metas), costo_paso)
        self.metas = frozenset(metas)

    def es_meta(self, estado):
        return estado in self.metas


_titulo(8, "Adaptacion a una meta multiple")
_metas = {INSTANCIA.meta, (19, 0), (0, 24)}
_pm = ProblemaMultimeta(INSTANCIA.grafo, INSTANCIA.inicio, _metas)
for _buscar in (bfs, dfs, ucs, profundizacion_iterativa):
    _r = _buscar(_pm)
    assert condiciones_de_validez(_pm, _r) == []
    print("  {0:<6} llega a {1} en {2} pasos".format(_r.algoritmo, _r.camino[-1], _r.profundidad))
print("BFS, DFS, UCS, DLS e IDDFS solo consultan es_meta: funcionan sin cambios.")
print("BIDIR y LEE usan la meta como ESTADO (raiz de atras, reconstruccion):")
print("la bidireccional necesitaria una frontera de atras con TODAS las metas.")

# --- P9. Celda bloqueada durante la ejecucion ---------------------------------
_titulo(9, "Una celda se bloquea a mitad de camino: que hay que recalcular")
_camino = R_C10["BFS"].camino
_i_pos = len(_camino) // 3                        # el agente va por aqui
_i_bloq = min(_i_pos + 5, len(_camino) - 2)       # y se bloquea una celda
_pos, _bloq = _camino[_i_pos], _camino[max(_i_bloq, _i_pos + 1)]   # por delante
_copia = {c: set(v) for c, v in INSTANCIA_C10.grafo.items()}
for _v in _copia[_bloq]:
    _copia[_v].discard(_bloq)
_copia[_bloq] = set()
_g_bloq = MappingProxyType({c: frozenset(v) for c, v in _copia.items()})
_r9 = bfs(ProblemaLaberinto(_g_bloq, _pos, INSTANCIA.meta))
print("posicion actual {0}, celda bloqueada {1} ({2} pasos mas adelante)".format(
    _pos, _bloq, _camino.index(_bloq) - _i_pos))
print("replanificacion desde la posicion actual: {0}".format(
    "{0} pasos".format(_r9.profundidad) if _r9.encontrado else "sin camino"))
print("Hay que invalidar todo nodo cuyo camino pase por la celda bloqueada: los "
      "del subarbol, las etiquetas de Lee posteriores y las entradas de la "
      "frontera que cuelguen de ella. Lo ya recorrido sigue valido.")

# --- P11. Diagnosticar una discrepancia de metricas ---------------------------
_titulo(11, "Diagnostico de una discrepancia: BFS de AIMA {0} frente a {1}".format(
    AIMA["BFS"].expandidos, RESULTADO_BFS.expandidos))
print("1) condiciones_de_validez -> {0}: ambos resultados son validos.".format(
    condiciones_de_validez(PROBLEMA, AIMA["BFS"])))
print("2) mismo camino: {0}. 3) la diferencia esta en el conteo: AIMA para al "
      "GENERAR la meta, cuando expande a su padre, que es el {1}.o en nuestro "
      "orden.".format(AIMA["BFS"].camino == RESULTADO_BFS.camino,
                      _orden.index(CAMINO_REFERENCIA[-2]) + 1))

## 13. Entrega

El cuaderno final debe contener:

1. auditoría del generador;
2. formulación formal;
3. versión desde cero;
4. versión SimpleAI;
5. versión AIMA-Python;
6. pruebas automáticas;
7. extensión con ciclos y costos;
8. protocolo experimental y gráficas;
9. análisis teórico;
10. conclusiones y referencias;
11. enlace al historial de trabajo o bitácora incorporada.

Todo el cuaderno debe ejecutarse desde el inicio en un entorno limpio. Las celdas fuera de orden, dependencias implícitas y resultados pegados manualmente se consideran defectos reproducibles.

### Rúbrica

| Criterio | Peso |
|---|---:|
| Modelado, invariantes y auditoría de Hunt-and-Kill | 10% |
| Versión desde cero y estructuras de datos | 25% |
| Adaptación correcta a SimpleAI | 12% |
| Adaptación correcta a AIMA-Python | 13% |
| Pruebas, casos límite y concordancia | 15% |
| Experimentos, métricas y visualización | 12% |
| Análisis teórico y conclusiones | 8% |
| Defensa, trazabilidad y calidad del cuaderno | 5% |

Una solución que produzca un camino pero no satisfaga el contrato, las pruebas o la defensa no se considera completa.

### 13.1 Dónde está cada entregable

| # | Entregable | Dónde |
|---|---|---|
| 1 | Auditoría del generador | Actividad 1, apartados 1.1–1.7: once comprobaciones con evidencia estructurada |
| 2 | Formulación formal | Secciones 2 y 3: instancia, tabla formal, `ProblemaLaberinto`, invariantes |
| 3 | Versión desde cero | Secciones 4–7: contrato, BFS/DFS, UCS, DLS/IDDFS, bidireccional, Lee |
| 4 | Versión SimpleAI | «Versión 2», con V2.1 |
| 5 | Versión AIMA-Python | «Versión 3», con V3.1; biblioteca vendorizada en `aima/` |
| 6 | Pruebas automáticas | Sección 8: 15 pruebas sobre 46 instancias, con contra-pruebas |
| 7 | Extensión con ciclos y costos | Sección 9 |
| 8 | Protocolo experimental y gráficas | Sección 10: tablas de mediana y RIC, seis figuras en `resultados/` |
| 9 | Análisis teórico | Sección 11.1–11.2 |
| 10 | Conclusiones y referencias | Secciones 11.3 y 14.1 |
| 11 | Historial y bitácora | `BITACORA.md` (decisiones D-01 a D-47, con los errores corregidos marcados) e historial de commits por etapa en el repositorio `github.com/DjSantech/Taller_IA_UTP` |

**Ejecución en limpio.** `python tools/ejecutar_cuaderno.py` arranca un kernel nuevo, ejecuta todas las celdas en orden sobre una copia temporal y falla indicando la primera celda que lance una excepción. El cuaderno versionado no guarda salidas (D-04): todo lo que muestra se regenera al ejecutarlo.

**Errores corregidos documentados** (evidencia de autoría, punto 5): D-09 (prueba de no idempotencia mal planteada), D-10 (`inspect.getsource` bajo nbconvert), D-39 (criterio débil para el caso «pasos ≠ costo»), D-44 (un nombre temporal que tapaba una función) y D-47 (comprobaciones que suponían la instancia original). La traza manual sobre un laberinto pequeño (punto 2) corresponde a las trazas de la 3×3 en las secciones 6 y 7.

## 14. Referencias de partida

- Russell, S. J. y Norvig, P. *Artificial Intelligence: A Modern Approach*, 4.ª edición, capítulo 3.
- SimpleAI, documentación de problemas y búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/docs/search_problems.rst>
- SimpleAI, implementación oficial de búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/simpleai/search/traditional.py>
- AIMA-Python, repositorio oficial: <https://github.com/aimacode/aima-python>
- AIMA-Python, módulo de búsqueda: <https://github.com/aimacode/aima-python/blob/master/aima/search.py>
- C. Y. Lee, “An Algorithm for Path Connections and Its Applications”, 1961.
- A. Adamatzky, “Physical maze solvers. All twelve prototypes implement 1961 Lee algorithm”, 2016: <https://arxiv.org/abs/1601.04672>

Las referencias orientan el estudio de las interfaces; no sustituyen la explicación de las decisiones tomadas.

### 14.1 Referencias efectivamente consultadas

1. Russell, S. J. y Norvig, P. *Artificial Intelligence: A Modern Approach*, 4.ª ed., cap. 3: formulación de problemas, búsqueda en árbol y en grafo, figuras 3.7 (BFS/DFS), 3.11 (BFS en grafo), 3.14 (UCS), 3.17 (DLS) y 3.18 (IDDFS); definición del factor de ramificación efectivo.
2. SimpleAI 0.8.3 (PyPI). Código fuente leído del paquete instalado, sin modificarlo: `simpleai/search/traditional.py` (bucle `_search`, `limited_depth_first`, `iterative_limited_depth_first`), `models.py` (`SearchProblem`, `SearchNode`), `utils.py` (fronteras) y `viewers.py` (mecanismo de eventos). Documentación: <https://github.com/simpleai-team/simpleai/blob/master/docs/search_problems.rst>
3. aimacode/aima-python, commit `bbf6bc2` (2026-06-29), licencia MIT: `aima/search.py` y `aima/utils.py`, vendorizados sin modificar. <https://github.com/aimacode/aima-python>
4. Holte, R. C., Felner, A., Sharon, G. y Sturtevant, N. R. «Bidirectional Search That Is Guaranteed to Meet in the Middle», *AAAI*, 2016. Es el algoritmo MM que implementa `bidirectional_search` de AIMA-Python.
5. Lee, C. Y. «An Algorithm for Path Connections and Its Applications», *IRE Transactions on Electronic Computers*, 1961: propagación por frentes y reconstrucción por etiquetas decrecientes.
6. Documentación de Python 3.13: `heapq`, `collections.deque`, `random` (reproducibilidad por semilla), `types.MappingProxyType` y `dataclasses`. Documentación de matplotlib 3.11 y NumPy 2.5 para las figuras y los percentiles.

Las decisiones tomadas a partir de estas fuentes, y las discrepancias encontradas con ellas, están en `BITACORA.md`.